In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install astroquery pyvo astropy pandas pyarrow tqdm

import os
ROOT = '/content/drive/MyDrive/spherex_project'
for p in ['data/raw/spherex_qr2','data/raw/neowise_l1b','data/raw/catwise2020',
          'data/raw/iras_akari','data/interim','data/processed','data/manifests','notebooks','src']:
    os.makedirs(f'{ROOT}/{p}', exist_ok=True)

Mounted at /content/drive


In [ ]:
#New session: setup cell
from google.colab import drive
drive.mount('/content/drive')
!pip -q install astroquery pyvo astropy pandas pyarrow tqdm requests

import os, re, json, requests, pyvo
import numpy as np, pandas as pd
from astropy.io import fits

ROOT = '/content/drive/MyDrive/spherex_project'
fields = json.load(open(f'{ROOT}/configs/fields.json'))
sel = {n: pd.read_csv(f'{ROOT}/data/manifests/spherex_{n}_selected.csv') for n in fields}
for n, d in sel.items():
    print(n, fields[n]['role'], len(d), 'images')     # expect 139, 131, 116

Mounted at /content/drive
elon240 train 139 images
elon150 validation 131 images
elon030 test 116 images


In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord

fields = {}
for name, lon in {'field_A':100, 'field_B':160, 'field_C':220}.items():   # ecliptic longitude, deg
    c = SkyCoord(lon=lon*u.deg, lat=0*u.deg, frame='barycentrictrueecliptic').icrs
    fields[name] = (c.ra.deg, c.dec.deg)
print(fields)

{'field_A': (np.float64(100.88298081874369), np.float64(23.062137684054704)), 'field_B': (np.float64(161.53758373898614), np.float64(7.817745808718158)), 'field_C': (np.float64(217.59499184263856), np.float64(-14.815468591599052))}


In [ ]:
from astroquery.ipac.irsa import Irsa
import pandas as pd

name = 'field_A'
ra, dec = fields[name]
pos = SkyCoord(ra, dec, unit='deg')
tab = Irsa.query_sia(pos=(pos, 0.2*u.deg), collection='spherex_qr')
df = tab.to_pandas()
print(len(df)); print(df.columns.tolist()); df.head()
df.to_csv(f'{ROOT}/data/manifests/spherex_{name}_query.csv', index=False)

0
['s_ra', 's_dec', 'facility_name', 'instrument_name', 'dataproduct_subtype', 'calib_level', 'dataproduct_type', 'energy_bandpassname', 'energy_emband', 'obs_id', 's_resolution', 'em_min', 'em_max', 'em_res_power', 'proposal_title', 'access_url', 'access_format', 'access_estsize', 't_exptime', 's_region', 'obs_collection', 'obs_intent', 'algorithm_name', 'facility_keywords', 'instrument_keywords', 'environment_photometric', 'proposal_id', 'proposal_pi', 'proposal_project', 'target_name', 'target_type', 'target_standard', 'target_moving', 'target_keywords', 'obs_release_date', 's_xel1', 's_xel2', 's_pixel_scale', 'position_timedependent', 't_min', 't_max', 't_resolution', 't_xel', 'obs_publisher_did', 's_fov', 'em_xel', 'pol_states', 'pol_xel', 'cloud_access', 'o_ucd', 'upload_row_id']


In [ ]:
from astroquery.ipac.irsa import Irsa
import astropy.units as u
from astropy.coordinates import SkyCoord

def count_cover(ra, dec):
    q = f"SELECT COUNT(*) AS n FROM spherex.obscore WHERE CONTAINS(POINT('ICRS',{ra},{dec}), s_region)=1"
    return int(Irsa.query_tap(q).to_table()['n'][0])

# CONTROL: north ecliptic pole (SPHEREx observes deep fields at the ecliptic poles, so this should have data)
nep = SkyCoord(lon=0*u.deg, lat=90*u.deg, frame='barycentrictrueecliptic').icrs
print('NEP control:', count_cover(nep.ra.deg, nep.dec.deg))

# SCAN: coverage along the ecliptic, every 30 degrees
for lon in range(0, 360, 30):
    c = SkyCoord(lon=lon*u.deg, lat=0*u.deg, frame='barycentrictrueecliptic').icrs
    print(lon, round(c.ra.deg,2), round(c.dec.deg,2), count_cover(c.ra.deg, c.dec.deg))

NEP control: 29188
0 0.0 0.0 335
30 27.91 11.47 326
60 57.82 20.15 332
90 90.0 23.44 328
120 122.19 20.15 219
150 152.09 11.47 247
180 180.0 -0.0 341
210 207.91 -11.47 347
240 237.82 -20.15 376
270 270.0 -23.44 151
300 302.19 -20.15 293
330 332.09 -11.47 269


In [ ]:
def exposures(ra, dec, top=500):
    q = f"""SELECT TOP {top} obs_id, t_min, t_max, energy_bandpassname, em_min, em_max,
                   access_url, access_estsize
            FROM spherex.obscore
            WHERE CONTAINS(POINT('ICRS',{ra},{dec}), s_region)=1
            ORDER BY t_min"""
    return Irsa.query_tap(q).to_table().to_pandas()

df = exposures(ra, dec)           # the field you chose from the scan
print(len(df), 'images; MJD', df.t_min.min(), 'to', df.t_max.max())
print('est. size (GB):', df.access_estsize.sum()/1e6)   # ObsCore defines this column in kilobytes; confirm on one file
df.to_csv(f'{ROOT}/data/manifests/spherex_field_A_exposures.csv', index=False)

297 images; MJD 60945.699319 to 61314.085251
est. size (GB): 0.0


In [ ]:
import numpy as np, pandas as pd

chosen = {'field_A': (57.82, 20.15),    # ecliptic lon 60,  332 footprints
          'field_B': (180.0,  0.0),     # ecliptic lon 180, 341 footprints
          'field_C': (237.82, -20.15)}  # ecliptic lon 240, 376 footprints

for name, (ra, dec) in chosen.items():
    df = exposures(ra, dec, top=2000).sort_values('t_min').reset_index(drop=True)
    df['visit'] = (df.t_min.diff().fillna(999) > 1.0).cumsum()   # new "visit" after a >1 day gap
    v = df.groupby('visit').agg(n=('obs_id','size'), t0=('t_min','min'), t1=('t_max','max'))
    v['hours'] = (v.t1 - v.t0) * 24
    print(f"\n{name}: {len(df)} images, {len(v)} visits, "
          f"median {v.n.median():.0f} images/visit, median visit length {v.hours.median():.1f} h")
    print("wavelength range (um):", round(df.em_min.min()*1e6,2), "-", round(df.em_max.max()*1e6,2),
          "| distinct bands:", df.energy_bandpassname.nunique())
    df.to_csv(f'{ROOT}/data/manifests/spherex_{name}_exposures.csv', index=False)

print(df.access_estsize.describe())          # confirms whether the size column is empty
print(df.access_url.iloc[0]); print(df.access_format.iloc[0])


field_A: 332 images, 10 visits, median 22 images/visit, median visit length 29.5 h
wavelength range (um): 0.75 - 5.01 | distinct bands: 6

field_B: 341 images, 19 visits, median 8 images/visit, median visit length 0.1 h
wavelength range (um): 0.75 - 5.01 | distinct bands: 6

field_C: 376 images, 21 visits, median 10 images/visit, median visit length 13.0 h
wavelength range (um): 0.75 - 5.01 | distinct bands: 6
count     0.0
mean     <NA>
std      <NA>
min      <NA>
25%      <NA>
50%      <NA>
75%      <NA>
max      <NA>
Name: access_estsize, dtype: Float64
https://irsa.ipac.caltech.edu/datalink/links/spherex?ID=ivo://irsa.ipac/spherex_qr2?2025W30_1B_0483_1/D6


AttributeError: 'DataFrame' object has no attribute 'access_format'

In [ ]:
rows = []
for lon in range(0, 360, 30):
    c = SkyCoord(lon=lon*u.deg, lat=0*u.deg, frame='barycentrictrueecliptic').icrs
    df = exposures(c.ra.deg, c.dec.deg, top=2000).sort_values('t_min').reset_index(drop=True)
    visit = (df.t_min.diff().fillna(999) > 1.0).cumsum()
    v = df.groupby(visit).agg(n=('obs_id','size'), t0=('t_min','min'), t1=('t_max','max'))
    hrs = (v.t1 - v.t0) * 24
    rows.append(dict(lon=lon, ra=round(c.ra.deg,2), dec=round(c.dec.deg,2), images=len(df),
                     visits=len(v), med_imgs=v.n.median(), med_hours=round(hrs.median(),1),
                     good_visits=int(((hrs>=1) & (hrs<=36) & (v.n>=5)).sum())))
print(pd.DataFrame(rows).to_string(index=False))

 lon     ra    dec  images  visits  med_imgs  med_hours  good_visits
   0   0.00   0.00     335      18       8.0        6.6            4
  30  27.91  11.47     326      23       9.0        1.7            8
  60  57.82  20.15     332      10      22.0       29.5            3
  90  90.00  23.44     328      10      17.0        7.5            3
 120 122.19  20.15     219      11      14.0        5.1            4
 150 152.09  11.47     247      12      15.0        9.9            7
 180 180.00  -0.00     341      19       8.0        0.1            6
 210 207.91 -11.47     347      14       9.0        1.8            3
 240 237.82 -20.15     376      21      10.0       13.0            9
 270 270.00 -23.44     151      13       4.0        0.1            1
 300 302.19 -20.15     293      16       8.0        0.1            3
 330 332.09 -11.47     269      15      10.0        3.4            3


In [ ]:
import pyvo
dl = pyvo.dal.adhoc.DatalinkResults.from_result_url(df.access_url.iloc[0])
t = dl.to_table()
print(t.colnames)
print(t.to_pandas().drop(columns=['ID'], errors='ignore').to_string())

['id', 'access_url', 'service_def', 'error_message', 'semantics', 'description', 'content_type', 'content_length', 'content_qualifier', 'local_semantics', 'link_auth', 'link_authorized', 'path', 'center_ra', 'center_dec', 'cloud_access']
                                                  id                                                                                                                                    access_url service_def error_message     semantics                                                     description      content_type  content_length content_qualifier local_semantics link_auth  link_authorized                                                                                                   path   center_ra  center_dec                                                                                                                                                                 cloud_access
0   ivo://irsa.ipac/spherex_qr2?2025W20_1C_0165_1/D4  https://irsa.i

In [ ]:
#Step A: Record the fields and select the usable images
import json, os
fields = {
  'elon240': dict(ra=237.82, dec=-20.15, role='train'),
  'elon150': dict(ra=152.09, dec= 11.47, role='validation'),
  'elon030': dict(ra= 27.91, dec= 11.47, role='test'),
}
os.makedirs(f'{ROOT}/configs', exist_ok=True)
json.dump(fields, open(f'{ROOT}/configs/fields.json','w'), indent=2)

for name, f in fields.items():
    df = exposures(f['ra'], f['dec'], top=2000).sort_values('t_min').reset_index(drop=True)
    df['visit'] = (df.t_min.diff().fillna(999) > 1.0).cumsum()
    g = df.groupby('visit')
    df['n_in_visit'] = g.obs_id.transform('size')
    df['visit_hours'] = (g.t_max.transform('max') - g.t_min.transform('min')) * 24
    keep = df[(df.n_in_visit >= 5) & (df.visit_hours >= 1) & (df.visit_hours <= 36)].copy()
    keep['field'] = name
    keep.to_csv(f'{ROOT}/data/manifests/spherex_{name}_selected.csv', index=False)
    print(f"{name} ({f['role']}): {len(df)} -> {len(keep)} images in {keep.visit.nunique()} visits | "
          f"full size ~{len(keep)*71.6/1000:.1f} GB, as ~5 MB cutouts ~{len(keep)*5/1000:.1f} GB")

elon240 (train): 376 -> 139 images in 9 visits | full size ~10.0 GB, as ~5 MB cutouts ~0.7 GB
elon150 (validation): 247 -> 131 images in 7 visits | full size ~9.4 GB, as ~5 MB cutouts ~0.7 GB
elon030 (test): 326 -> 116 images in 8 visits | full size ~8.3 GB, as ~5 MB cutouts ~0.6 GB


In [ ]:
#Step B: Test-download two full images and check the structure
import pyvo, requests, pandas as pd
from astropy.io import fits

sel = pd.read_csv(f'{ROOT}/data/manifests/spherex_elon240_selected.csv')
tdir = f'{ROOT}/data/raw/spherex_qr2/elon240/_test'; os.makedirs(tdir, exist_ok=True)

for url in sel.access_url.head(2):
    d = pyvo.dal.adhoc.DatalinkResults.from_result_url(url).to_table().to_pandas()
    row = d[d.semantics == '#this'].iloc[0]
    out = f"{tdir}/{row.access_url.split('/')[-1]}"
    with requests.get(row.access_url, stream=True) as r:
        r.raise_for_status()
        with open(out, 'wb') as fh:
            for chunk in r.iter_content(1 << 20): fh.write(chunk)
    print(out.split('/')[-1], round(os.path.getsize(out)/1e6,1), 'MB; expected', round(row.content_length/1e6,1))
    fits.open(out).info()

level2_2025W30_2B_0345_1D6_spx_l2b-v20-2025-262.fits 71.6 MB; expected 71.6
Filename: /content/drive/MyDrive/spherex_project/data/raw/spherex_qr2/elon240/_test/level2_2025W30_2B_0345_1D6_spx_l2b-v20-2025-262.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1  IMAGE         1 ImageHDU       237   (2040, 2040)   float32   
  2  FLAGS         1 ImageHDU       146   (2040, 2040)   int32   
  3  VARIANCE      1 ImageHDU       119   (2040, 2040)   float32   
  4  ZODI          1 ImageHDU       119   (2040, 2040)   float32   
  5  PSF           1 ImageHDU       509   (101, 101, 121)   float32   
  6  WCS-WAVE      1 BinTableHDU     18   1R x 3C   [9J, 9J, 162E]   
level2_2025W30_2B_0345_1D3_spx_l2b-v20-2025-262.fits 71.6 MB; expected 71.6
Filename: /content/drive/MyDrive/spherex_project/data/raw/spherex_qr2/elon240/_test/level2_2025W30_2B_0345_1D3_spx_l2b-v20-2025-262.fits
No.    Name      Ver    Type      Cards   Dimensio

In [ ]:
#Step C: Read the cutout service description
xml = requests.get(sel['elon240'].access_url.iloc[0], timeout=60).text
m = re.search(r"<RESOURCE[^>]*ID=[\"']ibe-cutout[\"'].*?</RESOURCE>", xml, re.S)
print(m.group(0)[:4000] if m else xml[:4000])

<RESOURCE ID="ibe-cutout" type="meta" utype="adhoc:service">
    <DESCRIPTION>Link to generate an image cutout</DESCRIPTION>
    <GROUP name="inputParams">
      <PARAM name="ra" datatype="double" ref="col_13" ucd="pos.eq.ra;meta.main" value="">
        <DESCRIPTION>center RA (J2000)</DESCRIPTION>
      </PARAM>
      <PARAM name="dec" datatype="double" ref="col_14" ucd="pos.eq.dec;meta.main" value="">
        <DESCRIPTION>center DEC (J2000)</DESCRIPTION>
      </PARAM>
      <PARAM name="size" datatype="double" ucd="phys.size" value="0.01">
        <DESCRIPTION>cutout size in degrees</DESCRIPTION>
      </PARAM>
      <PARAM name="path" datatype="char" arraysize="*" ref="col_12" value="">
        <DESCRIPTION>Path to input file</DESCRIPTION>
      </PARAM>
    </GROUP>
    <PARAM name="accessURL" datatype="char" arraysize="*" ucd="meta.ref.url" value="https://irsa.ipac.caltech.edu/ibe/cutout"/>
  </RESOURCE>


In [ ]:
#Step D: images versus exposures
for n, d in sel.items():
    d['exposure'] = d.obs_id.astype(str).str.replace(r'/?D\d$', '', regex=True)
    ev = d.groupby('visit').exposure.nunique()
    print(f"{n}: {len(d)} images, {d.exposure.nunique()} distinct exposures | "
          f"exposures per visit: min {ev.min()}, median {ev.median()}, max {ev.max()}")
print(sel['elon240'][['obs_id', 'exposure']].head(4))

elon240: 139 images, 70 distinct exposures | exposures per visit: min 5, median 8.0, max 12
elon150: 131 images, 66 distinct exposures | exposures per visit: min 6, median 8.0, max 15
elon030: 116 images, 59 distinct exposures | exposures per visit: min 5, median 7.0, max 12
              obs_id           exposure
0  2025W30_2B_0345_1  2025W30_2B_0345_1
1  2025W30_2B_0345_1  2025W30_2B_0345_1
2  2025W30_2B_0345_2  2025W30_2B_0345_2
3  2025W30_2B_0345_2  2025W30_2B_0345_2


In [ ]:
#Step E0: confirm the exposure/detector structure (no download)
import time, requests, pyvo, os
import numpy as np, pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm

def add_cols(d):
    d = d.copy()
    d['exposure'] = d.obs_id.astype(str)
    d['detector'] = d.access_url.str.extract(r'/(D\d)$')[0]
    return d

d = add_cols(sel['elon240'])
print(d.groupby('exposure').size().value_counts())            # images per exposure
ex = d[d.exposure == d.exposure.iloc[0]]
print(ex[['detector','em_min','em_max','t_min']].assign(em_min_um=ex.em_min*1e6, em_max_um=ex.em_max*1e6))

2    69
1     1
Name: count, dtype: int64
  detector    em_min    em_max         t_min  em_min_um  em_max_um
0       D6  0.000004  0.000005  60882.355751     4.4115     5.0096
1       D3  0.000002  0.000002  60882.355751     1.6301     2.4346


In [ ]:
#Step E1: download five cutouts and inspect them
SIZE_DEG, CUT_URL = 0.5, 'https://irsa.ipac.caltech.edu/ibe/cutout'

def resolve(access_url):
    t = pyvo.dal.adhoc.DatalinkResults.from_result_url(access_url).to_table().to_pandas()
    r = t[t.semantics == '#cutout']
    return r.iloc[0]['path'] if len(r) else None

def get_cutout(path, ra, dec, out, retries=3):
    if os.path.exists(out) and os.path.getsize(out) > 1e5: return 'cached'
    err = ''
    for k in range(retries):
        try:
            r = requests.get(CUT_URL, params=dict(path=path, ra=ra, dec=dec, size=SIZE_DEG), timeout=180)
            if r.status_code == 200 and r.content[:6] == b'SIMPLE':   # real FITS, not an error page
                open(out, 'wb').write(r.content); return 'ok'
            err = f'{r.status_code} {r.text[:150]}'
        except Exception as e:
            err = str(e)
        time.sleep(2 * (k + 1))
    return 'FAILED: ' + err

def qc(fn):
    with fits.open(fn) as h:
        im, fl = h['IMAGE'].data, h['FLAGS'].data
        return dict(ny=im.shape[0], nx=im.shape[1], nan_frac=float(np.isnan(im).mean()),
                    flagged_frac=float((fl != 0).mean()), n_ext=len(h), mb=os.path.getsize(fn)/1e6)

def run(field, n=None):
    d, f = add_cols(sel[field]), fields[field]
    d = d.head(n) if n else d
    outdir = f"{ROOT}/data/raw/spherex_qr2/{field}/cutouts"; os.makedirs(outdir, exist_ok=True)
    rows = []
    for _, r in tqdm(d.iterrows(), total=len(d)):
        try: path = resolve(r.access_url)
        except Exception: path = None
        fn, status = None, 'no cutout path'
        if path:
            fn = f"{outdir}/{os.path.basename(path)[:-5]}_{field}.fits"
            status = get_cutout(path, f['ra'], f['dec'], fn)
        rec = r.to_dict() | dict(field=field, role=f['role'], path=path, local_file=fn, status=status)
        if status in ('ok', 'cached'): rec |= qc(fn)
        rows.append(rec); time.sleep(0.2)               # be polite to the archive
    out = pd.DataFrame(rows)
    out.to_csv(f"{ROOT}/data/manifests/cutouts_{field}{'_test' if n else ''}.csv", index=False)
    return out

t = run('elon240', n=5)
print(t[['obs_id','detector','status','nx','ny','mb','nan_frac','flagged_frac']])
fits.open(t.local_file.iloc[0]).info()

  0%|          | 0/5 [00:00<?, ?it/s]

              obs_id detector          status     nx     ny       mb  \
0  2025W30_2B_0345_1       D6          cached  292.0  292.0  6.41952   
1  2025W30_2B_0345_1       D3  no cutout path    NaN    NaN      NaN   
2  2025W30_2B_0345_2       D3          cached  292.0  293.0  6.41952   
3  2025W30_2B_0345_2       D6          cached  292.0  292.0  6.41952   
4  2025W30_2B_0345_3       D3          cached  292.0  293.0  6.41952   

   nan_frac  flagged_frac  
0  0.000364      0.430721  
1       NaN           NaN  
2  0.000047      0.745336  
3  0.000352      0.431202  
4  0.000070      0.744471  
Filename: /content/drive/MyDrive/spherex_project/data/raw/spherex_qr2/elon240/cutouts/level2_2025W30_2B_0345_1D6_spx_l2b-v20-2025-262_elon240.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1  IMAGE         1 ImageHDU       237   (292, 292)   float32   
  2  FLAGS         1 ImageHDU       146   (292, 292)   int32   
  3  VARI

In [ ]:
import socket
socket.setdefaulttimeout(60)          # any request that stalls now fails after 60 s instead of hanging
from concurrent.futures import ThreadPoolExecutor
def run_fast(field, workers=4):
    d, f = add_cols(sel[field]), fields[field]
    outdir = f"{ROOT}/data/raw/spherex_qr2/{field}/cutouts"; os.makedirs(outdir, exist_ok=True)
    pm = f"{ROOT}/data/manifests/cutouts_{field}.csv"
    prev = {}
    if os.path.exists(pm):            # reuse earlier results so finished files skip the lookup
        p = pd.read_csv(pm)
        p = p[p.status.isin(['ok', 'cached']) & p.local_file.notna()]
        prev = {r.access_url: (r.path, r.local_file) for r in p.itertuples()}

    def one(r):
        path, fn, status = None, None, 'no cutout path'
        hit = prev.get(r['access_url'])
        if hit and os.path.exists(hit[1]) and os.path.getsize(hit[1]) > 5e6:
            path, fn, status = hit[0], hit[1], 'cached'
        else:
            try: path = resolve(r['access_url'])
            except Exception: path = None
            if path:
                fn = f"{outdir}/{os.path.basename(path)[:-5]}_{field}.fits"
                status = get_cutout(path, f['ra'], f['dec'], fn)
        rec = dict(r) | dict(field=field, role=f['role'], path=path, local_file=fn, status=status)
        if status in ('ok', 'cached'): rec |= qc(fn)
        return rec

    with ThreadPoolExecutor(workers) as ex:
        rows = list(tqdm(ex.map(one, d.to_dict('records')), total=len(d)))
    out = pd.DataFrame(rows)
    out.to_csv(pm, index=False)
    return out

for fld in ['elon240', 'elon150', 'elon030']:
    m = run_fast(fld)
    print(fld, m.status.value_counts().to_dict(), '| total', round(m.mb.sum()/1000, 2), 'GB',
          '| files under 6 MB:', int((m.mb < 6).sum()))

  0%|          | 0/139 [00:00<?, ?it/s]

elon240 {'cached': 139} | total 0.84 GB | files under 6 MB: 48


  0%|          | 0/131 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
m = pd.read_csv(f'{ROOT}/data/manifests/cutouts_elon240.csv')
m['clipped'] = (m.nx < 285) | (m.ny < 285)
print(m.groupby('detector').agg(n=('mb','size'), clipped=('clipped','sum'), mean_mb=('mb','mean')))
print(m[m.mb < 6][['obs_id','detector','nx','ny','mb']].head(8))

           n  clipped   mean_mb
detector                       
D1        30        5  6.211872
D2        20        5  5.788224
D3        20        2  5.974272
D4        30        5  6.211488
D5        20        6  5.784480
D6        19        2  5.920067
               obs_id detector   nx   ny       mb
42  2025W32_2D_0544_1       D4  216  211  5.78592
43  2025W32_2D_0544_1       D1  213  221  5.80896
93  2026W09_2A_0077_1       D6  292  179  5.89248
94  2026W09_2A_0077_1       D3  292  167  5.83488
95  2026W30_1B_0237_1       D5  293  213  4.94208
96  2026W30_1B_0237_1       D2  293  202  4.89888
97  2026W30_1B_0237_2       D5  293  293  5.32224
98  2026W30_1B_0237_2       D2  293  293  5.32512


In [ ]:
import io, json, glob, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from astropy.io.votable import parse_single_table

SIZE_DEG, CUT_URL = 0.5, 'https://irsa.ipac.caltech.edu/ibe/cutout'
LOCK = threading.Lock()
CACHE_F = f'{ROOT}/data/manifests/links_cache.json'
cache = json.load(open(CACHE_F)) if os.path.exists(CACHE_F) else {}
for fp in glob.glob(f'{ROOT}/data/manifests/cutouts_elon*.csv'):      # reuse links already found
    p = pd.read_csv(fp)
    cache.update({r.access_url: r.path for r in p.itertuples() if isinstance(r.path, str)})

def resolve(access_url, retries=3):
    if access_url in cache: return cache[access_url]
    for k in range(retries):
        try:
            r = requests.get(access_url, timeout=60); r.raise_for_status()
            t = parse_single_table(io.BytesIO(r.content), pedantic=False).to_table().to_pandas()
            rows = t[t.semantics == '#cutout']
            path = rows.iloc[0]['path'] if len(rows) else None
            if isinstance(path, str):
                with LOCK: cache[access_url] = path
                return path
        except Exception:
            time.sleep(2 * (k + 1))
    return None

def valid(fn):
    try:
        with fits.open(fn) as h:
            _ = h['IMAGE'].data; _ = h['WCS-WAVE'].data       # reading the last extension fails on a truncated file
            return len(h) == 7
    except Exception:
        return False

def get_cutout(path, ra, dec, out, retries=3):
    if os.path.exists(out) and valid(out): return 'cached'
    err = ''
    for k in range(retries):
        try:
            r = requests.get(CUT_URL, params=dict(path=path, ra=ra, dec=dec, size=SIZE_DEG), timeout=(15, 120))
            if r.status_code == 200 and r.content[:6] == b'SIMPLE':
                open(out + '.part', 'wb').write(r.content); os.replace(out + '.part', out)
                return 'ok'
            err = f'{r.status_code} {r.text[:150]}'
        except Exception as e:
            err = str(e)
        time.sleep(2 * (k + 1))
    return 'FAILED: ' + err

def run_fast(field, workers=4):
    d, f = add_cols(sel[field]), fields[field]
    outdir = f"{ROOT}/data/raw/spherex_qr2/{field}/cutouts"; os.makedirs(outdir, exist_ok=True)
    def one(r):
        path = resolve(r['access_url'])
        fn, status = None, 'no cutout path'
        if path:
            fn = f"{outdir}/{os.path.basename(path)[:-5]}_{field}.fits"
            status = get_cutout(path, f['ra'], f['dec'], fn)
        rec = dict(r) | dict(field=field, role=f['role'], path=path, local_file=fn, status=status)
        if status in ('ok', 'cached'): rec |= qc(fn)
        return rec
    rows = []
    try:
        with ThreadPoolExecutor(workers) as ex:
            futs = [ex.submit(one, r) for r in d.to_dict('records')]
            for fu in tqdm(as_completed(futs), total=len(futs)): rows.append(fu.result())
    finally:
        json.dump(cache, open(CACHE_F, 'w'))                  # keep found links even if interrupted
    out = pd.DataFrame(rows).sort_values(['t_min', 'detector']).reset_index(drop=True)
    out.to_csv(f"{ROOT}/data/manifests/cutouts_{field}.csv", index=False)
    return out

for fld in ['elon240', 'elon150', 'elon030']:
    m = run_fast(fld)
    print(fld, m.status.value_counts().to_dict(), '| total', round(m.mb.sum()/1000, 2), 'GB')

  0%|          | 0/139 [00:00<?, ?it/s]

elon240 {'cached': 139} | total 0.84 GB


  0%|          | 0/131 [00:00<?, ?it/s]

AttributeError: 'DataFrame' object has no attribute 'mb'

status
no cutout path    131
Name: count, dtype: int64
no cutout path
rows with no cutout path: 131


HTTP 200 12701 application/x-votable+xml
PARSE ERROR: TypeError("parse() got an unexpected keyword argument 'pedantic'")
                                                path
1  spherex/qr2/level2/2025W17_4B/l2b-v20-2025-240...


In [ ]:
def resolve(access_url, retries=3):
    if access_url in cache: return cache[access_url]
    last = 'no #cutout row'
    for k in range(retries):
        try:
            r = requests.get(access_url, timeout=60); r.raise_for_status()
            t = parse_single_table(io.BytesIO(r.content), verify='ignore').to_table(use_names_over_ids=True).to_pandas()
            rows = t[t.semantics == '#cutout']
            path = rows.iloc[0]['path'] if len(rows) else None
            if isinstance(path, str):
                with LOCK: cache[access_url] = path
                return path
        except Exception as e:
            last = repr(e)
            time.sleep(2 * (k + 1))
    print('lookup failed:', last)
    return None

u = sel['elon150'].access_url.iloc[0]
print(resolve(u))                 # should print a path starting with spherex/qr2/level2/...

spherex/qr2/level2/2025W17_4B/l2b-v20-2025-240/4/level2_2025W17_4B_0063_1D4_spx_l2b-v20-2025-240.fits


In [ ]:
full = sel['elon150']
sel['elon150'] = full.head(5)
t5 = run_fast('elon150', workers=1)
sel['elon150'] = full                                   # restore immediately
print(t5[['obs_id','detector','status','nx','ny','mb']])

  0%|          | 0/5 [00:00<?, ?it/s]

              obs_id detector  status   nx   ny       mb
0  2025W17_4B_0063_1       D1  cached  292  293  6.41952
1  2025W17_4B_0063_1       D4  cached  292  293  6.41952
2  2025W17_4B_0063_2       D1  cached  292  293  6.41952
3  2025W17_4B_0063_2       D4  cached  292  293  6.41952
4  2025W17_4B_0063_3       D4  cached  292  292  6.41952


In [ ]:
for fld in ['elon150', 'elon030']:
    m = run_fast(fld)
    print(fld, m.status.value_counts().to_dict(), '| total', round(m.get('mb', pd.Series(dtype=float)).sum()/1000, 2), 'GB')

  0%|          | 0/131 [00:00<?, ?it/s]

lookup failed: ConnectionError(ProtocolError('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')))
elon150 {'cached': 130, 'no cutout path': 1} | total 0.83 GB


  0%|          | 0/116 [00:00<?, ?it/s]

elon030 {'ok': 116} | total 0.72 GB


In [ ]:
# 1. fetch the one missing cutout (everything else is skipped)
m = run_fast('elon150')
print(m.status.value_counts().to_dict())

# 2. verify every file on Drive opens and is complete
tot = 0
for fld in fields:
    files = sorted(glob.glob(f'{ROOT}/data/raw/spherex_qr2/{fld}/cutouts/*.fits'))
    bad = [f for f in files if not valid(f)]
    gb = sum(os.path.getsize(f) for f in files) / 1e9; tot += gb
    print(fld, fields[fld]['role'], len(files), 'files | bad:', len(bad), f'| {gb:.2f} GB')
print('TOTAL', round(tot, 2), 'GB')

# 3. build one master table and look at coverage (the 48 small files from elon240)
M = pd.concat([pd.read_csv(f'{ROOT}/data/manifests/cutouts_{f}.csv') for f in fields], ignore_index=True)
M['coverage'] = (M.nx * M.ny) / (292 * 292)
print(M.groupby(['field', 'detector']).agg(n=('mb', 'size'), mean_cov=('coverage', 'mean'),
                                           clipped=('coverage', lambda s: int((s < 0.97).sum()))))
M.to_csv(f'{ROOT}/data/manifests/master_manifest.csv', index=False)

  0%|          | 0/131 [00:00<?, ?it/s]

{'cached': 131}
elon240 train 139 files | bad: 0 | 0.84 GB
elon150 validation 131 files | bad: 0 | 0.83 GB
elon030 test 116 files | bad: 0 | 0.72 GB
TOTAL 2.39 GB
                   n  mean_cov  clipped
field   detector                       
elon030 D1        29  0.972485        3
        D2        19  0.959855        3
        D3        10  0.931074        3
        D4        30  0.972372        3
        D5        19  0.957847        3
        D6         9  0.957382        1
elon150 D1        24  0.984876        2
        D2        17  0.967600        4
        D3        25  0.939529        5
        D4        24  0.982877        2
        D5        17  0.971703        4
        D6        24  0.936732        5
elon240 D1        30  0.957786        5
        D2        20  0.943360        5
        D3        20  0.975685        2
        D4        30  0.956058        5
        D5        20  0.941578        6
        D6        19  0.975487        2


In [ ]:
import re, glob
from astropy.wcs import WCS

M = pd.read_csv(f'{ROOT}/data/manifests/master_manifest.csv')
M = M[M.status.isin(['ok', 'cached'])]

# 1. which header keywords exist (so the QC scan uses real names)
fn = M.local_file.iloc[0]
pat = re.compile(r'DETECTOR|MJD|DATE|EXP|TELAPSE|FINAST|TRANS|VERSION|SPS|BUNIT|PIXEL|L2DQA|OBSID', re.I)
n = 0
for k in fits.getheader(fn, 'IMAGE'):
    if k not in ('COMMENT', 'HISTORY') and pat.search(k):
        print(f"{k:14s} {str(fits.getheader(fn, 'IMAGE')[k])[:60]}"); n += 1
        if n >= 40: break

# 2. which flag bits dominate (one cutout per field x detector)
BITS = {0:'TRANSIENT',1:'OVERFLOW',2:'SUR_ERROR',6:'NONFUNC',7:'DICHROIC',9:'MISSING',10:'HOT',11:'COLD',
        12:'FULLSAMPLE',14:'PHANMISS',15:'NONLINEAR',17:'PERSIST',19:'OUTLIER',21:'SOURCE',22:'GHOST',
        24:'GHOST_EXT',26:'BLOOM',27:'SNOWBALL',28:'HALO',29:'SAT_HALO'}
rows = []
for r in M.groupby(['field', 'detector']).head(1).itertuples():
    fl = fits.getdata(r.local_file, 'FLAGS')
    d = {BITS[b]: float(((fl >> b) & 1).mean()) for b in BITS}
    d.update(field=r.field, det=r.detector); rows.append(d)
F = pd.DataFrame(rows).set_index(['field', 'det'])
print((F.loc[:, (F > 0.005).any()] * 100).round(1).to_string())      # % of pixels, bits above 0.5%

# 3. is the cutout's wavelength consistent with the full image's? (the 2 full test files)
def wave_at(h, x, y):
    w = WCS(h[1].header, h, fix=False, key='W')
    return w.pixel_to_world(x, y)[0]

for full in sorted(glob.glob(f'{ROOT}/data/raw/spherex_qr2/elon240/_test/*.fits')):
    cut = f"{ROOT}/data/raw/spherex_qr2/elon240/cutouts/{os.path.basename(full)[:-5]}_elon240.fits"
    try:
        with fits.open(cut) as hc, fits.open(full) as hf:
            ny, nx = hc[1].data.shape
            sky = WCS(hc[1].header).pixel_to_world(nx // 2, ny // 2)
            xf, yf = WCS(hf[1].header).world_to_pixel(sky)
            print(os.path.basename(full)[:36], '| cutout:', wave_at(hc, nx // 2, ny // 2),
                  '| full image:', wave_at(hf, float(xf), float(yf)))
    except Exception as e:
        print('wavelength check failed:', repr(e))

BUNIT          MJy / sr
DETECTOR       3
OBSID          2025W30_2B_0345_1
EXPIDN         202530202034513
DATE           2025-09-20T09:08:13.934
SPS_TARGET     142045
SPS_FLAGS      all_sky
SPS_ELON       236.666667
SPS_ELAT       0.361867
SPS_EPA        180.0
FINAST         0
L2DQAFLG       Pass
L2 N_TRANSIENT 21900
MJDREF         0.0
DATE-OBS       2025-07-26T08:32:16.892
MJD-OBS        60882.355751065
DATE-BEG       2025-07-26T08:32:16.892
MJD-BEG        60882.355751065
DATE-AVG       2025-07-26T08:33:14.451
MJD-AVG        60882.356417254
DATE-END       2025-07-26T08:34:12.010
MJD-END        60882.357083443
TELAPSE        118.1873
             TRANSIENT  DICHROIC  PERSIST  OUTLIER  SOURCE
field   det                                               
elon240 D3         0.5       0.0      0.2      1.4    73.0
        D6         0.5       0.0      0.1      1.3    41.4
        D2         0.3       0.0      0.1      1.2    67.1
        D5         0.5       0.0      0.1      0.4    82.4
     

INFO:astropy:
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                


INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                 [astropy.wcs.wcs]


INFO:astropy:
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                


INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                 [astropy.wcs.wcs]
level2_2025W30_2B_03

INFO:astropy:
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                


INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                 [astropy.wcs.wcs]


INFO:astropy:
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                


INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                 [astropy.wcs.wcs]
level2_2025W30_2B_03

In [ ]:
#Preprocessing starts

In [ ]:
#Step 4b: scan all 386 cutouts and apply the gate
import logging, warnings
logging.getLogger('astropy').setLevel(logging.ERROR); warnings.filterwarnings('ignore')
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
# 0. test the bright-star idea (four well-known stars, hand-typed positions; a sanity check, not data)
stars = {'Regulus': (152.0929, 11.9672), 'Antares': (247.3519, -26.4320),
         'Spica': (201.2983, -11.1614), 'Aldebaran': (68.9802, 16.5093)}
for n, f in fields.items():
    c = SkyCoord(f['ra'], f['dec'], unit='deg')
    print(n, {s: round(float(c.separation(SkyCoord(*xy, unit='deg')).deg), 2) for s, xy in stars.items()})

# 1. per-cutout scan
def mask_of(bits): return sum(1 << b for b in bits)
BAD = mask_of([0,1,2,6,7,9,10,11,14,15,17,19,22,24,26,27,28,29])        # all real problems; SOURCE (21) excluded
GROUPS = {'cr':[0,1,27,28], 'streak':[19,29], 'ghost':[22,24], 'persist':[17],
          'dichroic':[7], 'bloom':[26], 'badpix':[6,9,10,11,14,15]}

def scan(fn):
    with fits.open(fn, memmap=False) as h:
        im, fl, var, zodi = [h[k].data for k in ('IMAGE', 'FLAGS', 'VARIANCE', 'ZODI')]
        hd = h['IMAGE'].header; ny, nx = im.shape
        w = WCS(h[1].header, h, fix=False, key='W')
        lam = u.Quantity(w.pixel_to_world([0, nx-1, 0, nx-1, nx//2], [0, 0, ny-1, ny-1, ny//2])[0]).to_value('um')
    bad = (fl & BAD) != 0; src = ((fl >> 21) & 1).astype(bool)
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    r, s = (im - zodi)[ok], np.sqrt(var[ok]); good = ok.sum() > 100
    med = np.median(r) if good else np.nan
    out = dict(local_file=fn, mjd_avg=hd.get('MJD-AVG'), telapse=hd.get('TELAPSE'), finast=hd.get('FINAST'),
               dqa=hd.get('L2DQAFLG'), n_transient=hd.get('L2 N_TRANSIENT'), epa=hd.get('SPS_EPA'),
               lam_center=lam[4], lam_min=lam.min(), lam_max=lam.max(),
               bad_frac=bad.mean(), src_frac=src.mean(), ok_frac=ok.mean(),
               zodi_med=float(np.nanmedian(zodi)), bkg_med=float(med),
               noise_mad=float(1.4826*np.median(np.abs(r - med))) if good else np.nan,
               noise_model=float(np.median(s)) if good else np.nan)
    for g, bits in GROUPS.items(): out[g + '_frac'] = float(((fl & mask_of(bits)) != 0).mean())
    return out

M = pd.read_csv(f'{ROOT}/data/manifests/master_manifest.csv'); M = M[M.status.isin(['ok', 'cached'])]
with ThreadPoolExecutor(4) as ex:
    rows = list(tqdm(ex.map(scan, M.local_file.tolist()), total=len(M)))
Q = M.merge(pd.DataFrame(rows), on='local_file')

# 2. the pre-registered gate
Q['pass_gate'] = ((Q.dqa == 'Pass') & (Q.finast == 0) & (Q.bad_frac <= 0.25) &
                  (Q.src_frac <= 0.90) & (Q.coverage >= 0.90))
print(Q.groupby(['field', 'role']).agg(n=('pass_gate', 'size'), passed=('pass_gate', 'sum')))
print(Q.groupby(['field', 'detector']).agg(pass_rate=('pass_gate', 'mean'), src=('src_frac', 'mean'),
                                           bad=('bad_frac', 'mean')).round(2).to_string())
Q.to_csv(f'{ROOT}/data/manifests/qc_manifest.csv', index=False)

elon240 {'Regulus': 90.17, 'Antares': 10.77, 'Spica': 36.21, 'Aldebaran': 168.8}
elon150 {'Regulus': 0.5, 'Antares': 99.73, 'Spica': 53.87, 'Aldebaran': 80.26}
elon030 {'Regulus': 119.83, 'Antares': 140.02, 'Spica': 173.51, 'Aldebaran': 40.1}


  0%|          | 0/386 [00:00<?, ?it/s]

                      n  passed
field   role                   
elon030 test        116     101
elon150 validation  131      96
elon240 train       139     118
                  pass_rate   src   bad
field   detector                       
elon030 D1             0.90  0.17  0.03
        D2             0.84  0.23  0.03
        D3             0.80  0.24  0.06
        D4             0.90  0.29  0.06
        D5             0.84  0.33  0.03
        D6             0.89  0.16  0.03
elon150 D1             0.42  0.68  0.02
        D2             0.53  0.57  0.03
        D3             0.80  0.37  0.07
        D4             0.92  0.42  0.05
        D5             0.88  0.37  0.03
        D6             0.83  0.21  0.03
elon240 D1             0.83  0.56  0.03
        D2             0.80  0.70  0.02
        D3             0.90  0.69  0.05
        D4             0.83  0.71  0.04
        D5             0.80  0.72  0.02
        D6             0.95  0.47  0.03


In [ ]:
# Step 5a: known-asteroid ground truth from SkyBoT
import time, warnings, numpy as np, pandas as pd
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.imcce import Skybot
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

Q = pd.read_csv(f'{ROOT}/data/manifests/qc_manifest.csv')
Q = Q[Q.pass_gate].copy()
Q['t_key'] = Q.mjd_avg.round(3)                 # detectors of one exposure share a time
RAD = 0.37 * u.deg                              # half-diagonal of a 0.5 deg cutout is 0.354

def movers_at(ra, dec, mjd, tries=3):
    for k in range(tries):
        try:
            t = Skybot.cone_search(SkyCoord(ra, dec, unit='deg'), RAD,
                                   Time(mjd, format='mjd'), location='500')
            return t.to_pandas()
        except Exception as e:
            if 'No solar system object' in str(e):
                return pd.DataFrame()            # empty cone, not an error
            time.sleep(3 * (k + 1))
    return None                                  # real failure; retry later

# 1. one SkyBoT query per (field, exposure time)
epochs = Q.groupby(['field', 't_key']).mjd_avg.mean().reset_index()
cache = {}
for r in tqdm(epochs.itertuples(), total=len(epochs)):
    f = fields[r.field]
    cache[(r.field, r.t_key)] = movers_at(f['ra'], f['dec'], r.mjd_avg)
print('failed queries:', sum(v is None for v in cache.values()))

# 2. project every mover into every image's pixel grid
rows = []
for r in tqdm(Q.itertuples(), total=len(Q)):
    mv = cache.get((r.field, r.t_key))
    if mv is None or mv.empty: continue
    with fits.open(r.local_file, memmap=False) as h:
        w = WCS(h['IMAGE'].header); ny, nx = h['IMAGE'].data.shape
    x, y = w.world_to_pixel(SkyCoord(mv.RA.values, mv.DEC.values, unit='deg'))
    inside = (x >= 3) & (x <= nx - 4) & (y >= 3) & (y <= ny - 4)
    for i in np.where(inside)[0]:
        rows.append(dict(field=r.field, role=r.role, local_file=r.local_file, obs_id=r.obs_id,
                         detector=r.detector, visit=r.visit, mjd=r.mjd_avg, lam=r.lam_center,
                         name=str(mv.Name.iloc[i]), V=mv.V.iloc[i], x=float(x[i]), y=float(y[i]),
                         dra_ash=mv.RA_rate.iloc[i], ddec_ash=mv.DEC_rate.iloc[i]))
G = pd.DataFrame(rows)
G.to_csv(f'{ROOT}/data/manifests/gt_movers.csv', index=False)

# 3. what did we get?
print(G.groupby('field').agg(frames=('local_file', 'nunique'), objects=('name', 'nunique'),
                             bright_V19=('V', lambda s: int((s < 19).sum())),
                             median_V=('V', 'median')))
rate = np.hypot(G.dra_ash, G.ddec_ash)                      # arcsec/hour
print('motion per hour (arcsec): median', round(rate.median(), 1), '| per pixel-hour (~6.2"):', round(rate.median() / 6.2, 2))

  0%|          | 0/169 [00:00<?, ?it/s]

failed queries: 0


  0%|          | 0/315 [00:00<?, ?it/s]

         frames  objects  bright_V19   median_V
field                                          
elon030     101      418         108  22.629999
elon150      96      360          16  22.799999
elon240     118      393          77  22.620001
motion per hour (arcsec): median 23.5 | per pixel-hour (~6.2"): 3.78


In [ ]:
# Step 5b: S/N at known-mover positions vs random control positions
import numpy as np, pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm

G = pd.read_csv(f'{ROOT}/data/manifests/gt_movers.csv')
BADMASK = sum(1 << b for b in [0,1,2,6,7,9,10,11,14,15,17,19,22,24,26,27,28,29])
R_AP, R_IN, R_OUT = 2.0, 5.0, 9.0          # aperture and sky annulus radii, in pixels
rng = np.random.default_rng(0)

def measure(im, var, bad, x, y):
    ny, nx = im.shape
    yy, xx = np.ogrid[:ny, :nx]
    d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~bad & np.isfinite(im)
    if an.sum() < 30 or ap.sum() == 0: return np.nan, np.nan
    bkg = np.median(im[an])
    ok = ap & ~bad & np.isfinite(im) & np.isfinite(var) & (var > 0)
    if ok.sum() < 0.6 * ap.sum(): return np.nan, np.nan          # too many bad pixels in aperture
    flux = (im[ok] - bkg).sum() * ap.sum() / ok.sum()
    sig = np.sqrt(var[ok].sum()) * np.sqrt(ap.sum() / ok.sum())
    sky_sd = 1.4826 * np.median(np.abs(im[an] - bkg))            # empirical sky scatter
    noise = np.hypot(sig, sky_sd * np.sqrt(ap.sum()))
    return flux, flux / noise

rows = []
for fn, g in tqdm(G.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = (h['IMAGE'].data - h['ZODI'].data).astype(float)
        var = h['VARIANCE'].data.astype(float)
        bad = (h['FLAGS'].data & BADMASK) != 0
    ny, nx = im.shape
    for r in g.itertuples():
        f, s = measure(im, var, bad, r.x, r.y)
        for _ in range(3):                                       # 3 random controls per mover
            while True:
                cx, cy = rng.uniform(10, nx - 10), rng.uniform(10, ny - 10)
                if np.hypot(cx - r.x, cy - r.y) > 12: break
            cf, cs = measure(im, var, bad, cx, cy)
            rows.append(dict(kind='control', V=r.V, snr=cs, lam=r.lam, detector=r.detector, field=r.field))
        rows.append(dict(kind='mover', V=r.V, snr=s, lam=r.lam, detector=r.detector, field=r.field,
                         local_file=fn, name=r.name, x=r.x, y=r.y, flux=f))
S = pd.DataFrame(rows)
S.to_csv(f'{ROOT}/data/manifests/gt_snr.csv', index=False)

S['Vbin'] = pd.cut(S.V, [0, 17, 18, 19, 20, 21, 22, 30])
print(S.groupby(['Vbin', 'kind']).snr.agg(n='count', median='median',
      frac_gt3=lambda s: (s > 3).mean(), frac_gt5=lambda s: (s > 5).mean()).round(2).to_string())

  0%|          | 0/315 [00:00<?, ?it/s]

                      n  median  frac_gt3  frac_gt5
Vbin     kind                                      
(0, 17]  control      0     NaN       NaN       NaN
         mover        0     NaN       NaN       NaN
(17, 18] control    174   -0.15      0.07      0.04
         mover       56    1.06      0.09      0.00
(18, 19] control    473    0.09      0.13      0.09
         mover      159    0.80      0.19      0.11
(19, 20] control   1259    0.05      0.11      0.08
         mover      421    0.35      0.11      0.05
(20, 21] control   3116    0.04      0.10      0.07
         mover     1039    0.17      0.09      0.06
(21, 22] control   7005    0.02      0.11      0.07
         mover     2335    0.04      0.09      0.06
(22, 30] control  32059    0.03      0.10      0.07
         mover    10695    0.08      0.11      0.08


In [ ]:
# Step 5c: source-split S/N, offset scan, track-level S/N (existing data only)
import numpy as np, pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm

G = pd.read_csv(f'{ROOT}/data/manifests/gt_movers.csv')
BADMASK = sum(1 << b for b in [0,1,2,6,7,9,10,11,14,15,17,19,22,24,26,27,28,29])
R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11
rng = np.random.default_rng(0)

def meas(im, var, bad, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny: return None
    I, V, B, S = (a[ya:yb, xa:xb] for a in (im, var, bad, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~B & np.isfinite(I)
    ok = ap & ~B & np.isfinite(I) & np.isfinite(V) & (V > 0)
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum(): return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return flux, noise, bool(S[ap].any())

# one fixed set of 3 control shifts per (field, visit, object), reused for every frame of that track
G['key'] = G.field + '|' + G.visit.astype(str) + '|' + G.name.astype(str)
ctrl = {}
for k in G.key.unique():
    ctrl[k] = [(r * np.cos(a), r * np.sin(a)) for r, a in
               zip(rng.uniform(15, 40, 3), rng.uniform(0, 2 * np.pi, 3))]

OFFS = [(dx, dy) for dx in range(-8, 9, 2) for dy in range(-8, 9, 2)]
rows, offs = [], []
for fn, g in tqdm(G.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = (h['IMAGE'].data - h['ZODI'].data).astype(float)
        var = h['VARIANCE'].data.astype(float); fl = h['FLAGS'].data
    bad = (fl & BADMASK) != 0; src = ((fl >> 21) & 1).astype(bool)
    for r in g.itertuples():
        m = meas(im, var, bad, src, r.x, r.y)
        if m is None: continue
        rows.append(dict(key=r.key, kind='mover', V=r.V, flux=m[0], noise=m[1], on_src=m[2]))
        for c, (dx, dy) in enumerate(ctrl[r.key]):
            mc = meas(im, var, bad, src, r.x + dx, r.y + dy)
            if mc: rows.append(dict(key=r.key, kind=f'ctrl{c}', V=r.V, flux=mc[0], noise=mc[1], on_src=mc[2]))
        if r.V < 20:
            for dx, dy in OFFS:
                mo = meas(im, var, bad, src, r.x + dx, r.y + dy)
                if mo: offs.append((dx, dy, mo[0] / mo[1]))

R = pd.DataFrame(rows); R['snr'] = R.flux / R.noise; R['grp'] = np.where(R.kind == 'mover', 'mover', 'control')
R.to_csv(f'{ROOT}/data/manifests/gt_snr_v2.csv', index=False)

# 1. frame level, split by whether the aperture touches SOURCE-flagged pixels
R['Vbin'] = pd.cut(R.V, [0, 19, 20, 22, 30])
print('--- 1. frame level S/N, split by SOURCE flag ---')
print(R.groupby(['Vbin', 'on_src', 'grp']).snr.agg(n='count', median='median',
      gt3=lambda s: (s > 3).mean()).round(2).to_string())

# 2. offset scan (mean S/N of V<20 movers on a pixel grid around the predicted position)
O = pd.DataFrame(offs, columns=['dx', 'dy', 'snr'])
print('\n--- 2. mean S/N vs offset (rows dy, cols dx; pixels) ---')
print(O.pivot_table(index='dy', columns='dx', values='snr', aggfunc='mean').round(2).to_string())

# 3. track level: sum flux over frames of one visit and one object
def tracks(d):
    t = d.groupby('key').agg(n=('flux', 'size'), f=('flux', 'sum'), v=('noise', lambda s: (s ** 2).sum()),
                             V=('V', 'median'))
    t['snr'] = t.f / np.sqrt(t.v); return t[t.n >= 4]
tm = tracks(R[R.kind == 'mover']); tm['grp'] = 'mover'
tc = pd.concat([tracks(R[R.kind == f'ctrl{c}']) for c in range(3)]); tc['grp'] = 'control'
T = pd.concat([tm, tc]); T['Vbin'] = pd.cut(T.V, [0, 19, 20, 21, 22, 30])
print('\n--- 3. track level S/N (tracks with >= 4 frames) ---')
print(T.groupby(['Vbin', 'grp']).snr.agg(n='count', median='median',
      gt3=lambda s: (s > 3).mean(), gt5=lambda s: (s > 5).mean()).round(2).to_string())

  0%|          | 0/315 [00:00<?, ?it/s]

--- 1. frame level S/N, split by SOURCE flag ---
                             n  median   gt3
Vbin     on_src grp                         
(0, 19]  False  control    137   -0.24  0.00
                mover       64    0.52  0.00
         True   control    349    0.26  0.14
                mover      110    1.16  0.27
(19, 20] False  control    358   -0.31  0.00
                mover      133    0.01  0.00
         True   control    785    0.20  0.14
                mover      274    0.65  0.17
(20, 22] False  control   3010   -0.32  0.00
                mover     1159   -0.14  0.00
         True   control   4996    0.32  0.17
                mover     1848    0.32  0.15
(22, 30] False  control   9156   -0.29  0.00
                mover     3438   -0.30  0.00
         True   control  15457    0.32  0.16
                mover     6026    0.41  0.18

--- 2. mean S/N vs offset (rows dy, cols dx; pixels) ---
dx    -8    -6    -4    -2     0     2     4     6     8
dy                        

In [ ]:
# Step 5d: robust offset check (median, clean pixels only, finer grid)
Gb = G[G.V < 19.5]
OFFS = [(dx, dy) for dx in range(-6, 7) for dy in range(-6, 7)]
vals = {o: [] for o in OFFS}
for fn, g in tqdm(Gb.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = (h['IMAGE'].data - h['ZODI'].data).astype(float)
        var = h['VARIANCE'].data.astype(float); fl = h['FLAGS'].data
    bad = (fl & BADMASK) != 0; src = ((fl >> 21) & 1).astype(bool)
    for r in g.itertuples():
        for dx, dy in OFFS:
            m = meas(im, var, bad, src, r.x + dx, r.y + dy)
            if m and not m[2]:                       # clean pixels only
                vals[(dx, dy)].append(m[0] / m[1])

M = pd.DataFrame([(dx, dy, np.median(v), len(v)) for (dx, dy), v in vals.items() if len(v) >= 20],
                 columns=['dx', 'dy', 'med', 'n'])
print(M.pivot(index='dy', columns='dx', values='med').round(2).to_string())
b = M.loc[M.med.idxmax()]
c = M[(M.dx == 0) & (M.dy == 0)].med.iloc[0]
print(f'peak at dx={b.dx:.0f}, dy={b.dy:.0f}: median S/N {b.med:.2f} (n={b.n:.0f}) | at (0,0): {c:.2f}')

  0%|          | 0/203 [00:00<?, ?it/s]

dx    -6    -5    -4    -3    -2    -1     0     1     2     3     4     5     6
dy                                                                              
-6 -0.24 -0.38 -0.27 -0.36 -0.44 -0.36 -0.18 -0.20 -0.15 -0.12 -0.24 -0.37 -0.26
-5 -0.30 -0.24 -0.24 -0.34 -0.35 -0.39 -0.22 -0.41 -0.24 -0.26 -0.35 -0.45 -0.46
-4 -0.35 -0.26 -0.06 -0.17 -0.32 -0.32 -0.30 -0.36 -0.41 -0.36 -0.38 -0.35 -0.35
-3 -0.20 -0.12 -0.07 -0.22 -0.28 -0.22 -0.17 -0.45 -0.40 -0.34 -0.50 -0.42 -0.23
-2 -0.32 -0.24 -0.24 -0.18 -0.16 -0.15  0.05 -0.20 -0.43 -0.34 -0.49 -0.32 -0.23
-1 -0.42 -0.27 -0.36 -0.12 -0.20  0.18  0.25  0.14 -0.15 -0.22 -0.44 -0.37 -0.36
 0 -0.40 -0.39 -0.39 -0.37 -0.17  0.18  0.18  0.13 -0.10 -0.39 -0.35 -0.36 -0.29
 1 -0.30 -0.43 -0.29 -0.35 -0.27  0.02  0.25 -0.02 -0.03 -0.26 -0.20 -0.26 -0.24
 2 -0.19 -0.16 -0.36 -0.32 -0.41 -0.25 -0.09 -0.05 -0.19 -0.38 -0.26 -0.27 -0.25
 3 -0.01 -0.30 -0.09 -0.25 -0.40 -0.35 -0.22 -0.27 -0.26 -0.28 -0.25 -0.32 -0.28
 4 -0.23 -0.13 -0.21 -0.34 -

In [ ]:
# Setup: restores the helpers lost with the old session (run before Step 6)
import os, numpy as np, pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm

BADMASK = sum(1 << b for b in [0,1,2,6,7,9,10,11,14,15,17,19,22,24,26,27,28,29])
R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11

def meas(im, var, bad, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny: return None
    I, V, B, S = (a[ya:yb, xa:xb] for a in (im, var, bad, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~B & np.isfinite(I)
    ok = ap & ~B & np.isfinite(I) & np.isfinite(V) & (V > 0)
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum(): return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return flux, noise, bool(S[ap].any())

# ground-truth movers + track keys + the same 3 control shifts as in step 5c (same seed, same order)
G = pd.read_csv(f'{ROOT}/data/manifests/gt_movers.csv')
G['key'] = G.field + '|' + G.visit.astype(str) + '|' + G.name.astype(str)
rng = np.random.default_rng(0)
ctrl = {}
for k in G.key.unique():
    ctrl[k] = [(r * np.cos(a), r * np.sin(a)) for r, a in
               zip(rng.uniform(15, 40, 3), rng.uniform(0, 2 * np.pi, 3))]

Q = pd.read_csv(f'{ROOT}/data/manifests/qc_manifest.csv')
Q = Q[Q.pass_gate].copy()
print('ready:', len(G), 'mover-frame rows,', len(ctrl), 'tracks,', len(Q), 'passing frames')

ready: 14854 mover-frame rows, 1330 tracks, 315 passing frames


In [ ]:
# Step 6: track table, image-only vs +metadata ranker, leave-one-field-out, JSON export
import json, numpy as np, pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

print(list(Q.columns))
META = [c for c in ['bad_frac', 'src_frac', 'noise_mad', 'ghost_frac', 'persist_frac',
                    'cr_frac', 'streak_frac', 'halo_frac'] if c in Q.columns]
print('metadata columns used:', META)

# 1. per-frame measurements at the predicted track (mover) and at the 3 control tracks
rows = []
for fn, g in tqdm(G.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = (h['IMAGE'].data - h['ZODI'].data).astype(float)
        var = h['VARIANCE'].data.astype(float); fl = h['FLAGS'].data
    bad = (fl & BADMASK) != 0; src = ((fl >> 21) & 1).astype(bool)
    for r in g.itertuples():
        for kind, (dx, dy) in [('mover', (0, 0))] + [(f'ctrl{c}', s) for c, s in enumerate(ctrl[r.key])]:
            m = meas(im, var, bad, src, r.x + dx, r.y + dy)
            if m: rows.append(dict(local_file=fn, key=r.key, kind=kind, field=r.field, V=r.V, lam=r.lam,
                                   det=r.detector, mjd=r.mjd, flux=m[0], noise=m[1], on_src=m[2]))
F = pd.DataFrame(rows).merge(Q[['local_file'] + META].drop_duplicates('local_file'), on='local_file', how='left')

# 2. one row per track
def feats(d):
    s = d.flux / d.noise
    return pd.Series(dict(n=len(d), track_snr=d.flux.sum() / np.sqrt((d.noise ** 2).sum()),
        med_snr=s.median(), frac_gt1=(s > 1).mean(), src_frac_ap=d.on_src.mean(),
        lam_mean=d.lam.mean(), lam_span=d.lam.max() - d.lam.min(), n_det=d.det.nunique(),
        span_h=(d.mjd.max() - d.mjd.min()) * 24,
        **{f'm_{c}': d[c].mean() for c in META}, V=d.V.median(), field=d.field.iloc[0]))
T = F.groupby(['key', 'kind']).apply(feats).reset_index()
T = T[T.n >= 4]
T['y'] = ((T.kind == 'mover') & (T.V <= 20)).astype(int)
T = T[(T.kind != 'mover') | (T.V <= 20)].reset_index(drop=True)   # drop faint movers (not labelable)
T['snr_x_src'] = T.track_snr * T.src_frac_ap
if 'm_bad_frac' in T: T['snr_x_bad'] = T.track_snr * T.m_bad_frac
print(T.groupby('field').y.agg(positives='sum', negatives=lambda s: int((s == 0).sum())))

IMG = ['track_snr', 'med_snr', 'frac_gt1', 'n']
MET = IMG + ['src_frac_ap', 'lam_mean', 'lam_span', 'n_det', 'span_h', 'snr_x_src'] \
      + [f'm_{c}' for c in META] + (['snr_x_bad'] if 'snr_x_bad' in T else [])

def fit(Xtr, ytr, C=0.5):
    med = Xtr.median(); sc = StandardScaler().fit(Xtr.fillna(med))
    lr = LogisticRegression(C=C, class_weight='balanced', max_iter=2000).fit(sc.transform(Xtr.fillna(med)), ytr)
    return med, sc, lr

def oof(cols):                                   # leave-one-field-out predictions
    p = np.zeros(len(T))
    for f in T.field.unique():
        tr, te = (T.field != f).values, (T.field == f).values
        med, sc, lr = fit(T.loc[tr, cols], T.y[tr])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

def report(p, y, B=500):
    def one(p, y):
        thr = np.quantile(p[y == 1], 0.2)        # keeps 80% of real movers
        return roc_auc_score(y, p), (p[y == 0] >= thr).mean()
    base = one(p, y); rng = np.random.default_rng(0); bs = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() >= 3: bs.append(one(p[i], y[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    return f'AUC {base[0]:.2f} [{lo[0]:.2f}-{hi[0]:.2f}] | false alerts @80% recall {base[1]:.3f} [{lo[1]:.3f}-{hi[1]:.3f}]'

y = T.y.values
print('\nrule: track S/N only      ', report(T.track_snr.values, y))
print('image-only ranker (LOFO)  ', report(oof(IMG), y))
print('image + metadata (LOFO)   ', report(oof(MET), y))

# 3. export for the backend: plain JSON, scored with numpy only (50 bootstrap models -> uncertainty)
med, sc, _ = fit(T[MET], T.y)
Z = sc.transform(T[MET].fillna(med)); rng = np.random.default_rng(1); coefs, ints = [], []
while len(coefs) < 50:
    i = rng.integers(0, len(T), len(T))
    if T.y.values[i].sum() < 3: continue
    lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=2000).fit(Z[i], T.y.values[i])
    coefs.append(lr.coef_[0].tolist()); ints.append(float(lr.intercept_[0]))
os.makedirs(f'{ROOT}/models', exist_ok=True)
json.dump(dict(version='ranker_v1', features=MET, fill=med.to_dict(), mean=sc.mean_.tolist(),
               scale=sc.scale_.tolist(), coefs=coefs, intercepts=ints),
          open(f'{ROOT}/models/ranker_v1.json', 'w'))
T.to_csv(f'{ROOT}/data/manifests/track_table.csv', index=False)

def score(feat: dict, m=None):                   # what the backend will run (numpy only)
    m = m or json.load(open(f'{ROOT}/models/ranker_v1.json'))
    x = (np.array([feat.get(k, m['fill'][k]) for k in m['features']]) - m['mean']) / m['scale']
    p = 1 / (1 + np.exp(-(np.array(m['coefs']) @ x + np.array(m['intercepts']))))
    return dict(score=float(p.mean()), uncertainty=float(p.std()))
print('\nexample:', score(T.loc[T.y == 1, MET].iloc[0].to_dict()))

['obs_id', 't_min', 't_max', 'energy_bandpassname', 'em_min', 'em_max', 'access_url', 'access_estsize', 'visit', 'n_in_visit', 'visit_hours', 'field', 'exposure', 'detector', 'role', 'path', 'local_file', 'status', 'ny', 'nx', 'nan_frac', 'flagged_frac', 'n_ext', 'mb', 'coverage', 'mjd_avg', 'telapse', 'finast', 'dqa', 'n_transient', 'epa', 'lam_center', 'lam_min', 'lam_max', 'bad_frac', 'src_frac', 'ok_frac', 'zodi_med', 'bkg_med', 'noise_mad', 'noise_model', 'cr_frac', 'streak_frac', 'ghost_frac', 'persist_frac', 'dichroic_frac', 'bloom_frac', 'badpix_frac', 'pass_gate']
metadata columns used: ['bad_frac', 'src_frac', 'noise_mad', 'ghost_frac', 'persist_frac', 'cr_frac', 'streak_frac']


  0%|          | 0/315 [00:00<?, ?it/s]

         positives  negatives
field                        
elon030         23       1083
elon150         10       1025
elon240         18       1041

rule: track S/N only       AUC 0.63 [0.57-0.70] | false alerts @80% recall 0.547 [0.458-0.687]
image-only ranker (LOFO)   AUC 0.60 [0.51-0.66] | false alerts @80% recall 0.653 [0.526-0.829]
image + metadata (LOFO)    AUC 0.54 [0.45-0.62] | false alerts @80% recall 0.762 [0.600-0.894]

example: {'score': 0.7464158835858733, 'uncertainty': 0.08236831975119095}


In [ ]:
# Step A1: feasibility check, no downloads. How many SPHEREx frames cover bright known asteroids?
import numpy as np, pandas as pd, pyvo, os
from astropy.table import Table
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from scipy.interpolate import CubicSpline
from tqdm.auto import tqdm

TAP = pyvo.dal.TAPService('https://irsa.ipac.caltech.edu/TAP')
AST = [6, 7, 8, 9, 14, 15, 18, 19, 20, 29, 40, 43, 44, 63, 89, 129, 192, 349,     # numbered main-belt asteroids,
       85, 93, 105, 120, 147, 173, 217, 238]                                       # mixed sizes; V comes from Horizons
VMIN, VMAX = 9.0, 17.0      # "usable" magnitude window: my guess, we tune it from measured S/N later

# 0. time span of SPHEREx data in the archive
span = TAP.run_sync("SELECT MIN(t_min) AS a, MAX(t_max) AS b FROM spherex.obscore").to_table()
t0, t1 = float(span['a'][0]), float(span['b'][0])
print(f'archive: MJD {t0:.1f} to {t1:.1f} ({t1 - t0:.0f} days)')

# 1. daily ephemerides (geocentric) with V magnitude and solar elongation
d0, d1 = Time(t0 - 1, format='mjd').iso[:10], Time(t1 + 1, format='mjd').iso[:10]
eph, fails = {}, []
for n in tqdm(AST):
    try:
        e = Horizons(id=str(n), id_type='smallbody', location='500',
                     epochs=dict(start=d0, stop=d1, step='1d')).ephemerides(quantities='1,9,23').to_pandas()
        e['mjd'] = e.datetime_jd - 2400000.5
        eph[n] = e[['mjd', 'RA', 'DEC', 'V', 'elong']].dropna().reset_index(drop=True)
    except Exception as ex:
        fails.append((n, str(ex)[:80]))
print('ephemerides ok:', len(eph), '| failed:', fails)

spl = {n: (CubicSpline(e.mjd, np.degrees(np.unwrap(np.radians(e.RA.values)))),
           CubicSpline(e.mjd, e.DEC.values), CubicSpline(e.mjd, e.V.values),
           CubicSpline(e.mjd, e.elong.values)) for n, e in eph.items()}

# 2. coarse match on the server: exposures within 0.5 deg of the asteroid's daily position, same day
up = Table({'ast': np.concatenate([[n] * len(e) for n, e in eph.items()]).astype('int32'),
            'emjd': np.concatenate([e.mjd.values for e in eph.values()]),
            'era': np.concatenate([e.RA.values for e in eph.values()]),
            'edec': np.concatenate([e.DEC.values for e in eph.values()])})
q1 = """SELECT o.obs_id, o.t_min, o.t_max, o.access_url, e.ast
        FROM spherex.obscore AS o JOIN TAP_UPLOAD.eph AS e
        ON 1=INTERSECTS(CIRCLE('ICRS', e.era, e.edec, 0.5), o.s_region)
        WHERE o.t_min <= e.emjd + 0.5 AND o.t_max >= e.emjd - 0.5"""
res1 = TAP.run_async(q1, uploads={'eph': up}).to_table().to_pandas()
R = res1.drop_duplicates(['ast', 'obs_id', 'access_url']).copy()
print('coarse candidates:', len(R))

# 3. exact position of each asteroid at its exposure time, then check it really falls on a detector
R['mjd'] = (R.t_min + R.t_max) / 2
R['ra'] = [float(spl[a][0](m)) % 360 for a, m in zip(R.ast, R.mjd)]
R['dec'] = [float(spl[a][1](m)) for a, m in zip(R.ast, R.mjd)]
R['V'] = [float(spl[a][2](m)) for a, m in zip(R.ast, R.mjd)]
R['elong'] = [float(spl[a][3](m)) for a, m in zip(R.ast, R.mjd)]

C = R.drop_duplicates(['obs_id', 'ast'])
chk = Table({'obs_id': C.obs_id.astype(str).values, 'ast': C.ast.values.astype('int32'),
             'cra': C.ra.values, 'cdec': C.dec.values})
q2 = """SELECT o.obs_id, o.access_url, c.ast
        FROM spherex.obscore AS o JOIN TAP_UPLOAD.chk AS c ON o.obs_id = c.obs_id
        WHERE 1=CONTAINS(POINT('ICRS', c.cra, c.cdec), o.s_region)"""
ins = TAP.run_async(q2, uploads={'chk': chk}).to_table().to_pandas().drop_duplicates(['ast', 'obs_id', 'access_url'])
F = R.merge(ins, on=['ast', 'obs_id', 'access_url'])           # frames that truly contain the asteroid
F = F.sort_values(['ast', 'mjd']).reset_index(drop=True)
F['visit'] = (F.groupby('ast').mjd.diff().fillna(99) > 0.5).cumsum()      # new visit after a >12 h gap
F.to_csv(f'{ROOT}/data/manifests/feasibility_frames.csv', index=False)

# 4. how much usable material is there?
U = F[(F.V >= VMIN) & (F.V <= VMAX)]
tr = U.groupby('visit').agg(ast=('ast', 'first'), n_exp=('obs_id', 'nunique'), V=('V', 'median'),
                            hours=('mjd', lambda s: (s.max() - s.min()) * 24))
tr4 = tr[tr.n_exp >= 4]
summ = U.groupby('ast').agg(frames=('obs_id', 'nunique'), V_med=('V', 'median'), elong=('elong', 'median'))
summ['tracks_ge4'] = tr4.groupby('ast').size()
print(summ.fillna(0).round(1).to_string())

mb = pd.read_csv(f'{ROOT}/data/manifests/qc_manifest.csv').mb.mean()
print(f'\nall covering frames: {F.obs_id.nunique()} exposures | in V window {VMIN}-{VMAX}: {U.obs_id.nunique()}')
print(f'tracks with >=4 usable frames: {len(tr4)} (median {tr4.n_exp.median():.0f} frames, '
      f'median span {tr4.hours.median():.1f} h) | your current data: ~16 at V<=19, ~35 at V 19-20')
print(f'approx download if we take them all: {U.shape[0] * mb / 1000:.1f} GB at {mb:.1f} MB per cutout')

archive: MJD 60789.8 to 61318.0 (528 days)


  0%|          | 0/26 [00:00<?, ?it/s]

ephemerides ok: 26 | failed: []


DALServiceError: HTTPSConnectionPool(host='irsa.ipac.caltech.edu', port=443): Read timed out. (read timeout=10)

In [ ]:
# Step A1 (light): small point queries, only days that matter, run in parallel
import time, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from astroquery.ipac.irsa import Irsa
from tqdm.auto import tqdm

ids = list(eph)[:12]                                  # start with 12 asteroids; extend later if needed
days = []
for n in ids:
    e = eph[n]
    ok = e[e.elong.between(70, 110) & e.V.between(VMIN - 1, VMAX + 1)]
    days += [(n, r.mjd, r.RA, r.DEC) for r in ok.itertuples()]
print(len(days), 'asteroid-days to query')

def one(d, tries=3):
    n, m, ra, dec = d
    q = f"""SELECT obs_id, t_min, t_max, access_url FROM spherex.obscore
            WHERE CONTAINS(POINT('ICRS',{ra:.5f},{dec:.5f}), s_region)=1
              AND t_min >= {m - 0.5:.4f} AND t_min < {m + 0.5:.4f}"""
    for k in range(tries):
        try:
            t = Irsa.query_tap(q).to_table().to_pandas(); t['ast'] = n; return t
        except Exception:
            time.sleep(2 * (k + 1))
    return None

t = time.time(); test = one(days[0])                  # time ONE query first
print(f'one query: {time.time() - t:.1f}s, rows: {None if test is None else len(test)}')
print(f'estimated total with 5 workers: {len(days) * (time.time() - t) / 5 / 60:.0f} min')

1407 asteroid-days to query
one query: 7.7s, rows: 0
estimated total with 5 workers: 36 min


In [ ]:
# run all queries (5 in parallel), then build the frame table
out, bad = [], 0
with ThreadPoolExecutor(5) as ex:
    futs = [ex.submit(one, d) for d in days]
    for f in tqdm(as_completed(futs), total=len(futs)):
        r = f.result()
        if r is None: bad += 1
        elif len(r): out.append(r)
print('failed queries:', bad)

R = pd.concat(out).drop_duplicates(['ast', 'obs_id', 'access_url']).copy()
R['mjd'] = (R.t_min + R.t_max) / 2
R['ra'] = [float(spl[a][0](m)) % 360 for a, m in zip(R.ast, R.mjd)]
R['dec'] = [float(spl[a][1](m)) for a, m in zip(R.ast, R.mjd)]
R['V'] = [float(spl[a][2](m)) for a, m in zip(R.ast, R.mjd)]
R['elong'] = [float(spl[a][3](m)) for a, m in zip(R.ast, R.mjd)]
F = R.sort_values(['ast', 'mjd']).reset_index(drop=True)
F['visit'] = (F.groupby('ast').mjd.diff().fillna(99) > 0.5).cumsum()
F.to_csv(f'{ROOT}/data/manifests/feasibility_frames.csv', index=False)

U = F[(F.V >= VMIN) & (F.V <= VMAX)]
tr = U.groupby('visit').agg(ast=('ast', 'first'), n_exp=('obs_id', 'nunique'), V=('V', 'median'),
                            hours=('mjd', lambda s: (s.max() - s.min()) * 24))
tr4 = tr[tr.n_exp >= 4]
summ = U.groupby('ast').agg(frames=('obs_id', 'nunique'), V_med=('V', 'median'), elong=('elong', 'median'))
summ['tracks_ge4'] = tr4.groupby('ast').size()
print(summ.fillna(0).round(1).to_string())
mb = pd.read_csv(f'{ROOT}/data/manifests/qc_manifest.csv').mb.mean()
print(f'\ncovering exposures: {F.obs_id.nunique()} | in V window: {U.obs_id.nunique()}')
print(f'tracks with >=4 usable frames: {len(tr4)} (median {tr4.n_exp.median():.0f} frames, '
      f'median span {tr4.hours.median():.1f} h) | current data: ~16 at V<=19, ~35 at V 19-20')
print(f'elongation of returned frames: {F.elong.min():.0f} to {F.elong.max():.0f} deg')
print(f'approx download: {U.shape[0] * mb / 1000:.1f} GB at {mb:.1f} MB per cutout')

  0%|          | 0/1407 [00:00<?, ?it/s]

failed queries: 0
     frames  V_med  elong  tracks_ge4
ast                                  
6       142    9.7   93.3          20
7       115   10.3   90.5          14
8       105   11.1   96.8          11
9        90   11.1   90.6          10
14      105   11.1   96.4          15
15       98   10.9   87.8           8
18       87   11.0   91.8          13
19      104   12.6   91.3          13
20       96   10.7   91.5          12
29       97   11.1   92.5          14
40      143   11.0   90.9          19
43      153   12.8   92.5          21

covering exposures: 1307 | in V window: 1297
tracks with >=4 usable frames: 170 (median 6 frames, median span 5.0 h) | current data: ~16 at V<=19, ~35 at V 19-20
elongation of returned frames: 81 to 110 deg
approx download: 16.2 GB at 6.2 MB per cutout


In [ ]:
# Step A2 (fixed): 0.25 deg cutouts for a subset of tracks (resumable)
import io, os, json, time, threading, requests, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from astropy.io import fits
from astropy.io.votable import parse_single_table
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from tqdm.auto import tqdm
import pyvo
SIZE_DEG, CUT_URL = 0.25, 'https://irsa.ipac.caltech.edu/ibe/cutout'
LIMIT, VIS_PER_AST, WORKERS = None, 3, 2           # LIMIT = None for the full subset
OUT = f'{ROOT}/data/raw/spherex_qr2/astA/cutouts'; os.makedirs(OUT, exist_ok=True)
LOCK = threading.Lock()
CACHE_F = f'{ROOT}/data/manifests/links_cache.json'
cache = json.load(open(CACHE_F)) if os.path.exists(CACHE_F) else {}

F = pd.read_csv(f'{ROOT}/data/manifests/feasibility_frames.csv')
U = F[(F.V >= 9) & (F.V <= 17)]
A = U[U.groupby('visit').obs_id.transform('nunique') >= 4].copy()
A['detector'] = A.access_url.str.extract(r'/(D\d)$')[0]
vis = A.drop_duplicates('visit')[['ast', 'visit']]
keep = vis.groupby('ast', group_keys=False).apply(
    lambda d: d.sample(min(len(d), VIS_PER_AST), random_state=0)).visit
A = A[A.visit.isin(keep)].reset_index(drop=True)
N_FULL = len(A)
if LIMIT: A = A.sample(LIMIT, random_state=0).reset_index(drop=True)
print(f'{len(A)} cutouts now | full subset = {N_FULL} cutouts, {A.ast.nunique()} asteroids')

def resolve(access_url):
    if access_url in cache: return cache[access_url], ''
    err = ''
    for wait in (5, 20):
        try:
            d = pyvo.dal.adhoc.DatalinkResults.from_result_url(access_url).to_table().to_pandas()
            dec = lambda s: s.decode() if isinstance(s, bytes) else s
            rows = d[d.semantics.map(dec) == '#cutout']
            if len(rows):
                p = dec(rows.iloc[0]['path'])
                with LOCK: cache[access_url] = p
                return p, ''
            err = 'no #cutout row: ' + str(d.semantics.map(dec).tolist())[:80]
        except Exception as e:
            err = f'{type(e).__name__}: {str(e)[:100]}'
        time.sleep(wait)
    return None, err

def valid(fn):
    try:
        with fits.open(fn) as h:
            _ = h['IMAGE'].data; _ = h['WCS-WAVE'].data
            return len(h) == 7
    except Exception:
        return False

def get_cutout(path, ra, dec, out):
    if os.path.exists(out) and valid(out): return 'cached'
    err = ''
    for k in range(3):
        try:
            r = requests.get(CUT_URL, params=dict(path=path, ra=ra, dec=dec, size=SIZE_DEG), timeout=(15, 180))
            if r.status_code == 200 and r.content[:6] == b'SIMPLE':
                open(out + '.part', 'wb').write(r.content); os.replace(out + '.part', out)
                return 'ok' if valid(out) else 'invalid file'
            err = f'{r.status_code} {r.text[:80]}'
        except Exception as e:
            err = f'{type(e).__name__}: {str(e)[:80]}'
        time.sleep(2 * (k + 1))
    return 'FAILED: ' + err

def one(r):
    path, err = resolve(r['access_url'])
    if not path: return dict(r) | dict(path=None, local_file=None, status='link error: ' + err)
    fn = f"{OUT}/{os.path.basename(path)[:-5]}_ast{int(r['ast'])}.fits"
    st = get_cutout(path, r['ra'], r['dec'], fn)
    rec = dict(r) | dict(path=path, local_file=fn, status=st)
    if st in ('ok', 'cached'):
        with fits.open(fn, memmap=False) as h:
            w = WCS(h['IMAGE'].header); ny, nx = h['IMAGE'].data.shape
        x, y = w.world_to_pixel(SkyCoord(r['ra'], r['dec'], unit='deg'))
        rec |= dict(nx=nx, ny=ny, x=float(x), y=float(y),
                    inside=bool(12 <= x <= nx - 13 and 12 <= y <= ny - 13))
    return rec

t0 = time.time(); rows = []
try:
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(one, r) for r in A.to_dict('records')]
        for fu in tqdm(as_completed(futs), total=len(futs)): rows.append(fu.result())
finally:
    json.dump(cache, open(CACHE_F, 'w'))
M = pd.DataFrame(rows).sort_values(['ast', 'mjd']).reset_index(drop=True)
M.to_csv(f"{ROOT}/data/manifests/astA_cutouts{'_test' if LIMIT else ''}.csv", index=False)

el = time.time() - t0
print(f'{el:.0f}s | statuses:', M.status.str.slice(0, 140).value_counts().to_dict())
if 'inside' in M: print('asteroid inside with margin:', int(M.inside.sum()), 'of', len(M))
if 'nx' in M: print(M[['ast', 'detector', 'status', 'nx', 'ny', 'x', 'y', 'inside']].head(8).to_string())
print(f'estimated time for the full subset: {el / len(A) * N_FULL / 60:.0f} min')

599 cutouts now | full subset = 599 cutouts, 12 asteroids


/tmp/ipykernel_2900/1636101136.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep = vis.groupby('ast', group_keys=False).apply(


  0%|          | 0/599 [00:00<?, ?it/s]

1009s | statuses: {'ok': 344, 'cached': 255}
asteroid inside with margin: 573 of 599
   ast detector  status   nx   ny          x          y  inside
0    6       D4  cached  146  146  72.453691  72.192588    True
1    6       D1  cached  146  146  72.359299  72.512948    True
2    6       D4  cached  146  146  72.383407  72.181340    True
3    6       D1  cached  146  146  72.531282  72.412543    True
4    6       D1  cached  146  146  72.110397  72.265073    True
5    6       D4  cached  146  146  72.770013  72.154904    True
6    6       D1  cached  146  146  72.380665  72.125750    True
7    6       D4  cached  146  146  72.841824  72.132454    True
estimated time for the full subset: 17 min


In [ ]:
# Step A3: QC -> photometry -> track table -> comparison (leave-one-asteroid-out) -> export
import logging, warnings, os, json
logging.getLogger('astropy').setLevel(logging.ERROR); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ---- 1. QC: same scan and gate as your first dataset -------------------------------------------
def mask_of(bits): return sum(1 << b for b in bits)
BAD = mask_of([0,1,2,6,7,9,10,11,14,15,17,19,22,24,26,27,28,29])
GROUPS = {'cr':[0,1,27,28], 'streak':[19,29], 'ghost':[22,24], 'persist':[17],
          'dichroic':[7], 'bloom':[26], 'badpix':[6,9,10,11,14,15]}

def scan(fn):
    with fits.open(fn, memmap=False) as h:
        im, fl, var, zodi = [h[k].data for k in ('IMAGE', 'FLAGS', 'VARIANCE', 'ZODI')]
        hd = h['IMAGE'].header; ny, nx = im.shape
        w = WCS(h[1].header, h, fix=False, key='W')
        lam = u.Quantity(w.pixel_to_world([0, nx-1, 0, nx-1, nx//2], [0, 0, ny-1, ny-1, ny//2])[0]).to_value('um')
    bad = (fl & BAD) != 0; src = ((fl >> 21) & 1).astype(bool)
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    r, s = (im - zodi)[ok], np.sqrt(var[ok]); good = ok.sum() > 100
    med = np.median(r) if good else np.nan
    out = dict(local_file=fn, mjd_avg=hd.get('MJD-AVG'), finast=hd.get('FINAST'), dqa=hd.get('L2DQAFLG'),
               lam_center=lam[4], lam_min=lam.min(), lam_max=lam.max(), coverage=float(np.isfinite(im).mean()),
               bad_frac=bad.mean(), src_frac=src.mean(), ok_frac=ok.mean(),
               zodi_med=float(np.nanmedian(zodi)), bkg_med=float(med),
               noise_mad=float(1.4826*np.median(np.abs(r - med))) if good else np.nan)
    for g, bits in GROUPS.items(): out[g + '_frac'] = float(((fl & mask_of(bits)) != 0).mean())
    return out

M = pd.read_csv(f'{ROOT}/data/manifests/astA_cutouts.csv')
M = M[M.status.isin(['ok', 'cached']) & (M.inside == True)].copy()
with ThreadPoolExecutor(4) as ex:
    rows = list(tqdm(ex.map(scan, M.local_file.tolist()), total=len(M)))
Q = M.merge(pd.DataFrame(rows), on='local_file')
Q['pass_gate'] = ((Q.dqa == 'Pass') & (Q.finast == 0) & (Q.bad_frac <= 0.25) &
                  (Q.src_frac <= 0.90) & (Q.coverage >= 0.90))
print('gate failures by criterion:', dict(dqa=int((Q.dqa != 'Pass').sum()), finast=int((Q.finast != 0).sum()),
      bad=int((Q.bad_frac > 0.25).sum()), src=int((Q.src_frac > 0.90).sum()), cov=int((Q.coverage < 0.90).sum())))
print(Q.groupby('ast').pass_gate.agg(frames='size', passed='sum').T.to_string())
Q.to_csv(f'{ROOT}/data/manifests/astA_qc.csv', index=False)

# ---- 2. photometry: mover position + 5 displaced control tracks per visit ------------------------
R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11
META = ['bad_frac', 'src_frac', 'noise_mad', 'ghost_frac', 'persist_frac', 'cr_frac', 'streak_frac']

def meas(im, var, bad, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny: return None
    I, V, B, S = (a[ya:yb, xa:xb] for a in (im, var, bad, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~B & np.isfinite(I)
    ok = ap & ~B & np.isfinite(I) & np.isfinite(V) & (V > 0)
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum(): return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return flux, noise, bool(S[ap].any())

P = Q[Q.pass_gate].copy()
P = P[P.groupby('visit').obs_id.transform('nunique') >= 4]
rng = np.random.default_rng(0)
shifts = {v: [(r * np.cos(a), r * np.sin(a)) for r, a in zip(rng.uniform(15, 40, 5), rng.uniform(0, 2 * np.pi, 5))]
          for v in P.visit.unique()}
print(len(P), 'frames in', P.visit.nunique(), 'visits after the gate and the >=4-frame rule')

rows, fails = [], {'mover': 0, 'ctrl': 0}
for fn, g in tqdm(P.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = (h['IMAGE'].data - h['ZODI'].data).astype(float)
        var = h['VARIANCE'].data.astype(float); fl = h['FLAGS'].data
    bad = (fl & BAD) != 0; src = ((fl >> 21) & 1).astype(bool)
    for r in g.itertuples():
        for kind, (dx, dy) in [('mover', (0, 0))] + [(f'ctrl{c}', s) for c, s in enumerate(shifts[r.visit])]:
            m = meas(im, var, bad, src, r.x + dx, r.y + dy)
            if m is None: fails['mover' if kind == 'mover' else 'ctrl'] += 1; continue
            rows.append(dict(visit=r.visit, kind=kind, ast=r.ast, V=r.V, lam=r.lam_center, det=r.detector,
                             mjd=r.mjd, flux=m[0], noise=m[1], on_src=m[2], **{c: getattr(r, c) for c in META}))
F = pd.DataFrame(rows)
print('measurement failures (mover / control):', fails, '| these drop frames, mainly saturated or flagged apertures')

# ---- 3. one row per track -------------------------------------------------------------------
def feats(d):
    s = (d.flux / d.noise).values; fx, nz = d.flux.values, d.noise.values
    top = int(np.argmax(s)); rest = np.delete(np.arange(len(s)), top); pos = np.clip(s, 0, None)
    return pd.Series(dict(n=len(d), track_snr=fx.sum() / np.sqrt((nz ** 2).sum()),
        snr_wo_top=fx[rest].sum() / np.sqrt((nz[rest] ** 2).sum()), med_snr=np.median(s),
        frac_gt3=(s > 3).mean(), top_share=pos.max() / (pos.sum() + 1e-9),
        src_frac_ap=d.on_src.mean(), lam_mean=d.lam.mean(), lam_span=d.lam.max() - d.lam.min(),
        n_det=d.det.nunique(), span_h=(d.mjd.max() - d.mjd.min()) * 24,
        **{f'm_{c}': d[c].mean() for c in META}, V=d.V.median(), ast=d.ast.iloc[0]))
T = F.groupby(['visit', 'kind']).apply(feats).reset_index()
T = T[T.n >= 4].reset_index(drop=True)
T['y'] = (T.kind == 'mover').astype(int)
SL = ['track_snr', 'snr_wo_top', 'med_snr']
for c in SL: T[c + '_l'] = np.sign(T[c]) * np.log1p(np.abs(T[c]))
T.to_csv(f'{ROOT}/data/manifests/astA_track_table.csv', index=False)
print(T.groupby('ast').y.agg(positives='sum', negatives=lambda s: int((s == 0).sum())).T.to_string())
print('median track S/N: movers', round(T[T.y == 1].track_snr.median(), 1),
      '| controls', round(T[T.y == 0].track_snr.median(), 2))

# ---- 4. comparison: leave-one-asteroid-out, cluster bootstrap over asteroids -----------------------
IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
MET = IMG + ['src_frac_ap', 'lam_mean', 'lam_span', 'n_det', 'span_h'] + [f'm_{c}' for c in META]

def fit(X, y, C=0.5):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med))
    return med, sc, LogisticRegression(C=C, class_weight='balanced', max_iter=3000).fit(sc.transform(X.fillna(med)), y)

def oof(cols):
    p = np.zeros(len(T))
    for a in T.ast.unique():
        tr, te = (T.ast != a).values, (T.ast == a).values
        med, sc, lr = fit(T.loc[tr, cols], T.y[tr])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

def one(p, y, rec=0.9):
    thr = np.quantile(p[y == 1], 1 - rec)
    return roc_auc_score(y, p), (p[y == 0] >= thr).mean()

def report(p, B=300):
    y, g = T.y.values, T.ast.values
    base = one(p, y); rg = np.random.default_rng(0); ug = np.unique(g); bs = []
    idx = {a: np.where(g == a)[0] for a in ug}
    for _ in range(B):
        i = np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])
        if y[i].sum() >= 3 and (y[i] == 0).sum() >= 3: bs.append(one(p[i], y[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    return f'AUC {base[0]:.3f} [{lo[0]:.3f}-{hi[0]:.3f}] | false alerts @90% recall {base[1]:.3f} [{lo[1]:.3f}-{hi[1]:.3f}]'

print('\nrule: track S/N only          ', report(T.track_snr.values))
print('rule: S/N without top frame   ', report(T.snr_wo_top.values))
print('image-only ranker (LOAO)      ', report(oof(IMG)))
print('image + metadata (LOAO)       ', report(oof(MET)))

# ---- 5. export for the backend (numpy-only scorer, 50 bootstrap models for uncertainty) ----------
med, sc, _ = fit(T[MET], T.y); Z = sc.transform(T[MET].fillna(med))
rb = np.random.default_rng(1); coefs, ints = [], []
while len(coefs) < 50:
    i = rb.integers(0, len(T), len(T))
    if T.y.values[i].sum() < 3: continue
    lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000).fit(Z[i], T.y.values[i])
    coefs.append(lr.coef_[0].tolist()); ints.append(float(lr.intercept_[0]))
os.makedirs(f'{ROOT}/models', exist_ok=True)
json.dump(dict(version='ranker_A_v1', features=MET, signed_log=[c + '_l' for c in SL], fill=med.to_dict(),
               mean=sc.mean_.tolist(), scale=sc.scale_.tolist(), coefs=coefs, intercepts=ints),
          open(f'{ROOT}/models/ranker_A_v1.json', 'w'))
print('\nsaved models/ranker_A_v1.json')

  0%|          | 0/573 [00:00<?, ?it/s]

gate failures by criterion: {'dqa': 0, 'finast': 0, 'bad': 9, 'src': 137, 'cov': 0}
ast     6   7   8   9   14   15  18  19  20  29  40  43
frames  28  50  40  27  39  110  25  43  73  42  50  46
passed  28  50   5  26   0  108   0  43  71   4  49  46
430 frames in 26 visits after the gate and the >=4-frame rule


  0%|          | 0/430 [00:00<?, ?it/s]

measurement failures (mover / control): {'mover': 310, 'ctrl': 38} | these drop frames, mainly saturated or flagged apertures
ast        6.0   7.0   8.0   9.0   15.0  19.0  20.0  29.0  40.0  43.0
positives     1     1     0     1     2     2     0     1     3     3
negatives    15    15     5    15    15    15    15     5    15    15
median track S/N: movers 533.7 | controls 0.31

rule: track S/N only           AUC 0.999 [0.997-1.000] | false alerts @90% recall 0.000 [0.000-0.014]
rule: S/N without top frame    AUC 1.000 [1.000-1.000] | false alerts @90% recall 0.000 [0.000-0.000]
image-only ranker (LOAO)       AUC 1.000 [1.000-1.000] | false alerts @90% recall 0.000 [0.000-0.000]
image + metadata (LOAO)        AUC 1.000 [1.000-1.000] | false alerts @90% recall 0.000 [0.000-0.000]

saved models/ranker_A_v1.json


In [ ]:
# Step B1: fainter asteroids (V ~15-18.5): screening, coverage query (resumable), frame table
import os, time, numpy as np, pandas as pd
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from astroquery.ipac.irsa import Irsa
from scipy.interpolate import CubicSpline
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

T0, T1 = 60789.8, 61318.0                       # archive span from your earlier output
VLO, VHI = 14.5, 18.5                           # usable window for this set
d0, d1 = Time(T0 - 1, format='mjd').iso[:10], Time(T1 + 1, format='mjd').iso[:10]

def scr(n):
    try:
        e = Horizons(id=str(n), id_type='smallbody', location='500',
                     epochs=dict(start=d0, stop=d1, step='1d')).ephemerides(quantities='1,9,23').to_pandas()
        e['mjd'] = e.datetime_jd - 2400000.5
        e = e[['mjd', 'RA', 'DEC', 'V', 'elong']].dropna().reset_index(drop=True)
        s = e[e.elong.between(80, 100)]
        return n, e, (float(s.V.median()) if len(s) > 20 else np.nan)
    except Exception:
        return n, None, np.nan

# 1. screen ~120 candidates by their median V on days SPHEREx can see them
EPH, medV = {}, {}
with ThreadPoolExecutor(4) as ex:
    for n, e, v in tqdm(ex.map(scr, list(range(150, 1000, 7))), total=122):
        if e is not None: EPH[n] = e; medV[n] = v
mv = pd.Series(medV).dropna()
pool = mv[(mv >= 15.0) & (mv <= 18.0)].sort_values()
print(len(mv), 'screened;', len(pool), 'have median V between 15 and 18')
assert len(pool) >= 8, 'too few faint candidates: widen the candidate range and tell me'
pick = pool.iloc[np.linspace(0, len(pool) - 1, min(16, len(pool))).astype(int)]
print(pick.round(1).to_string())

# 2. interpolators and the days worth querying
ids = list(pick.index)
spl = {n: (CubicSpline(EPH[n].mjd, np.degrees(np.unwrap(np.radians(EPH[n].RA.values)))),
           CubicSpline(EPH[n].mjd, EPH[n].DEC.values), CubicSpline(EPH[n].mjd, EPH[n].V.values),
           CubicSpline(EPH[n].mjd, EPH[n].elong.values)) for n in ids}
days = []
for n in ids:
    e = EPH[n]; ok = e[e.elong.between(75, 110) & e.V.between(VLO - 1, VHI + 1)]
    days += [(n, r.mjd, r.RA, r.DEC) for r in ok.itertuples()]
print(len(days), 'asteroid-days to query; roughly', round(len(days) * 1.0 / 60), 'min at 8 workers')

def one(d, tries=3):
    n, m, ra, dec = d
    q = f"""SELECT obs_id, t_min, t_max, access_url FROM spherex.obscore
            WHERE CONTAINS(POINT('ICRS',{ra:.5f},{dec:.5f}), s_region)=1
              AND t_min >= {m - 0.5:.4f} AND t_min < {m + 0.5:.4f}"""
    for k in range(tries):
        try:
            t = Irsa.query_tap(q).to_table().to_pandas(); t['ast'] = n; return t
        except Exception:
            time.sleep(2 * (k + 1))
    return None

# 3. resumable run
DONE_F, HIT_F = f'{ROOT}/data/manifests/b1_done.csv', f'{ROOT}/data/manifests/b1_hits.csv'
done = set(map(tuple, pd.read_csv(DONE_F)[['ast', 'mjd']].round(3).values)) if os.path.exists(DONE_F) else set()
hits = [pd.read_csv(HIT_F)] if os.path.exists(HIT_F) else []
done_rows = [dict(ast=a, mjd=m) for a, m in done]
todo = [d for d in days if (d[0], round(d[1], 3)) not in done]
print(len(todo), 'queries left;', len(done), 'already done')
bad = 0
for i in range(0, len(todo), 160):
    with ThreadPoolExecutor(8) as ex:
        futs = {ex.submit(one, d): d for d in todo[i:i + 160]}
        for f in as_completed(futs):
            d, r = futs[f], f.result()
            if r is None: bad += 1; continue
            done_rows.append(dict(ast=d[0], mjd=round(d[1], 3)))
            if len(r): hits.append(r)
    pd.DataFrame(done_rows).to_csv(DONE_F, index=False)
    if hits: pd.concat(hits).to_csv(HIT_F, index=False)
    print(f'{min(i + 160, len(todo))}/{len(todo)} | hit rows: {sum(len(h) for h in hits)} | failed: {bad}')

# 4. frame table and summary (saved to NEW files; set A is untouched)
R = pd.concat(hits).drop_duplicates(['ast', 'obs_id', 'access_url']).copy()
R['mjd'] = (R.t_min + R.t_max) / 2
R['ra'] = [float(spl[a][0](m)) % 360 for a, m in zip(R.ast, R.mjd)]
R['dec'] = [float(spl[a][1](m)) for a, m in zip(R.ast, R.mjd)]
R['V'] = [float(spl[a][2](m)) for a, m in zip(R.ast, R.mjd)]
R['elong'] = [float(spl[a][3](m)) for a, m in zip(R.ast, R.mjd)]
FB = R.sort_values(['ast', 'mjd']).reset_index(drop=True)
FB['visit'] = (FB.groupby('ast').mjd.diff().fillna(99) > 0.5).cumsum() + 100000   # ids cannot clash with set A
FB.to_csv(f'{ROOT}/data/manifests/feasibility_frames_B.csv', index=False)

U = FB[(FB.V >= VLO) & (FB.V <= VHI)]
tr = U.groupby('visit').agg(ast=('ast', 'first'), n_exp=('obs_id', 'nunique'), V=('V', 'median'))
tr4 = tr[tr.n_exp >= 4]
print(tr4.groupby('ast').agg(tracks=('V', 'size'), V_med=('V', 'median')).round(1).T.to_string())
print('tracks with >=4 frames:', len(tr4), '| by V bin:',
      pd.cut(tr4.V, [14.5, 16, 17, 18, 18.5]).value_counts().sort_index().to_dict())

  0%|          | 0/122 [00:00<?, ?it/s]

114 screened; 62 have median V between 15 and 18
248    15.1
927    15.2
752    15.3
542    15.4
311    15.6
731    15.7
885    15.8
437    15.9
857    16.2
829    16.3
920    16.6
913    16.7
822    16.8
549    16.9
353    17.2
836    18.0
1640 asteroid-days to query; roughly 27 min at 8 workers
1640 queries left; 0 already done
160/1640 | hit rows: 303 | failed: 0
320/1640 | hit rows: 593 | failed: 0
480/1640 | hit rows: 985 | failed: 0
640/1640 | hit rows: 1290 | failed: 0
800/1640 | hit rows: 1662 | failed: 0
960/1640 | hit rows: 2009 | failed: 0
1120/1640 | hit rows: 2507 | failed: 0
1280/1640 | hit rows: 2891 | failed: 0
1440/1640 | hit rows: 3244 | failed: 0
1600/1640 | hit rows: 3362 | failed: 0
1640/1640 | hit rows: 3512 | failed: 0
ast      248   311   353   437   542   549   731   752   822   829   836   857   885   913   920   927
tracks  14.0  23.0   7.0  11.0  13.0  13.0  14.0   4.0  17.0  18.0   9.0  14.0  15.0  19.0  24.0  13.0
V_med   15.3  15.5  17.2  16.0  15.4  16.7

In [ ]:
# Step B2: download 0.25 deg cutouts for set B (resumable; set A untouched)
import io, os, json, time, threading, requests, warnings
import numpy as np, pandas as pd, pyvo
from concurrent.futures import ThreadPoolExecutor, as_completed
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SIZE_DEG, CUT_URL = 0.25, 'https://irsa.ipac.caltech.edu/ibe/cutout'
PER_AST, WORKERS, LIMIT = 6, 2, None          # LIMIT = 16 for a quick test run
OUT = f'{ROOT}/data/raw/spherex_qr2/astB/cutouts'; os.makedirs(OUT, exist_ok=True)
LOCK = threading.Lock()
CACHE_F = f'{ROOT}/data/manifests/links_cache.json'
cache = json.load(open(CACHE_F)) if os.path.exists(CACHE_F) else {}

# ---- selection: all faint tracks (V >= 17) + up to PER_AST random tracks per asteroid ----
FB = pd.read_csv(f'{ROOT}/data/manifests/feasibility_frames_B.csv')
U = FB[(FB.V >= 14.5) & (FB.V <= 18.5)].copy()
U = U[U.groupby('visit').obs_id.transform('nunique') >= 4]
vis = U.groupby('visit').agg(ast=('ast', 'first'), V=('V', 'median')).reset_index()
faint = vis[vis.V >= 17].visit
rest = vis[vis.V < 17].sample(frac=1, random_state=0).groupby('ast').head(PER_AST).visit
keep = pd.concat([faint, rest]).unique()
A = U[U.visit.isin(keep)].copy()
A['detector'] = A.access_url.str.extract(r'/(D\d)$')[0]
A = A.reset_index(drop=True)
if LIMIT: A = A.sample(LIMIT, random_state=0).reset_index(drop=True)
print(f'{len(A)} cutouts | {A.visit.nunique()} tracks | {A.ast.nunique()} asteroids')
print('tracks by V bin:', pd.cut(vis[vis.visit.isin(keep)].V, [14.5, 16, 17, 18, 18.5]).value_counts().sort_index().to_dict())
print(f'rough time: {len(A) * 3 / 60:.0f} min (estimated from set A speed); resumable, cached files are skipped')

def resolve(access_url):
    if access_url in cache: return cache[access_url], ''
    err = ''
    for wait in (5, 20):
        try:
            d = pyvo.dal.adhoc.DatalinkResults.from_result_url(access_url).to_table().to_pandas()
            dec = lambda s: s.decode() if isinstance(s, bytes) else s
            rows = d[d.semantics.map(dec) == '#cutout']
            if len(rows):
                p = dec(rows.iloc[0]['path'])
                with LOCK: cache[access_url] = p
                return p, ''
            err = 'no #cutout row: ' + str(d.semantics.map(dec).tolist())[:80]
        except Exception as e:
            err = f'{type(e).__name__}: {str(e)[:100]}'
        time.sleep(wait)
    return None, err

def valid(fn):
    try:
        with fits.open(fn) as h:
            _ = h['IMAGE'].data; _ = h['WCS-WAVE'].data
            return len(h) == 7
    except Exception:
        return False

def get_cutout(path, ra, dec, out):
    if os.path.exists(out) and valid(out): return 'cached'
    err = ''
    for k in range(3):
        try:
            r = requests.get(CUT_URL, params=dict(path=path, ra=ra, dec=dec, size=SIZE_DEG), timeout=(15, 180))
            if r.status_code == 200 and r.content[:6] == b'SIMPLE':
                open(out + '.part', 'wb').write(r.content); os.replace(out + '.part', out)
                return 'ok' if valid(out) else 'invalid file'
            err = f'{r.status_code} {r.text[:80]}'
        except Exception as e:
            err = f'{type(e).__name__}: {str(e)[:80]}'
        time.sleep(2 * (k + 1))
    return 'FAILED: ' + err

def one(r):
    path, err = resolve(r['access_url'])
    if not path: return dict(r) | dict(path=None, local_file=None, status='link error: ' + err)
    fn = f"{OUT}/{os.path.basename(path)[:-5]}_ast{int(r['ast'])}.fits"
    st = get_cutout(path, r['ra'], r['dec'], fn)
    rec = dict(r) | dict(path=path, local_file=fn, status=st)
    if st in ('ok', 'cached'):
        with fits.open(fn, memmap=False) as h:
            w = WCS(h['IMAGE'].header); ny, nx = h['IMAGE'].data.shape
        x, y = w.world_to_pixel(SkyCoord(r['ra'], r['dec'], unit='deg'))
        rec |= dict(nx=nx, ny=ny, x=float(x), y=float(y),
                    inside=bool(12 <= x <= nx - 13 and 12 <= y <= ny - 13))
    return rec

t0 = time.time(); rows = []
try:
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(one, r) for r in A.to_dict('records')]
        for fu in tqdm(as_completed(futs), total=len(futs)): rows.append(fu.result())
finally:
    json.dump(cache, open(CACHE_F, 'w'))
M = pd.DataFrame(rows).sort_values(['ast', 'mjd']).reset_index(drop=True)
M['ver'] = M.path.str.extract(r'(l2b-v\d+)')[0]
M.to_csv(f"{ROOT}/data/manifests/astB_cutouts{'_test' if LIMIT else ''}.csv", index=False)

print(f'{(time.time() - t0) / 60:.1f} min | statuses:', M.status.str.slice(0, 140).value_counts().to_dict())
if 'inside' in M: print('asteroid inside with margin:', int((M.inside == True).sum()), 'of', len(M))
print('processing versions:', M.ver.value_counts(dropna=False).to_dict())

1350 cutouts | 100 tracks | 16 asteroids
tracks by V bin: {Interval(14.5, 16.0, closed='right'): 44, Interval(16.0, 17.0, closed='right'): 38, Interval(17.0, 18.0, closed='right'): 17, Interval(18.0, 18.5, closed='right'): 1}
rough time: 68 min (estimated from set A speed); resumable, cached files are skipped


  0%|          | 0/1350 [00:00<?, ?it/s]

66.5 min | statuses: {'ok': 1350}
asteroid inside with margin: 1301 of 1350
processing versions: {'l2b-v20': 432, 'l2b-v24': 429, 'l2b-v27': 172, 'l2b-v25': 102, 'l2b-v23': 91, 'l2b-v22': 63, 'l2b-v21': 60, nan: 1}


In [ ]:
# Step B3: QC -> photometry -> track table -> comparison -> per-V-bin and per-version tables -> export
import logging, warnings, os, json
logging.getLogger('astropy').setLevel(logging.ERROR); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def mask_of(bits): return sum(1 << b for b in bits)
BAD = mask_of([0,1,2,6,7,9,10,11,14,15,17,19,22,24,26,27,28,29])
GROUPS = {'cr':[0,1,27,28], 'streak':[19,29], 'ghost':[22,24], 'persist':[17],
          'dichroic':[7], 'bloom':[26], 'badpix':[6,9,10,11,14,15]}

def scan(fn):
    with fits.open(fn, memmap=False) as h:
        im, fl, var, zodi = [h[k].data for k in ('IMAGE', 'FLAGS', 'VARIANCE', 'ZODI')]
        hd = h['IMAGE'].header; ny, nx = im.shape
        w = WCS(h[1].header, h, fix=False, key='W')
        lam = u.Quantity(w.pixel_to_world([0, nx-1, 0, nx-1, nx//2], [0, 0, ny-1, ny-1, ny//2])[0]).to_value('um')
    bad = (fl & BAD) != 0; src = ((fl >> 21) & 1).astype(bool)
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    r, s = (im - zodi)[ok], np.sqrt(var[ok]); good = ok.sum() > 100
    med = np.median(r) if good else np.nan
    out = dict(local_file=fn, finast=hd.get('FINAST'), dqa=hd.get('L2DQAFLG'),
               lam_center=lam[4], coverage=float(np.isfinite(im).mean()),
               bad_frac=bad.mean(), src_frac=src.mean(),
               noise_mad=float(1.4826*np.median(np.abs(r - med))) if good else np.nan)
    for g, bits in GROUPS.items(): out[g + '_frac'] = float(((fl & mask_of(bits)) != 0).mean())
    return out

# ---- 1. QC -------------------------------------------------------------------------------------
M = pd.read_csv(f'{ROOT}/data/manifests/astB_cutouts.csv')
M = M[M.status.isin(['ok', 'cached']) & (M.inside == True)].copy()
with ThreadPoolExecutor(4) as ex:
    rows = list(tqdm(ex.map(scan, M.local_file.tolist()), total=len(M)))
Q = M.merge(pd.DataFrame(rows), on='local_file')
Q['pass_gate'] = ((Q.dqa == 'Pass') & (Q.finast == 0) & (Q.bad_frac <= 0.25) &
                  (Q.src_frac <= 0.90) & (Q.coverage >= 0.90))
print('gate failures by criterion:', dict(dqa=int((Q.dqa != 'Pass').sum()), finast=int((Q.finast != 0).sum()),
      bad=int((Q.bad_frac > 0.25).sum()), src=int((Q.src_frac > 0.90).sum()), cov=int((Q.coverage < 0.90).sum())))
print(len(Q), 'frames,', int(Q.pass_gate.sum()), 'pass the gate')
Q.to_csv(f'{ROOT}/data/manifests/astB_qc.csv', index=False)

# ---- 2. photometry: mover position + 5 displaced control tracks per visit -----------------------
R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11
META = ['bad_frac', 'src_frac', 'noise_mad', 'ghost_frac', 'persist_frac', 'cr_frac', 'streak_frac']

def meas(im, var, bad, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny: return None
    I, V, B, S = (a[ya:yb, xa:xb] for a in (im, var, bad, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~B & np.isfinite(I)
    ok = ap & ~B & np.isfinite(I) & np.isfinite(V) & (V > 0)
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum(): return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return flux, noise, bool(S[ap].any())

P = Q[Q.pass_gate].copy()
P = P[P.groupby('visit').obs_id.transform('nunique') >= 4]
rng = np.random.default_rng(0)
shifts = {v: [(r * np.cos(a), r * np.sin(a)) for r, a in zip(rng.uniform(15, 40, 5), rng.uniform(0, 2 * np.pi, 5))]
          for v in P.visit.unique()}
print(len(P), 'frames in', P.visit.nunique(), 'visits after the gate and the >=4-frame rule')

rows, fails = [], {'mover': 0, 'ctrl': 0}
for fn, g in tqdm(P.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = (h['IMAGE'].data - h['ZODI'].data).astype(float)
        var = h['VARIANCE'].data.astype(float); fl = h['FLAGS'].data
    bad = (fl & BAD) != 0; src = ((fl >> 21) & 1).astype(bool)
    for r in g.itertuples():
        for kind, (dx, dy) in [('mover', (0, 0))] + [(f'ctrl{c}', s) for c, s in enumerate(shifts[r.visit])]:
            m = meas(im, var, bad, src, r.x + dx, r.y + dy)
            if m is None: fails['mover' if kind == 'mover' else 'ctrl'] += 1; continue
            rows.append(dict(visit=r.visit, kind=kind, ast=r.ast, V=r.V, ver=r.ver, lam=r.lam_center,
                             det=r.detector, mjd=r.mjd, flux=m[0], noise=m[1], on_src=m[2],
                             local_file=fn, **{c: getattr(r, c) for c in META}))
F = pd.DataFrame(rows)
nm = int((P.shape[0]))
print('measurement failures (mover / control):', fails, f'(of about {nm} / {5 * nm} measurements)')

# cap at 6 random frames per visit (the backend sends at most 6), chosen among frames where the mover was measured
rs = np.random.default_rng(1); sel = {}
for v, g in F[F.kind == 'mover'].groupby('visit'):
    files = g.local_file.unique(); sel[v] = set(rs.choice(files, min(6, len(files)), replace=False))
F = F[[fn in sel.get(v, ()) for v, fn in zip(F.visit, F.local_file)]]
F.to_csv(f'{ROOT}/data/manifests/astB_frames.csv', index=False)

# ---- 3. one row per track -----------------------------------------------------------------------
def feats(d):
    s = (d.flux / d.noise).values; fx, nz = d.flux.values, d.noise.values
    top = int(np.argmax(s)); rest = np.delete(np.arange(len(s)), top); pos = np.clip(s, 0, None)
    return pd.Series(dict(n=len(d), track_snr=fx.sum() / np.sqrt((nz ** 2).sum()),
        snr_wo_top=fx[rest].sum() / np.sqrt((nz[rest] ** 2).sum()), med_snr=np.median(s),
        frac_gt3=(s > 3).mean(), top_share=pos.max() / (pos.sum() + 1e-9),
        src_frac_ap=d.on_src.mean(), lam_mean=d.lam.mean(), lam_span=d.lam.max() - d.lam.min(),
        n_det=d.det.nunique(), span_h=(d.mjd.max() - d.mjd.min()) * 24,
        **{f'm_{c}': d[c].mean() for c in META}, V=d.V.median(), ast=d.ast.iloc[0], ver=d.ver.iloc[0]))
T = F.groupby(['visit', 'kind']).apply(feats).reset_index()
T = T[T.n >= 4].reset_index(drop=True)
T['y'] = (T.kind == 'mover').astype(int)
SL = ['track_snr', 'snr_wo_top', 'med_snr']
for c in SL: T[c + '_l'] = np.sign(T[c]) * np.log1p(np.abs(T[c]))
T['Vbin'] = pd.cut(T.V, [14.5, 16, 17, 18.5]).astype(str)
T['vgrp'] = T.ver.where(T.ver.isin(['l2b-v20', 'l2b-v24', 'l2b-v27']), 'other')
T.to_csv(f'{ROOT}/data/manifests/astB_track_table.csv', index=False)
pos = T[T.y == 1]
print('positive tracks:', len(pos), '| negatives:', int((T.y == 0).sum()),
      '| by V bin:', pos.Vbin.value_counts().to_dict(), '| by version:', pos.vgrp.value_counts().to_dict())
print('median track S/N by V bin (mover / control):')
print(T.groupby(['Vbin', 'y']).track_snr.median().unstack().round(2).to_string())

# ---- 4. comparison ----------------------------------------------------------------------------
IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
FLG = ['src_frac_ap'] + [f'm_{c}' for c in META]
LAM = ['lam_mean', 'lam_span', 'n_det', 'span_h']
SETS = {'image-only (consistency)': IMG, '+ quality flags': IMG + FLG,
        '+ wavelength/time': IMG + LAM, '+ both (full)': IMG + FLG + LAM}

def fit(X, y, C=0.5):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med))
    return med, sc, LogisticRegression(C=C, class_weight='balanced', max_iter=3000).fit(sc.transform(X.fillna(med)), y)

def oof(cols):
    p = np.zeros(len(T))
    for a in T.ast.unique():
        tr, te = (T.ast != a).values, (T.ast == a).values
        med, sc, lr = fit(T.loc[tr, cols], T.y[tr])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

y, g = T.y.values, T.ast.values
def met(p, y, far=0.05, rec=0.9):
    thr = np.quantile(p[y == 0], 1 - far)
    return roc_auc_score(y, p), (p[y == 1] >= thr).mean(), (p[y == 0] >= np.quantile(p[y == 1], 1 - rec)).mean()

def boot(p, B=300):
    rg = np.random.default_rng(0); ug = np.unique(g); idx = {a: np.where(g == a)[0] for a in ug}; bs = []
    for _ in range(B):
        i = np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])
        if y[i].sum() >= 5 and (y[i] == 0).sum() >= 20: bs.append(met(p[i], y[i]))
    return np.percentile(bs, [2.5, 97.5], axis=0)

methods = {'rule: track S/N': T.track_snr.values, 'rule: S/N without top frame': T.snr_wo_top.values}
for k, cols in SETS.items(): methods[k] = oof(cols)

res = []
for k, p in methods.items():
    a, r5, f90 = met(p, y); lo, hi = boot(p)
    thr = np.quantile(p[y == 0], 0.95)
    byb = {b: (p[(y == 1) & (T.Vbin == b).values] >= thr).mean() for b in sorted(T.Vbin.unique())}
    res.append({'method': k, 'AUC': f'{a:.3f}', 'recall @5% false alerts': f'{r5:.2f} [{lo[1]:.2f}-{hi[1]:.2f}]',
                'false alerts @90% recall': f'{f90:.3f} [{lo[2]:.3f}-{hi[2]:.3f}]',
                **{f'recall V{b}': f'{v:.2f}' for b, v in byb.items()}})
print('\n' + pd.DataFrame(res).to_string(index=False))

best = 'image-only (consistency)'
pbest = methods['+ both (full)']
thr = np.quantile(pbest[y == 0], 0.95)
print('\nfull model: recall at 5% false alerts, by processing version:')
print(pd.DataFrame({'vgrp': T.vgrp[y == 1].values, 'hit': pbest[y == 1] >= thr}).groupby('vgrp').hit.agg(['mean', 'size']).round(2).to_string())

# ---- 5. export (numpy-only scorer, 50 bootstrap models) + sanity check on the bright set A ----------
cols = SETS['+ both (full)']
med, sc, _ = fit(T[cols], T.y); Z = sc.transform(T[cols].fillna(med))
rb = np.random.default_rng(1); coefs, ints = [], []
while len(coefs) < 50:
    i = rb.integers(0, len(T), len(T))
    if T.y.values[i].sum() < 5: continue
    lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000).fit(Z[i], T.y.values[i])
    coefs.append(lr.coef_[0].tolist()); ints.append(float(lr.intercept_[0]))
model = dict(version='ranker_B_v1', features=cols, signed_log=[c + '_l' for c in SL], fill=med.to_dict(),
             mean=sc.mean_.tolist(), scale=sc.scale_.tolist(), coefs=coefs, intercepts=ints)
os.makedirs(f'{ROOT}/models', exist_ok=True)
json.dump(model, open(f'{ROOT}/models/ranker_B_v1.json', 'w'))
print('\nsaved models/ranker_B_v1.json')

def apply(df, m):
    X = df[m['features']].fillna(pd.Series(m['fill']))
    Zs = (X.values - np.array(m['mean'])) / np.array(m['scale'])
    return (1 / (1 + np.exp(-(Zs @ np.array(m['coefs']).T + np.array(m['intercepts']))))).mean(1)

fa = f'{ROOT}/data/manifests/astA_track_table.csv'
if os.path.exists(fa):
    TA = pd.read_csv(fa); pa = apply(TA, model)
    thrA = np.quantile(apply(T[T.y == 0], model), 0.95)      # threshold from set B controls (in-sample)
    print(f'sanity, bright set A: recall {np.mean(pa[TA.y.values == 1] >= thrA):.2f} '
          f'({int(TA.y.sum())} positives) | control false alerts {np.mean(pa[TA.y.values == 0] >= thrA):.3f}')

  0%|          | 0/1301 [00:00<?, ?it/s]

gate failures by criterion: {'dqa': 0, 'finast': 0, 'bad': 20, 'src': 131, 'cov': 0}
1301 frames, 1153 pass the gate
1145 frames in 89 visits after the gate and the >=4-frame rule


  0%|          | 0/1145 [00:00<?, ?it/s]

measurement failures (mover / control): {'mover': 251, 'ctrl': 108} (of about 1145 / 5725 measurements)
positive tracks: 81 | negatives: 403 | by V bin: {'(16.0, 17.0]': 36, '(14.5, 16.0]': 34, '(17.0, 18.5]': 11} | by version: {'l2b-v20': 28, 'other': 23, 'l2b-v24': 21, 'l2b-v27': 9}
median track S/N by V bin (mover / control):
y                0      1
Vbin                     
(14.5, 16.0] -0.17  22.18
(16.0, 17.0]  0.13   8.95
(17.0, 18.5]  0.16   7.62

                     method   AUC recall @5% false alerts false alerts @90% recall recall V(14.5, 16.0] recall V(16.0, 17.0] recall V(17.0, 18.5]
            rule: track S/N 0.916        0.38 [0.20-0.74]      0.166 [0.090-0.271]                 0.59                 0.28                 0.09
rule: S/N without top frame 0.940        0.56 [0.26-0.84]      0.114 [0.064-0.206]                 0.71                 0.50                 0.27
   image-only (consistency) 0.941        0.54 [0.25-0.95]      0.134 [0.035-0.200]                 0

In [ ]:
# Diagnostic: do flag bit definitions differ between processing versions?
import re
from astropy.io import fits

def flagdefs(header):                      # same logic as the backend's flag_definitions()
    d = {k: int(header[k]) for k in header if k.startswith('MP_')}
    for k in header:
        if re.fullmatch(r'MSKN\d{4}', k):
            m = int(header['MSKM' + k[4:]])
            d['MP_' + str(header[k])] = m.bit_length() - 1
    return d

GROUP_NAMES = {'cr': ['TRANSIENT','OVERFLOW','SNOWBALL','HALO'], 'streak': ['OUTLIER','SAT_HALO'],
               'ghost': ['GHOST','GHOST_EXT'], 'persist': ['PERSIST'], 'dichroic': ['DICHROIC'],
               'bloom': ['BLOOM'], 'badpix': ['NONFUNC','MISSING','HOT','COLD','PHANMISS','NONLINEAR']}
B = pd.read_csv(f'{ROOT}/data/manifests/astB_cutouts.csv'); B = B[B.status == 'ok']
maps, ref = {}, None
for v, g in B.groupby('ver'):
    maps[v] = [flagdefs(fits.getheader(fn, 'FLAGS')) for fn in g.local_file.sample(min(3, len(g)), random_state=0)]
for v, ms in sorted(maps.items()):
    d = ms[0]; ref = ref or d
    missing = [n for ns in GROUP_NAMES.values() for n in ns if 'MP_' + n not in d]
    print(f"{v}: files agree {all(m == d for m in ms)} | flags {len(d)} | has SOURCE {'MP_SOURCE' in d} "
          f"| differs from first version: {d != ref} | group names not found: {missing}")
print('\nfirst version:', dict(sorted(ref.items(), key=lambda kv: kv[1])))
for v, ms in sorted(maps.items()):
    diff = {k: b for k, b in ms[0].items() if ref.get(k) != b}
    if diff: print(v, 'bits that differ from the first version:', diff)

l2b-v20: files agree True | flags 16 | has SOURCE True | differs from first version: False | group names not found: ['SNOWBALL', 'HALO', 'SAT_HALO', 'GHOST', 'GHOST_EXT', 'BLOOM', 'MISSING']
l2b-v21: files agree False | flags 21 | has SOURCE True | differs from first version: True | group names not found: ['SNOWBALL', 'SAT_HALO', 'MISSING']
l2b-v22: files agree False | flags 22 | has SOURCE True | differs from first version: True | group names not found: ['SAT_HALO', 'MISSING']
l2b-v23: files agree False | flags 22 | has SOURCE True | differs from first version: True | group names not found: ['SAT_HALO', 'MISSING']
l2b-v24: files agree True | flags 22 | has SOURCE True | differs from first version: True | group names not found: ['SAT_HALO', 'MISSING']
l2b-v25: files agree True | flags 22 | has SOURCE True | differs from first version: True | group names not found: ['SAT_HALO', 'MISSING']
l2b-v27: files agree True | flags 22 | has SOURCE True | differs from first version: True | group n

In [ ]:
# Step B3 (final): header-driven masks, version-stable features, comparison, export
import logging, warnings, os, json, re
logging.getLogger('astropy').setLevel(logging.ERROR); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ---- flag handling: same rule as the backend (definitions come from each file's header) ----------
def flagdefs(header):
    d = {k: int(header[k]) for k in header if k.startswith('MP_')}
    for k in header:
        if re.fullmatch(r'MSKN\d{4}', k):
            d['MP_' + str(header[k])] = int(header['MSKM' + k[4:]]).bit_length() - 1
    return d

INFO = {'MP_SOURCE', 'MP_FULLSAMPLE'}                      # informational, not rejected
GROUPS = {'cr': ['TRANSIENT', 'OVERFLOW'], 'outlier': ['OUTLIER'], 'persist': ['PERSIST'],
          'dichroic': ['DICHROIC'], 'badpix': ['NONFUNC', 'MISSING_DATA', 'HOT', 'COLD', 'PHANMISS', 'NONLINEAR']}
EXTRA = ['GHOST', 'GHOST_FPA', 'GHOST_EXT', 'BLOOM', 'SNOWBALL', 'HALO', 'CROSSTALK', 'STREAK']   # warnings only

def masks(fl, header):
    d = flagdefs(header); fl = fl.astype(np.uint64)
    bad = (fl & np.uint64(sum(1 << b for k, b in d.items() if k not in INFO))) != 0
    src = ((fl >> np.uint64(d['MP_SOURCE'])) & np.uint64(1)).astype(bool)
    grp = {}
    for g, names in GROUPS.items():
        m = sum(1 << d['MP_' + n] for n in names if 'MP_' + n in d)
        grp[g + '_frac'] = float(((fl & np.uint64(m)) != 0).mean()) if m else 0.0
    for n in EXTRA:                                         # exist only in some versions: NaN where undefined
        grp['x_' + n.lower() + '_frac'] = (float(((fl >> np.uint64(d['MP_' + n])) & np.uint64(1)).mean())
                                           if 'MP_' + n in d else np.nan)
    return bad, src, grp

def scan(fn):
    with fits.open(fn, memmap=False) as h:
        im, fl, var = [h[k].data for k in ('IMAGE', 'FLAGS', 'VARIANCE')]
        bad, src, grp = masks(fl, h['FLAGS'].header)
        hd = h['IMAGE'].header; ny, nx = im.shape
        w = WCS(h[1].header, h, fix=False, key='W')
        lam = u.Quantity(w.pixel_to_world([0, nx-1, 0, nx-1, nx//2], [0, 0, ny-1, ny-1, ny//2])[0]).to_value('um')
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    r = im[ok]; good = ok.sum() > 100
    med = np.median(r) if good else np.nan
    out = dict(local_file=fn, finast=hd.get('FINAST'), dqa=hd.get('L2DQAFLG'),
               lam_center=lam[4], coverage=float(np.isfinite(im).mean()),
               bad_frac=bad.mean(), src_frac=src.mean(),
               noise_mad=float(1.4826 * np.median(np.abs(r - med))) if good else np.nan)
    out.update(grp)
    return out

# ---- 1. QC ---------------------------------------------------------------------------------------
M = pd.read_csv(f'{ROOT}/data/manifests/astB_cutouts.csv')
M = M[M.status.isin(['ok', 'cached']) & (M.inside == True)].copy()
with ThreadPoolExecutor(4) as ex:
    rows = list(tqdm(ex.map(scan, M.local_file.tolist()), total=len(M)))
Q = M.merge(pd.DataFrame(rows), on='local_file')
Q['pass_gate'] = ((Q.dqa == 'Pass') & (Q.finast == 0) & (Q.bad_frac <= 0.25) &
                  (Q.src_frac <= 0.90) & (Q.coverage >= 0.90))
print('gate failures by criterion:', dict(dqa=int((Q.dqa != 'Pass').sum()), finast=int((Q.finast != 0).sum()),
      bad=int((Q.bad_frac > 0.25).sum()), src=int((Q.src_frac > 0.90).sum()), cov=int((Q.coverage < 0.90).sum())))
print(len(Q), 'frames,', int(Q.pass_gate.sum()), 'pass the gate; pass rate by version:')
print(Q.groupby('ver').pass_gate.agg(frames='size', pass_rate='mean').round(2).T.to_string())
Q.to_csv(f'{ROOT}/data/manifests/astB_qc.csv', index=False)

# ---- 2. photometry: mover position + 5 displaced control tracks per visit ------------------------
R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11
META = ['bad_frac', 'src_frac', 'noise_mad', 'persist_frac', 'cr_frac', 'outlier_frac', 'badpix_frac']

def meas(im, var, bad, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny: return None
    I, V, B, S = (a[ya:yb, xa:xb] for a in (im, var, bad, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~B & np.isfinite(I)
    ok = ap & ~B & np.isfinite(I) & np.isfinite(V) & (V > 0)
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum(): return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return flux, noise, bool(S[ap].any())

P = Q[Q.pass_gate].copy()
P = P[P.groupby('visit').obs_id.transform('nunique') >= 4]
rng = np.random.default_rng(0)
shifts = {v: [(r * np.cos(a), r * np.sin(a)) for r, a in zip(rng.uniform(15, 40, 5), rng.uniform(0, 2 * np.pi, 5))]
          for v in P.visit.unique()}
print(len(P), 'frames in', P.visit.nunique(), 'visits after the gate and the >=4-frame rule')

rows, fails = [], {'mover': 0, 'ctrl': 0}
for fn, g in tqdm(P.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = h['IMAGE'].data.astype(float); var = h['VARIANCE'].data.astype(float)
        bad, src, _ = masks(h['FLAGS'].data, h['FLAGS'].header)
    for r in g.itertuples():
        for kind, (dx, dy) in [('mover', (0, 0))] + [(f'ctrl{c}', s) for c, s in enumerate(shifts[r.visit])]:
            m = meas(im, var, bad, src, r.x + dx, r.y + dy)
            if m is None: fails['mover' if kind == 'mover' else 'ctrl'] += 1; continue
            rows.append(dict(visit=r.visit, kind=kind, ast=r.ast, V=r.V, ver=r.ver, lam=r.lam_center,
                             det=r.detector, mjd=r.mjd, flux=m[0], noise=m[1], on_src=m[2],
                             local_file=fn, **{c: getattr(r, c) for c in META}))
F = pd.DataFrame(rows)
nm = int(P.shape[0])
print('measurement failures (mover / control):', fails, f'(of about {nm} / {5 * nm} measurements)')

# cap at 6 random frames per visit (the backend sends at most 6)
rs = np.random.default_rng(1); sel = {}
for v, g in F[F.kind == 'mover'].groupby('visit'):
    files = g.local_file.unique(); sel[v] = set(rs.choice(files, min(6, len(files)), replace=False))
F = F[[fn in sel.get(v, ()) for v, fn in zip(F.visit, F.local_file)]]
F.to_csv(f'{ROOT}/data/manifests/astB_frames.csv', index=False)

# ---- 3. one row per track ------------------------------------------------------------------------
def feats(d):
    s = (d.flux / d.noise).values; fx, nz = d.flux.values, d.noise.values
    top = int(np.argmax(s)); rest = np.delete(np.arange(len(s)), top); pos = np.clip(s, 0, None)
    return pd.Series(dict(n=len(d), track_snr=fx.sum() / np.sqrt((nz ** 2).sum()),
        snr_wo_top=fx[rest].sum() / np.sqrt((nz[rest] ** 2).sum()), med_snr=np.median(s),
        frac_gt3=(s > 3).mean(), top_share=pos.max() / (pos.sum() + 1e-9),
        src_frac_ap=d.on_src.mean(), lam_mean=d.lam.mean(), lam_span=d.lam.max() - d.lam.min(),
        n_det=d.det.nunique(), span_h=(d.mjd.max() - d.mjd.min()) * 24,
        **{f'm_{c}': d[c].mean() for c in META}, V=d.V.median(), ast=d.ast.iloc[0], ver=d.ver.iloc[0]))
T = F.groupby(['visit', 'kind']).apply(feats).reset_index()
T = T[T.n >= 4].reset_index(drop=True)
T['y'] = (T.kind == 'mover').astype(int)
SL = ['track_snr', 'snr_wo_top', 'med_snr']
for c in SL: T[c + '_l'] = np.sign(T[c]) * np.log1p(np.abs(T[c]))
T['Vbin'] = pd.cut(T.V, [14.5, 16, 17, 18.5]).astype(str)
T['vgrp'] = T.ver.where(T.ver.isin(['l2b-v20', 'l2b-v24', 'l2b-v27']), 'other')
T.to_csv(f'{ROOT}/data/manifests/astB_track_table.csv', index=False)
pos = T[T.y == 1]
print('positive tracks:', len(pos), '| negatives:', int((T.y == 0).sum()),
      '| by V bin:', pos.Vbin.value_counts().to_dict(), '| by version:', pos.vgrp.value_counts().to_dict())
print('median track S/N by V bin (mover=1 / control=0):')
print(T.groupby(['Vbin', 'y']).track_snr.median().unstack().round(2).to_string())
print('median quality metadata by version (watch for a version proxy):')
print(T.groupby('vgrp')[['m_bad_frac', 'm_src_frac', 'm_noise_mad', 'src_frac_ap']].median().round(3).to_string())

# ---- 4. comparison -------------------------------------------------------------------------------
IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
FLG = ['src_frac_ap'] + [f'm_{c}' for c in META]
LAM = ['lam_mean', 'lam_span', 'n_det', 'span_h']
SETS = {'image-only (consistency)': IMG, '+ quality flags': IMG + FLG,
        '+ wavelength/time': IMG + LAM, '+ both (full)': IMG + FLG + LAM}

def fit(X, y, C=0.5):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med))
    return med, sc, LogisticRegression(C=C, class_weight='balanced', max_iter=3000).fit(sc.transform(X.fillna(med)), y)

def oof(cols):
    p = np.zeros(len(T))
    for a in T.ast.unique():
        tr, te = (T.ast != a).values, (T.ast == a).values
        med, sc, lr = fit(T.loc[tr, cols], T.y[tr])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

y, g = T.y.values, T.ast.values
def met(p, y, far=0.05, rec=0.9):
    thr = np.quantile(p[y == 0], 1 - far)
    return roc_auc_score(y, p), (p[y == 1] >= thr).mean(), (p[y == 0] >= np.quantile(p[y == 1], 1 - rec)).mean()

def boot(p, B=300):
    rg = np.random.default_rng(0); ug = np.unique(g); idx = {a: np.where(g == a)[0] for a in ug}; bs = []
    for _ in range(B):
        i = np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])
        if y[i].sum() >= 5 and (y[i] == 0).sum() >= 20: bs.append(met(p[i], y[i]))
    return np.percentile(bs, [2.5, 97.5], axis=0)

methods = {'rule: track S/N': T.track_snr.values, 'rule: S/N without top frame': T.snr_wo_top.values}
for k, cols in SETS.items(): methods[k] = oof(cols)

res = []
for k, p in methods.items():
    a, r5, f90 = met(p, y); lo, hi = boot(p)
    thr = np.quantile(p[y == 0], 0.95)
    byb = {b: (p[(y == 1) & (T.Vbin == b).values] >= thr).mean() for b in sorted(T.Vbin.unique())}
    res.append({'method': k, 'AUC': f'{a:.3f}', 'recall @5% false alerts': f'{r5:.2f} [{lo[1]:.2f}-{hi[1]:.2f}]',
                'false alerts @90% recall': f'{f90:.3f} [{lo[2]:.3f}-{hi[2]:.3f}]',
                **{f'recall V{b}': f'{v:.2f}' for b, v in byb.items()}})
print('\n' + pd.DataFrame(res).to_string(index=False))

pbest = methods['+ both (full)']; thr = np.quantile(pbest[y == 0], 0.95)
print('\nfull model: recall at 5% false alerts, by processing version:')
print(pd.DataFrame({'vgrp': T.vgrp[y == 1].values, 'hit': pbest[y == 1] >= thr}).groupby('vgrp').hit.agg(['mean', 'size']).round(2).to_string())

# ---- 5. export (numpy-only scorer, 50 bootstrap models) + sanity check on the bright set A ----------
cols = SETS['+ both (full)']
med, sc, _ = fit(T[cols], T.y); Z = sc.transform(T[cols].fillna(med))
rb = np.random.default_rng(1); coefs, ints = [], []
while len(coefs) < 50:
    i = rb.integers(0, len(T), len(T))
    if T.y.values[i].sum() < 5: continue
    lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000).fit(Z[i], T.y.values[i])
    coefs.append(lr.coef_[0].tolist()); ints.append(float(lr.intercept_[0]))
model = dict(version='ranker_B_v1', features=cols, signed_log=[c + '_l' for c in SL], fill=med.to_dict(),
             mean=sc.mean_.tolist(), scale=sc.scale_.tolist(), coefs=coefs, intercepts=ints)
os.makedirs(f'{ROOT}/models', exist_ok=True)
json.dump(model, open(f'{ROOT}/models/ranker_B_v1.json', 'w'))
print('\nsaved models/ranker_B_v1.json')

def apply(df, m):
    X = df[m['features']].fillna(pd.Series(m['fill']))
    Zs = (X.values - np.array(m['mean'])) / np.array(m['scale'])
    return (1 / (1 + np.exp(-(Zs @ np.array(m['coefs']).T + np.array(m['intercepts']))))).mean(1)

fa = f'{ROOT}/data/manifests/astA_track_table.csv'
if os.path.exists(fa):
    TA = pd.read_csv(fa)
    TA = TA.rename(columns={'m_ghost_frac': '_unused', 'm_streak_frac': 'm_outlier_frac'})   # set A used the old groups
    for c in model['features']:
        if c not in TA: TA[c] = np.nan
    pa = apply(TA, model)
    thrA = np.quantile(apply(T[T.y == 0], model), 0.95)
    print(f'sanity, bright set A (features not in A are filled with medians, so read this as rough): '
          f'recall {np.mean(pa[TA.y.values == 1] >= thrA):.2f} ({int(TA.y.sum())} positives) | '
          f'control false alerts {np.mean(pa[TA.y.values == 0] >= thrA):.3f}')

  0%|          | 0/1301 [00:00<?, ?it/s]

gate failures by criterion: {'dqa': 0, 'finast': 0, 'bad': 20, 'src': 131, 'cov': 0}
1301 frames, 1153 pass the gate; pass rate by version:
ver        l2b-v20  l2b-v21  l2b-v22  l2b-v23  l2b-v24  l2b-v25  l2b-v27
frames      424.00    58.00    54.00    90.00   424.00   102.00   148.00
pass_rate     0.92     0.98     0.98     0.99     0.81     0.99     0.82
1145 frames in 89 visits after the gate and the >=4-frame rule


  0%|          | 0/1145 [00:00<?, ?it/s]

measurement failures (mover / control): {'mover': 251, 'ctrl': 108} (of about 1145 / 5725 measurements)
positive tracks: 81 | negatives: 403 | by V bin: {'(16.0, 17.0]': 36, '(14.5, 16.0]': 34, '(17.0, 18.5]': 11} | by version: {'l2b-v20': 28, 'other': 23, 'l2b-v24': 21, 'l2b-v27': 9}
median track S/N by V bin (mover=1 / control=0):
y                0      1
Vbin                     
(14.5, 16.0] -0.18  22.18
(16.0, 17.0]  0.11   8.96
(17.0, 18.5]  0.15   7.67
median quality metadata by version (watch for a version proxy):
         m_bad_frac  m_src_frac  m_noise_mad  src_frac_ap
vgrp                                                     
l2b-v20       0.027       0.399        0.031        0.667
l2b-v24       0.032       0.467        0.030        0.833
l2b-v27       0.036       0.398        0.033        1.000
other         0.036       0.220        0.038        0.367

                     method   AUC recall @5% false alerts false alerts @90% recall recall V(14.5, 16.0] recall V(16.0, 17.

In [ ]:
# Step B4: (a) why do mover measurements fail? (b) version confound check
from collections import defaultdict

# ---- (a) flags inside mover vs control apertures -------------------------------------------------
cnt = {'mover': defaultdict(int), 'ctrl': defaultdict(int)}
tot, fail = {'mover': 0, 'ctrl': 0}, {'mover': 0, 'ctrl': 0}
sub = P.sample(min(400, len(P)), random_state=0)
for fn, g_ in tqdm(sub.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = h['IMAGE'].data.astype(float); var = h['VARIANCE'].data.astype(float)
        fl = h['FLAGS'].data.astype(np.uint64); hdr = h['FLAGS'].header
    d = flagdefs(hdr); bad, src, _ = masks(fl, hdr)
    yy, xx = np.indices(im.shape)
    for r in g_.itertuples():
        for kind, (dx, dy) in (('mover', (0, 0)), ('ctrl', shifts[r.visit][0])):
            x, y_ = r.x + dx, r.y + dy
            ap = (xx - x) ** 2 + (yy - y_) ** 2 <= R_AP ** 2
            tot[kind] += 1
            if meas(im, var, bad, src, x, y_) is None: fail[kind] += 1
            for k, b in d.items():
                if ((fl[ap] >> np.uint64(b)) & np.uint64(1)).any(): cnt[kind][k] += 1
tab = pd.DataFrame({k: {n: cnt[k][n] / tot[k] for n in cnt[k]} for k in cnt}).fillna(0)
tab = tab[(tab > 0.02).any(axis=1)].sort_values('mover', ascending=False)
print('failed measurements:', {k: f'{fail[k]}/{tot[k]}' for k in fail})
print('share of apertures containing each flag (mover vs control):'); print(tab.round(2).to_string())

# ---- (b) version confound: asteroid-split vs version-split, with and without SOURCE features -----------
NOSRC = IMG + ['m_bad_frac', 'm_noise_mad', 'm_persist_frac', 'm_cr_frac', 'm_outlier_frac', 'm_badpix_frac']
SETS2 = {'image-only': IMG, 'flags without SOURCE features': NOSRC, 'full (B3 export)': IMG + FLG + LAM}

def oof_by(cols, key):
    p = np.zeros(len(T))
    for a in T[key].unique():
        tr, te = (T[key] != a).values, (T[key] == a).values
        med, sc, lr = fit(T.loc[tr, cols], T.y[tr])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

rows = []
a_, r5_, f90_ = met(T.snr_wo_top.values, y); lo, hi = boot(T.snr_wo_top.values)
rows.append(dict(model='rule: S/N without top frame', split='none', AUC=f'{a_:.3f}',
                 recall_at_5pct_FA=f'{r5_:.2f} [{lo[1]:.2f}-{hi[1]:.2f}]', FA_at_90pct_recall=f'{f90_:.3f}'))
for name, cols in SETS2.items():
    for split, key in (('leave-one-asteroid-out', 'ast'), ('leave-one-version-out', 'vgrp')):
        p = oof_by(cols, key); a_, r5_, f90_ = met(p, y); lo, hi = boot(p)
        rows.append(dict(model=name, split=split, AUC=f'{a_:.3f}',
                         recall_at_5pct_FA=f'{r5_:.2f} [{lo[1]:.2f}-{hi[1]:.2f}]', FA_at_90pct_recall=f'{f90_:.3f}'))
print('\n' + pd.DataFrame(rows).to_string(index=False))

  0%|          | 0/400 [00:00<?, ?it/s]

failed measurements: {'mover': '91/400', 'ctrl': '11/400'}
share of apertures containing each flag (mover vs control):
              mover  ctrl
MP_OUTLIER     0.70  0.19
MP_SOURCE      0.62  0.59
MP_TRANSIENT   0.05  0.05
MP_SUR_ERROR   0.05  0.04
MP_HALO        0.04  0.04
MP_NONFUNC     0.04  0.03
MP_PERSIST     0.03  0.02

                        model                  split   AUC recall_at_5pct_FA FA_at_90pct_recall
  rule: S/N without top frame                   none 0.940  0.56 [0.26-0.84]              0.114
                   image-only leave-one-asteroid-out 0.940  0.54 [0.25-0.95]              0.129
                   image-only  leave-one-version-out 0.937  0.53 [0.25-0.93]              0.134
flags without SOURCE features leave-one-asteroid-out 0.940  0.62 [0.29-0.91]              0.114
flags without SOURCE features  leave-one-version-out 0.935  0.64 [0.29-0.82]              0.134
             full (B3 export) leave-one-asteroid-out 0.944  0.75 [0.40-0.94]              0.112


In [ ]:
# Step B5: outlier-tolerant aperture + aperture-level outlier flag -> final comparison -> export v2
def meas2(im, var, bcore, bfull, outl, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny: return None
    I, V, BC, BF, O, S = (a[ya:yb, xa:xb] for a in (im, var, bcore, bfull, outl, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~BF & np.isfinite(I)        # background excludes OUTLIER pixels
    ok = ap & ~BC & np.isfinite(I) & np.isfinite(V) & (V > 0)               # aperture tolerates OUTLIER pixels
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum(): return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return flux, noise, bool(S[ap].any()), float(O[ap].mean())

rows, fails = [], {'mover': 0, 'ctrl': 0}
for fn, g_ in tqdm(P.groupby('local_file')):
    with fits.open(fn, memmap=False) as h:
        im = h['IMAGE'].data.astype(float); var = h['VARIANCE'].data.astype(float)
        fl = h['FLAGS'].data.astype(np.uint64); hdr = h['FLAGS'].header
    d = flagdefs(hdr); bfull, src, _ = masks(fl, hdr)
    core_mask = sum(1 << b for k, b in d.items() if k not in INFO and k != 'MP_OUTLIER')
    bcore = (fl & np.uint64(core_mask)) != 0
    outl = (((fl >> np.uint64(d['MP_OUTLIER'])) & np.uint64(1)).astype(bool)
            if 'MP_OUTLIER' in d else np.zeros(im.shape, bool))
    for r in g_.itertuples():
        for kind, (dx, dy) in [('mover', (0, 0))] + [(f'ctrl{c}', s) for c, s in enumerate(shifts[r.visit])]:
            m = meas2(im, var, bcore, bfull, outl, src, r.x + dx, r.y + dy)
            if m is None: fails['mover' if kind == 'mover' else 'ctrl'] += 1; continue
            rows.append(dict(visit=r.visit, kind=kind, ast=r.ast, V=r.V, ver=r.ver, lam=r.lam_center,
                             det=r.detector, mjd=r.mjd, flux=m[0], noise=m[1], on_src=m[2], outlier_ap=m[3],
                             local_file=fn, **{c: getattr(r, c) for c in META}))
F2 = pd.DataFrame(rows)
print('measurement failures (mover / control):', fails, f'| before (B3): mover 251, control 108')

rs = np.random.default_rng(1); sel = {}                                   # cap at 6 frames per visit, as the backend does
for v, g_ in F2[F2.kind == 'mover'].groupby('visit'):
    files = g_.local_file.unique(); sel[v] = set(rs.choice(files, min(6, len(files)), replace=False))
F2 = F2[[fn in sel.get(v, ()) for v, fn in zip(F2.visit, F2.local_file)]]
F2.to_csv(f'{ROOT}/data/manifests/astB_frames_v2.csv', index=False)

def feats2(d):
    s = (d.flux / d.noise).values; fx, nz = d.flux.values, d.noise.values
    top = int(np.argmax(s)); rest = np.delete(np.arange(len(s)), top); pos = np.clip(s, 0, None)
    return pd.Series(dict(n=len(d), track_snr=fx.sum() / np.sqrt((nz ** 2).sum()),
        snr_wo_top=fx[rest].sum() / np.sqrt((nz[rest] ** 2).sum()), med_snr=np.median(s),
        frac_gt3=(s > 3).mean(), top_share=pos.max() / (pos.sum() + 1e-9),
        outlier_ap=d.outlier_ap.mean(), outlier_frames=(d.outlier_ap > 0).mean(),
        src_frac_ap=d.on_src.mean(), lam_mean=d.lam.mean(), lam_span=d.lam.max() - d.lam.min(),
        n_det=d.det.nunique(), span_h=(d.mjd.max() - d.mjd.min()) * 24,
        **{f'm_{c}': d[c].mean() for c in META}, V=d.V.median(), ast=d.ast.iloc[0], ver=d.ver.iloc[0]))
T = F2.groupby(['visit', 'kind']).apply(feats2).reset_index()
T = T[T.n >= 4].reset_index(drop=True)
T['y'] = (T.kind == 'mover').astype(int)
for c in SL: T[c + '_l'] = np.sign(T[c]) * np.log1p(np.abs(T[c]))
T['Vbin'] = pd.cut(T.V, [14.5, 16, 17, 18.5]).astype(str)
T['vgrp'] = T.ver.where(T.ver.isin(['l2b-v20', 'l2b-v24', 'l2b-v27']), 'other')
T.to_csv(f'{ROOT}/data/manifests/astB_track_table_v2.csv', index=False)
print('positive tracks:', int(T.y.sum()), '| negatives:', int((T.y == 0).sum()), '| by V bin:',
      T[T.y == 1].Vbin.value_counts().to_dict())
print('aperture OUTLIER share, mean over tracks (mover / control):',
      T.groupby('y').outlier_ap.mean().round(2).to_dict())

# ---- comparison ----------------------------------------------------------------------------------
IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
FLGN = ['m_bad_frac', 'm_noise_mad', 'm_persist_frac', 'm_cr_frac', 'm_outlier_frac', 'm_badpix_frac']
OUT = ['outlier_ap', 'outlier_frames']
LAM = ['lam_mean', 'lam_span', 'n_det', 'span_h']
SETS = {'image-only (consistency)': IMG, '+ cutout quality flags': IMG + FLGN,
        '+ aperture outlier flag': IMG + OUT, '+ both flags (FINAL)': IMG + FLGN + OUT,
        '+ both flags + wavelength/time': IMG + FLGN + OUT + LAM}

def fit(X, y, C=0.5):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med))
    return med, sc, LogisticRegression(C=C, class_weight='balanced', max_iter=3000).fit(sc.transform(X.fillna(med)), y)

def oof_by(cols, key):
    p = np.zeros(len(T))
    for a in T[key].unique():
        tr, te = (T[key] != a).values, (T[key] == a).values
        med, sc, lr = fit(T.loc[tr, cols], T.y[tr])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

y, g = T.y.values, T.ast.values
def met(p, y, far=0.05, rec=0.9):
    thr = np.quantile(p[y == 0], 1 - far)
    return roc_auc_score(y, p), (p[y == 1] >= thr).mean(), (p[y == 0] >= np.quantile(p[y == 1], 1 - rec)).mean()

def boot(p, B=300):
    rg = np.random.default_rng(0); ug = np.unique(g); idx = {a: np.where(g == a)[0] for a in ug}; bs = []
    for _ in range(B):
        i = np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])
        if y[i].sum() >= 5 and (y[i] == 0).sum() >= 20: bs.append(met(p[i], y[i]))
    return np.percentile(bs, [2.5, 97.5], axis=0)

def row(name, split, p):
    a, r5, f90 = met(p, y); lo, hi = boot(p); thr = np.quantile(p[y == 0], 0.95)
    byb = {b: (p[(y == 1) & (T.Vbin == b).values] >= thr).mean() for b in sorted(T.Vbin.unique())}
    return {'method': name, 'split': split, 'AUC': f'{a:.3f}',
            'recall @5% FA': f'{r5:.2f} [{lo[1]:.2f}-{hi[1]:.2f}]', 'FA @90% recall': f'{f90:.3f} [{lo[2]:.3f}-{hi[2]:.3f}]',
            **{f'rec V{b}': f'{v:.2f}' for b, v in byb.items()}}

res = [row('rule: track S/N', '-', T.track_snr.values), row('rule: S/N without top frame', '-', T.snr_wo_top.values),
       row('rule: aperture outlier flag alone', '-', T.outlier_ap.values)]
for k, cols in SETS.items():
    res.append(row(k, 'asteroid', oof_by(cols, 'ast')))
print('\n' + pd.DataFrame(res).to_string(index=False))
print('\nversion split (leave-one-version-out) for image-only vs FINAL:')
print(pd.DataFrame([row(k, 'version', oof_by(SETS[k], 'vgrp')) for k in ('image-only (consistency)', '+ both flags (FINAL)')]
                   )[['method', 'split', 'AUC', 'recall @5% FA', 'FA @90% recall']].to_string(index=False))

# ---- export: the FINAL feature set was fixed before these results ---------------------------------------
cols = SETS['+ both flags (FINAL)']
med, sc, _ = fit(T[cols], T.y); Z = sc.transform(T[cols].fillna(med))
rb = np.random.default_rng(1); coefs, ints = [], []
while len(coefs) < 50:
    i = rb.integers(0, len(T), len(T))
    if T.y.values[i].sum() < 5: continue
    lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000).fit(Z[i], T.y.values[i])
    coefs.append(lr.coef_[0].tolist()); ints.append(float(lr.intercept_[0]))
json.dump(dict(version='ranker_B_v2', features=cols, signed_log=[c + '_l' for c in SL], fill=med.to_dict(),
               mean=sc.mean_.tolist(), scale=sc.scale_.tolist(), coefs=coefs, intercepts=ints),
          open(f'{ROOT}/models/ranker_B_v2.json', 'w'))
print('\nsaved models/ranker_B_v2.json')

  0%|          | 0/1145 [00:00<?, ?it/s]

measurement failures (mover / control): {'mover': 9, 'ctrl': 101} | before (B3): mover 251, control 108
positive tracks: 89 | negatives: 445 | by V bin: {'(14.5, 16.0]': 41, '(16.0, 17.0]': 37, '(17.0, 18.5]': 11}
aperture OUTLIER share, mean over tracks (mover / control): {0: 0.02, 1: 0.22}

                           method    split   AUC    recall @5% FA      FA @90% recall rec V(14.5, 16.0] rec V(16.0, 17.0] rec V(17.0, 18.5]
                  rule: track S/N        - 0.964 0.78 [0.50-0.98] 0.070 [0.025-0.136]              0.98              0.70              0.27
      rule: S/N without top frame        - 0.973 0.84 [0.50-1.00] 0.056 [0.011-0.110]              0.98              0.84              0.36
rule: aperture outlier flag alone        - 0.864 0.79 [0.67-0.89] 1.000 [0.061-1.000]              0.85              0.76              0.64
         image-only (consistency) asteroid 0.968 0.89 [0.58-0.99] 0.058 [0.009-0.100]              0.98              0.86              0.64
      

In [ ]:
# B6, cell 1: production module (features + scorer). Needs only ROOT.
import os, sys, importlib
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')   # change if your path differs
PKG = f'{ROOT}/package/aegis_ranker'; os.makedirs(PKG, exist_ok=True)

SRC = r'''
# AegisEarth candidate ranker: evidence builder + scorer. Needs only numpy and astropy.
import json, re, warnings
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11            # aperture and background annulus radii (pixels), window half-size
MIN_FRAMES, MAX_FRAMES = 4, 6
INFO = {'MP_SOURCE', 'MP_FULLSAMPLE'}                 # informational flags: never rejected
GROUPS = {'cr': ['TRANSIENT', 'OVERFLOW'], 'outlier': ['OUTLIER'], 'persist': ['PERSIST'],
          'dichroic': ['DICHROIC'], 'badpix': ['NONFUNC', 'MISSING_DATA', 'HOT', 'COLD', 'PHANMISS', 'NONLINEAR']}
SIGNED_LOG = ['track_snr', 'snr_wo_top', 'med_snr']
CUTOUT_META = ['bad_frac', 'noise_mad', 'persist_frac', 'cr_frac', 'outlier_frac', 'badpix_frac']


def flagdefs(header):
    d = {k: int(header[k]) for k in header if k.startswith('MP_')}
    for k in header:
        if re.fullmatch(r'MSKN\d{4}', k):
            d['MP_' + str(header[k])] = int(header['MSKM' + k[4:]]).bit_length() - 1
    return d


def masks(fl, header):
    d = flagdefs(header); fl = fl.astype(np.uint64)
    bad = (fl & np.uint64(sum(1 << b for k, b in d.items() if k not in INFO))) != 0
    src = ((fl >> np.uint64(d['MP_SOURCE'])) & np.uint64(1)).astype(bool)
    core = (fl & np.uint64(sum(1 << b for k, b in d.items() if k not in INFO and k != 'MP_OUTLIER'))) != 0
    if 'MP_OUTLIER' in d:
        outl = ((fl >> np.uint64(d['MP_OUTLIER'])) & np.uint64(1)).astype(bool)
    else:
        outl = np.zeros(fl.shape, bool)
    grp = {}
    for g, names in GROUPS.items():
        m = sum(1 << d['MP_' + n] for n in names if 'MP_' + n in d)
        grp[g + '_frac'] = float(((fl & np.uint64(m)) != 0).mean()) if m else 0.0
    return bad, core, outl, src, grp


def _meas(im, var, bcore, bfull, outl, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny:
        return None
    I, V, BC, BF, O, S = (a[ya:yb, xa:xb] for a in (im, var, bcore, bfull, outl, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~BF & np.isfinite(I)      # background excludes every flagged pixel
    ok = ap & ~BC & np.isfinite(I) & np.isfinite(V) & (V > 0)                # aperture tolerates OUTLIER pixels
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum():
        return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return float(flux), float(noise), bool(S[ap].any()), float(O[ap].mean())


def analyze_frame(path, ra, dec, shift=(0.0, 0.0)):
    # shift (pixels) is only used to build displaced control tracks for testing; production uses (0, 0)
    with fits.open(path, memmap=False) as h:
        im = h['IMAGE'].data.astype(float); var = h['VARIANCE'].data.astype(float)
        fl = h['FLAGS'].data; fh = h['FLAGS'].header
        hd = h['IMAGE'].header; wcs = WCS(hd)
    bad, core, outl, src, grp = masks(fl, fh)
    ny, nx = im.shape
    out = dict(path=path, dqa=hd.get('L2DQAFLG'), finast=hd.get('FINAST'),
               coverage=float(np.isfinite(im).mean()), bad_frac=float(bad.mean()), src_frac=float(src.mean()))
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    if ok.sum() > 100:
        r = im[ok]; med = np.median(r); out['noise_mad'] = float(1.4826 * np.median(np.abs(r - med)))
    else:
        out['noise_mad'] = float('nan')
    out.update(grp)
    out['qc_pass'] = bool(out['dqa'] == 'Pass' and out['finast'] == 0 and out['bad_frac'] <= 0.25
                          and out['src_frac'] <= 0.90 and out['coverage'] >= 0.90)
    x, y = wcs.world_to_pixel(SkyCoord(ra, dec, unit='deg'))
    out['inside'] = bool(np.isfinite(x) and np.isfinite(y) and 12 <= x <= nx - 13 and 12 <= y <= ny - 13)
    out['meas'] = _meas(im, var, core, bad, outl, src, x + shift[0], y + shift[1]) if out['inside'] else None
    return out


def track_features(frames):
    flux = np.array([f['meas'][0] for f in frames]); noise = np.array([f['meas'][1] for f in frames])
    s = flux / noise; top = int(np.argmax(s)); rest = np.delete(np.arange(len(s)), top)
    pos = np.clip(s, 0, None); outap = np.array([f['meas'][3] for f in frames])
    feat = dict(n=len(frames), track_snr=float(flux.sum() / np.sqrt((noise ** 2).sum())),
                snr_wo_top=float(flux[rest].sum() / np.sqrt((noise[rest] ** 2).sum())),
                med_snr=float(np.median(s)), frac_gt3=float((s > 3).mean()),
                top_share=float(pos.max() / (pos.sum() + 1e-9)),
                outlier_ap=float(outap.mean()), outlier_frames=float((outap > 0).mean()))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for c in CUTOUT_META:
            feat['m_' + c] = float(np.nanmean([f[c] for f in frames]))
    feat['src_frac_ap'] = float(np.mean([f['meas'][2] for f in frames]))      # warning only, not a model feature
    for c in SIGNED_LOG:
        feat[c + '_l'] = float(np.sign(feat[c]) * np.log1p(abs(feat[c])))
    return feat


def load_model(path):
    return json.load(open(path))


def score(feat, model):
    x = []
    for k in model['features']:
        v = feat.get(k)
        x.append(model['fill'][k] if v is None or not np.isfinite(v) else v)
    z = (np.array(x, float) - np.array(model['mean'])) / np.array(model['scale'])
    p = 1 / (1 + np.exp(-(np.array(model['coefs']) @ z + np.array(model['intercepts']))))
    return float(p.mean()), float(p.std())


def rank(frames, model, shift=(0.0, 0.0)):
    # frames: list of dicts with path, ra, dec (degrees), optional mjd. Returns a dict (score None = not scored).
    warns, used = [], []
    frames = sorted(frames, key=lambda f: f.get('mjd') or 0.0)
    if len(frames) > MAX_FRAMES:
        warns.append('EXTRA_FRAMES_IGNORED'); frames = frames[:MAX_FRAMES]
    for f in frames:
        a = analyze_frame(f['path'], f['ra'], f['dec'], shift)
        if not a['qc_pass']: warns.append('FRAME_FAILED_QC')
        elif not a['inside']: warns.append('POSITION_NEAR_EDGE_OR_OUTSIDE')
        elif a['meas'] is None: warns.append('APERTURE_UNMEASURABLE')
        else: used.append(a)
    res = dict(score=None, uncertainty=None, score_kind='candidate_score',
               model_version_id=model.get('version', 'unknown'), explanation_codes=[],
               warnings=[], evidence=None, n_frames_used=len(used))
    if len(used) < MIN_FRAMES:
        res['warnings'] = sorted(set(warns + ['TOO_FEW_USABLE_FRAMES']))
        return res
    feat = track_features(used); s, u = score(feat, model); codes = []
    if feat['snr_wo_top'] >= 3: codes.append('CONSISTENT_ACROSS_FRAMES')
    elif feat['track_snr'] >= 3: codes.append('SINGLE_FRAME_DOMINATES')
    else: codes.append('LOW_TRACK_SNR')
    if feat['outlier_frames'] >= 0.5: codes.append('PIPELINE_OUTLIER_FLAG_AT_POSITION')
    if feat['src_frac_ap'] >= 0.5: warns.append('APERTURE_ON_CATALOG_SOURCE')
    res.update(score=s, uncertainty=u, explanation_codes=codes, warnings=sorted(set(warns)), evidence=feat)
    return res
'''
open(f'{PKG}/features.py', 'w').write(SRC.lstrip())
open(f'{PKG}/__init__.py', 'w').write('')
sys.path.insert(0, f'{ROOT}/package')
import aegis_ranker.features as AF; importlib.reload(AF)
print('module written,', os.path.getsize(f'{PKG}/features.py'), 'bytes; functions:',
      [n for n in ('analyze_frame', 'track_features', 'score', 'rank', 'load_model') if hasattr(AF, n)])

module written, 7332 bytes; functions: ['analyze_frame', 'track_features', 'score', 'rank', 'load_model']


In [ ]:
# B6, cell 2: parity test against the training table, then golden cases for the backend
import os, json, shutil, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
import aegis_ranker.features as AF

T = pd.read_csv(f'{ROOT}/data/manifests/astB_track_table_v2.csv')
F2 = pd.read_csv(f'{ROOT}/data/manifests/astB_frames_v2.csv')
Q = pd.read_csv(f'{ROOT}/data/manifests/astB_qc.csv')
model = AF.load_model(f'{ROOT}/models/ranker_B_v2.json')
Qd = Q.drop_duplicates('local_file').set_index('local_file')

# rebuild the same displaced-control shifts as in training (same seed, same visit order)
P = Q[Q.pass_gate == True]
P = P[P.groupby('visit').obs_id.transform('nunique') >= 4]
rg = np.random.default_rng(0)
shifts = {v: [(r * np.cos(a), r * np.sin(a)) for r, a in zip(rg.uniform(15, 40, 5), rg.uniform(0, 2 * np.pi, 5))]
          for v in P.visit.unique()}

def frames_for(visit, kind):
    files = F2[(F2.visit == visit) & (F2.kind == kind)].local_file.unique()
    fr = [dict(path=fn, ra=float(Qd.loc[fn, 'ra']), dec=float(Qd.loc[fn, 'dec']), mjd=float(Qd.loc[fn, 'mjd']))
          for fn in files]
    return sorted(fr, key=lambda d: d['mjd'])

def shift_for(visit, kind):
    return (0.0, 0.0) if kind == 'mover' else shifts[visit][int(kind[4:])]

def dif(a, b):
    a, b = float(a), float(b)
    return 0.0 if (np.isnan(a) and np.isnan(b)) else abs(a - b)

# ---- parity: 20 real-mover tracks + 20 displaced-control tracks -------------------------------------
sample = pd.concat([T[T.y == 1].sample(20, random_state=5), T[T.y == 0].sample(20, random_state=5)])
rows, notscored = [], []
for r in tqdm(sample.itertuples(), total=len(sample)):
    res = AF.rank(frames_for(r.visit, r.kind), model, shift=shift_for(r.visit, r.kind))
    if res['score'] is None:
        notscored.append((r.visit, r.kind, res['warnings'])); continue
    trow = T[(T.visit == r.visit) & (T.kind == r.kind)].iloc[0].to_dict()
    d = {k: dif(res['evidence'][k], trow[k]) for k in model['features']}
    s_ref, _ = AF.score(trow, model)
    rows.append(dict(visit=r.visit, kind=r.kind, max_feat_diff=max(d.values()), worst=max(d, key=d.get),
                     score_diff=abs(res['score'] - s_ref), warnings=res['warnings']))
D = pd.DataFrame(rows)
print(f'tracks compared: {len(D)} | not scored by the module: {len(notscored)} {notscored[:3]}')
if len(D):
    print('max feature difference:', f"{D.max_feat_diff.max():.2e}", '| max score difference:', f"{D.score_diff.max():.2e}")
    print(D.sort_values('max_feat_diff', ascending=False).head(5).to_string(index=False))
PARITY_OK = bool(len(D) and not notscored and D.max_feat_diff.max() < 1e-6 and D.score_diff.max() < 1e-6)
print('\nPARITY', 'PASSED' if PARITY_OK else 'FAILED')

# ---- golden cases (only if parity passed) ---------------------------------------------------------------
if PARITY_OK:
    gdir = f'{ROOT}/package/golden'; os.makedirs(f'{gdir}/fits', exist_ok=True)
    cases = []

    def to_json(o):
        if isinstance(o, dict): return {k: to_json(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [to_json(v) for v in o]
        if isinstance(o, (float, np.floating)): return None if not np.isfinite(o) else float(o)
        if isinstance(o, np.integer): return int(o)
        return o

    def add_case(name, frames, shift, note):
        fr_json = []
        for f in frames:
            base = os.path.basename(f['path']); shutil.copy(f['path'], f'{gdir}/fits/{base}')
            fr_json.append(dict(file='fits/' + base, ra=f['ra'], dec=f['dec'], mjd=f.get('mjd')))
        res = AF.rank(frames, model, shift=tuple(shift))
        cases.append(dict(name=name, note=note, shift=list(shift), frames=fr_json, expected=to_json(res)))

    for b in sorted(T.Vbin.unique()):
        pr = T[(T.y == 1) & (T.Vbin == b)].iloc[0]
        add_case(f'real_mover_V{b}', frames_for(pr.visit, 'mover'), (0.0, 0.0), 'known asteroid track')
        cr = T[(T.visit == pr.visit) & (T.y == 0)].iloc[0]
        add_case(f'displaced_control_V{b}', frames_for(pr.visit, cr.kind), shift_for(pr.visit, cr.kind),
                 'same frames, path displaced by 15-40 px (shift is only for building controls)')
    pr0 = T[T.y == 1].iloc[0]; base = frames_for(pr0.visit, 'mover')
    add_case('edge_too_few_frames', base[:3], (0.0, 0.0), 'only 3 frames: no score expected')
    add_case('edge_position_outside', [dict(f, ra=f['ra'] + 1.0) for f in base], (0.0, 0.0),
             'position 1 degree away: outside the cutout')
    bad = Q[(Q.pass_gate == False) & (Q.inside == True)].iloc[0]
    add_case('edge_qc_failed_frame', base[:3] + [dict(path=bad.local_file, ra=float(bad.ra), dec=float(bad.dec),
             mjd=float(bad.mjd))], (0.0, 0.0), 'one frame fails QC and is dropped, leaving too few frames')
    json.dump(dict(tolerance=dict(abs=1e-6), cases=cases), open(f'{gdir}/cases.json', 'w'), indent=1)

    CHECK = r'''
import json, sys, os
here = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, here + '/..')
import aegis_ranker.features as AF
model = AF.load_model(sys.argv[1])
cases = json.load(open(here + '/cases.json')); tol = cases['tolerance']; ok_all = True
for c in cases['cases']:
    frames = [dict(path=here + '/' + f['file'], ra=f['ra'], dec=f['dec'], mjd=f.get('mjd') or 0.0) for f in c['frames']]
    got = AF.rank(frames, model, shift=tuple(c['shift'])); exp = c['expected']; errs = []
    if (got['score'] is None) != (exp['score'] is None): errs.append('scored/unscored mismatch')
    elif got['score'] is not None:
        for k in ('score', 'uncertainty'):
            if abs(got[k] - exp[k]) > tol['abs']: errs.append(k + ' differs')
        for k, v in exp['evidence'].items():
            if v is not None and abs(got['evidence'][k] - v) > tol['abs']: errs.append('evidence ' + k + ' differs')
    if sorted(got['warnings']) != sorted(exp['warnings']): errs.append('warnings differ')
    if sorted(got['explanation_codes']) != sorted(exp['explanation_codes']): errs.append('explanation codes differ')
    print(c['name'], 'OK' if not errs else 'FAILED: ' + '; '.join(errs)); ok_all = ok_all and not errs
sys.exit(0 if ok_all else 1)
'''
    open(f'{gdir}/check_golden.py', 'w').write(CHECK.lstrip())
    mb = sum(os.path.getsize(f'{gdir}/fits/{f}') for f in os.listdir(f'{gdir}/fits')) / 1e6
    print(f'\n{len(cases)} golden cases written ({mb:.0f} MB of FITS):')
    for c in cases:
        e = c['expected']
        print(f"  {c['name']:34s} score={e['score'] if e['score'] is None else round(e['score'], 3)} "
              f"codes={e['explanation_codes']} warnings={e['warnings']}")

  0%|          | 0/40 [00:00<?, ?it/s]

tracks compared: 40 | not scored by the module: 0 []
max feature difference: 1.44e-08 | max score difference: 6.14e-09
 visit  kind  max_feat_diff       worst   score_diff                     warnings
100167 mover   1.441240e-08 m_noise_mad 8.033636e-10                           []
100247 ctrl3   8.951426e-09 m_noise_mad 1.719030e-11 [APERTURE_ON_CATALOG_SOURCE]
100089 ctrl3   7.545352e-09 m_noise_mad 7.147813e-11                           []
100086 mover   5.915761e-09 m_noise_mad 2.372658e-11 [APERTURE_ON_CATALOG_SOURCE]
100324 ctrl2   5.394816e-09 m_noise_mad 1.674741e-11                           []

PARITY PASSED

9 golden cases written (95 MB of FITS):
  real_mover_V(14.5, 16.0]           score=1.0 codes=['CONSISTENT_ACROSS_FRAMES', 'PIPELINE_OUTLIER_FLAG_AT_POSITION'] warnings=[]
  displaced_control_V(14.5, 16.0]    score=0.003 codes=['LOW_TRACK_SNR'] warnings=['APERTURE_ON_CATALOG_SOURCE']
  real_mover_V(16.0, 17.0]           score=0.999 codes=['CONSISTENT_ACROSS_FRAMES', 'PIPE

In [ ]:
# B7: operating points, paired significance test, package assembly, model card, golden self-check
import os, json, time, hashlib, platform, subprocess, sys, shutil, datetime, importlib, warnings
import numpy as np, pandas as pd, sklearn, astropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK = f'{ROOT}/package'
T = pd.read_csv(f'{ROOT}/data/manifests/astB_track_table_v2.csv')
F2 = pd.read_csv(f'{ROOT}/data/manifests/astB_frames_v2.csv')
model = json.load(open(f'{ROOT}/models/ranker_B_v2.json'))
FINAL = model['features']
IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
y, g = T.y.values, T.ast.values
ug = np.unique(g); idx = {a: np.where(g == a)[0] for a in ug}

# ---- out-of-fold scores using the PRODUCTION procedure (ensemble of 50 bootstrap models) ----------------
def fit_ens(X, yy, seed, nb=50):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med)); Z = sc.transform(X.fillna(med))
    rb = np.random.default_rng(seed); co, ic = [], []
    while len(co) < nb:
        i = rb.integers(0, len(X), len(X))
        if yy[i].sum() < 3: continue
        lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000).fit(Z[i], yy[i])
        co.append(lr.coef_[0]); ic.append(lr.intercept_[0])
    return med, sc, np.array(co), np.array(ic)

def oof_ens(cols):
    p = np.zeros(len(T))
    for a in ug:
        te = g == a; tr = ~te
        med, sc, co, ic = fit_ens(T.loc[tr, cols], y[tr], seed=int(a))
        Z = sc.transform(T.loc[te, cols].fillna(med))
        p[te] = (1 / (1 + np.exp(-(Z @ co.T + ic)))).mean(1)
    return p

pF, pI, pR = oof_ens(FINAL), oof_ens(IMG), T.snr_wo_top.values
print('out-of-fold scores done:', len(T), 'tracks,', int(y.sum()), 'positives,', len(ug), 'asteroids')

def met2(p, yy):                                      # recall at 5% false alerts, false alerts at 90% recall
    thr = np.quantile(p[yy == 0], 0.95)
    return (p[yy == 1] >= thr).mean(), (p[yy == 0] >= np.quantile(p[yy == 1], 0.10)).mean()

def resample(rg):
    return np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])

def boot_ci(p, B=1000, seed=3):
    rg = np.random.default_rng(seed); bs = []
    for _ in range(B):
        i = resample(rg)
        if y[i].sum() < 5 or (y[i] == 0).sum() < 20: continue
        bs.append(met2(p[i], y[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); r5, f90 = met2(p, y)
    return dict(auc=roc_auc_score(y, p), rec5=r5, rec5_ci=(lo[0], hi[0]), fa90=f90, fa90_ci=(lo[1], hi[1]))

def paired(pa, pb, B=2000, seed=7):
    rg = np.random.default_rng(seed); d = []
    for _ in range(B):
        i = resample(rg)
        if y[i].sum() < 5 or (y[i] == 0).sum() < 20: continue
        a_, b_ = met2(pa[i], y[i]), met2(pb[i], y[i]); d.append((a_[0] - b_[0], a_[1] - b_[1]))
    d = np.array(d)
    return dict(n=len(d), d_rec=d[:, 0].mean(), d_rec_ci=np.percentile(d[:, 0], [2.5, 97.5]), rec_better=(d[:, 0] > 0).mean(),
                d_fa=d[:, 1].mean(), d_fa_ci=np.percentile(d[:, 1], [2.5, 97.5]), fa_better=(d[:, 1] < 0).mean())

# ---- 1. methods with intervals -----------------------------------------------------------------------
R = {'rule: S/N without top frame': boot_ci(pR), 'image-only ranker': boot_ci(pI), 'FINAL ranker': boot_ci(pF)}
print('\nleave-one-asteroid-out, production procedure:')
print(pd.DataFrame({k: dict(AUC=f"{v['auc']:.3f}",
        recall_at_5pct_FA=f"{v['rec5']:.2f} [{v['rec5_ci'][0]:.2f}-{v['rec5_ci'][1]:.2f}]",
        FA_at_90pct_recall=f"{v['fa90']:.3f} [{v['fa90_ci'][0]:.3f}-{v['fa90_ci'][1]:.3f}]") for k, v in R.items()}).T.to_string())

# ---- 2. paired significance (same asteroid resamples for both methods) -----------------------------------
PT = {'FINAL vs image-only ranker': paired(pF, pI), 'FINAL vs rule (S/N without top frame)': paired(pF, pR)}
print('\npaired comparison (asteroid-level bootstrap; recall difference > 0 and false-alert difference < 0 favour FINAL):')
for k, v in PT.items():
    print(f"  {k}: recall diff {v['d_rec']:+.2f} [{v['d_rec_ci'][0]:+.2f}, {v['d_rec_ci'][1]:+.2f}], FINAL better in {v['rec_better']:.0%} "
          f"| false-alert diff @90% recall {v['d_fa']:+.3f} [{v['d_fa_ci'][0]:+.3f}, {v['d_fa_ci'][1]:+.3f}], FINAL better in {v['fa_better']:.0%}")

# ---- 3. operating points -----------------------------------------------------------------------------------
rows = []
for far in (0.05, 0.10):
    t = np.quantile(pF[y == 0], 1 - far); hit = pF >= t
    r = dict(false_alert_target=far, threshold=round(float(t), 4), recall=round(float(hit[y == 1].mean()), 2))
    for b in sorted(T.Vbin.unique()): r[f'rec V{b}'] = round(float(hit[(y == 1) & (T.Vbin == b).values].mean()), 2)
    for v in sorted(T.vgrp.unique()):
        m = (y == 1) & (T.vgrp == v).values
        if m.sum() >= 5: r[f'rec {v}'] = round(float(hit[m].mean()), 2)
    rows.append(r)
OP = pd.DataFrame(rows); print('\noperating points (thresholds from the out-of-fold control scores):'); print(OP.to_string(index=False))
chk = {}
for far in (0.05, 0.10):
    h, f = [], []
    for a in ug:
        te = g == a; t = np.quantile(pF[~te & (y == 0)], 1 - far)
        h += list(pF[te & (y == 1)] >= t); f += list(pF[te & (y == 0)] >= t)
    chk[far] = (np.mean(h), np.mean(f))
    print(f'threshold chosen on the OTHER asteroids, target {far:.0%}: realised recall {np.mean(h):.2f}, realised false alerts {np.mean(f):.3f}')
print('false-alert rate of controls by version at the 5% threshold:',
      pd.Series(pF >= np.quantile(pF[y == 0], 0.95)).groupby(T.vgrp.values).mean()[y == 0].round(3).to_dict()
      if False else {v: round(float((pF[(y == 0) & (T.vgrp == v).values] >= np.quantile(pF[y == 0], 0.95)).mean()), 3) for v in sorted(T.vgrp.unique())})

# ---- 4. package assembly --------------------------------------------------------------------------------------
shutil.copy(f'{ROOT}/models/ranker_B_v2.json', f'{PK}/ranker_B_v2.json')
sys.path.insert(0, PK); import aegis_ranker.features as AF; importlib.reload(AF)
cases = json.load(open(f'{PK}/golden/cases.json'))['cases']
c0 = cases[0]; fr = [dict(path=f'{PK}/golden/' + f['file'], ra=f['ra'], dec=f['dec'], mjd=f.get('mjd') or 0.0) for f in c0['frames']]
lat = min((lambda t0: (AF.rank(fr, model), time.perf_counter() - t0)[1])(time.perf_counter()) for _ in range(3))
hw = f"{platform.processor() or platform.machine()}, {os.cpu_count()} CPU cores, Python {platform.python_version()}"
print(f'\nlatency for one {len(fr)}-frame track: {lat:.2f} s on {hw}')

pos = T[T.y == 1]
n_frames = F2[F2.kind == 'mover'].local_file.nunique()
vb = pos.Vbin.value_counts().to_dict(); vg = pos.vgrp.value_counts().to_dict()
fm, im_, rl = R['FINAL ranker'], R['image-only ranker'], R['rule: S/N without top frame']
pf, pr_ = PT['FINAL vs image-only ranker'], PT['FINAL vs rule (S/N without top frame)']
def f2(v): return f'{v:.2f}'
card = f"""# AegisEarth candidate ranker ({model['version']})

Optional hint for the AegisEarth viewer. The viewer works without it. The score is an UNCALIBRATED candidate score (rank-ordering only), not a probability.

## What it does
Given 4-6 native SPHEREx L2 cutouts from ONE visit (hours apart) and one ICRS position per frame for a candidate moving object, it returns:
- score in [0,1] (candidate_score; higher = more consistent with a real moving source), or null when it cannot score;
- uncertainty = standard deviation across 50 bootstrap models (spread of the score, not a confidence interval);
- explanation_codes: CONSISTENT_ACROSS_FRAMES, SINGLE_FRAME_DOMINATES, LOW_TRACK_SNR, PIPELINE_OUTLIER_FLAG_AT_POSITION;
- warnings: FRAME_FAILED_QC, POSITION_NEAR_EDGE_OR_OUTSIDE, APERTURE_UNMEASURABLE, TOO_FEW_USABLE_FRAMES, EXTRA_FRAMES_IGNORED, APERTURE_ON_CATALOG_SOURCE.
It does not find candidates; candidate positions must come from elsewhere (for example known-object ephemerides).

## Input boundary
- Native IRSA cutout FITS per frame with IMAGE, VARIANCE, FLAGS (flag bit definitions are read from the FLAGS header; QR2 MP_* and QR3 MSKN/MSKM cards are both handled) and WCS. Not the prepared/resampled product.
- Cutout large enough that every candidate position lies at least 12 pixels from the edge. Training used 0.25 degree (about 146 px) cutouts; 10-15 arcmin is recommended. A 3 arcmin cutout is too small for tracks lasting hours.
- 4-6 frames of one visit, sorted by time (more frames are ignored, fewer give score = null).
- QC gate per frame: L2DQAFLG == Pass, FINAST == 0, bad-pixel fraction <= 0.25, SOURCE fraction <= 0.90, non-NaN coverage >= 0.90.

## Processing (identical in training and serving: aegis_ranker/features.py)
Aperture 2 px, background annulus 5-9 px, native pixels, IMAGE used directly (no ZODI subtraction). Rejected pixels: every flag defined in the header except MP_SOURCE and MP_FULLSAMPLE. In the aperture only, MP_OUTLIER pixels are tolerated (a moving object is flagged by a static-sky pipeline); the background excludes them. Frame features are combined per track.

## Model
Logistic regression ensemble (50 bootstrap models, class-balanced, C = 0.5) on {len(FINAL)} features: {', '.join(FINAL)}. Signed-log transform for S/N features. Inference needs only numpy and astropy.

## Training data
{int(y.sum())} real-mover tracks and {int((y == 0).sum())} displaced-control tracks from {len(ug)} known asteroids (positions from JPL Horizons; V {pos.V.min():.1f}-{pos.V.max():.1f}), {n_frames} frames, SPHEREx QR2/QR3 cutouts with processing versions l2b-v20 to v27. Positives by magnitude bin: {vb}. Positives by processing version group: {vg}. Controls are the same frames with the path displaced by 15-40 px (5 per visit). No synthetic data.

## Evaluation (leave-one-asteroid-out, production procedure, 95% intervals from resampling asteroids)
| method | AUC | recall at 5% false alerts | false alerts at 90% recall |
|---|---|---|---|
| rule: S/N without top frame | {rl['auc']:.3f} | {f2(rl['rec5'])} [{f2(rl['rec5_ci'][0])}-{f2(rl['rec5_ci'][1])}] | {rl['fa90']:.3f} [{rl['fa90_ci'][0]:.3f}-{rl['fa90_ci'][1]:.3f}] |
| image-only ranker | {im_['auc']:.3f} | {f2(im_['rec5'])} [{f2(im_['rec5_ci'][0])}-{f2(im_['rec5_ci'][1])}] | {im_['fa90']:.3f} [{im_['fa90_ci'][0]:.3f}-{im_['fa90_ci'][1]:.3f}] |
| FINAL ranker | {fm['auc']:.3f} | {f2(fm['rec5'])} [{f2(fm['rec5_ci'][0])}-{f2(fm['rec5_ci'][1])}] | {fm['fa90']:.3f} [{fm['fa90_ci'][0]:.3f}-{fm['fa90_ci'][1]:.3f}] |

Paired comparison, FINAL minus image-only: recall at 5% false alerts {pf['d_rec']:+.2f} [{pf['d_rec_ci'][0]:+.2f}, {pf['d_rec_ci'][1]:+.2f}] (FINAL better in {pf['rec_better']:.0%} of resamples); false alerts at 90% recall {pf['d_fa']:+.3f} [{pf['d_fa_ci'][0]:+.3f}, {pf['d_fa_ci'][1]:+.3f}] (better in {pf['fa_better']:.0%}).
FINAL minus the simple rule: recall {pr_['d_rec']:+.2f} [{pr_['d_rec_ci'][0]:+.2f}, {pr_['d_rec_ci'][1]:+.2f}]; false alerts {pr_['d_fa']:+.3f} [{pr_['d_fa_ci'][0]:+.3f}, {pr_['d_fa_ci'][1]:+.3f}].
Operating points (thresholds from out-of-fold control scores):

{OP.to_markdown(index=False)}

Threshold chosen on the other asteroids (held-out check): 5% target -> recall {chk[0.05][0]:.2f}, false alerts {chk[0.05][1]:.3f}; 10% target -> recall {chk[0.10][0]:.2f}, false alerts {chk[0.10][1]:.3f}.
Only differences whose paired interval excludes zero should be described as improvements.

## Limitations
- Trained and tested on {len(ug)} known bright asteroids (V about 14.5-18.5) in 4-6-frame visits. Fainter objects, comets, other motion regimes and other fields are untested. The faintest bin (V 17-18.5) has {vb.get('(17.0, 18.5]', 0)} positives; treat its numbers as qualitative.
- Part of the performance comes from the SPHEREx pipeline's own OUTLIER flag at the candidate position. The ranker combines that flag with frame-to-frame S/N consistency; it is not an independent detector.
- Negatives are displaced paths through the same real frames, not output from a candidate generator. False alerts from real candidate generation may differ.
- Processing versions are mixed (l2b-v20 to v27). Leave-one-version-out validation was run, but some version groups are small.
- Not suitable for the preset targets (Barnard's Star, Epsilon Indi B, WISE 0855, RW Aurigae, R Draconis): their motion between frames is far below one pixel or they are variable stars.
- Thresholds were chosen on the same out-of-fold scores they are reported on; the held-out check above reduces but does not remove that optimism.
- Scores are not calibrated probabilities. Frames with source-flagged apertures are reported through APERTURE_ON_CATALOG_SOURCE and are not used by the model.

## Resources
One {len(fr)}-frame track: {lat:.2f} s on {hw} (dominated by reading the FITS files). Memory is modest (a few cutouts at a time). Environment: numpy {np.__version__}, astropy {astropy.__version__}; scikit-learn {sklearn.__version__} was used for training only.

## Golden cases
golden/cases.json with FITS files and expected outputs (absolute tolerance 1e-6). Run: python golden/check_golden.py ranker_B_v2.json
"""
open(f'{PK}/MODEL_CARD.md', 'w').write(card)

open(f'{PK}/requirements.txt', 'w').write(f'numpy=={np.__version__}\nastropy=={astropy.__version__}\n# python {platform.python_version()}; inference only\n')
open(f'{PK}/README.md', 'w').write(f"""# aegis_ranker package ({model['version']})
Usage:
    import json, aegis_ranker.features as AF
    model = AF.load_model('ranker_B_v2.json')
    res = AF.rank([dict(path='frame1.fits', ra=..., dec=..., mjd=...), ...], model)   # 4-6 frames, one visit
Output keys: score, uncertainty, score_kind, model_version_id, explanation_codes, warnings, evidence, n_frames_used.
score is null when the track cannot be scored; then only warnings apply (do not build a RankingResult).
Mapping to RankingResult: score, score_kind='candidate_score', model_version_id, explanation_codes. uncertainty and warnings need optional fields.
See MODEL_CARD.md. Parity check: python golden/check_golden.py ranker_B_v2.json
""")

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for blk in iter(lambda: f.read(1 << 20), b''): h.update(blk)
    return h.hexdigest()
files = {}
for root_, _, names in os.walk(PK):
    for n in sorted(names):
        p = os.path.join(root_, n)
        if n != 'MANIFEST.json' and '__pycache__' not in p: files[os.path.relpath(p, PK)] = dict(sha256=sha(p), bytes=os.path.getsize(p))
json.dump(dict(model_version=model['version'], created_utc=datetime.datetime.utcnow().isoformat() + 'Z', hardware=hw,
               numpy=np.__version__, astropy=astropy.__version__, sklearn_training_only=sklearn.__version__,
               score_kind='candidate_score', calibrated=False, files=files), open(f'{PK}/MANIFEST.json', 'w'), indent=1)

# ---- 5. golden self-check the way the backend will run it ---------------------------------------------------------
r = subprocess.run([sys.executable, f'{PK}/golden/check_golden.py', f'{PK}/ranker_B_v2.json'], capture_output=True, text=True)
print('\ngolden check (separate process):'); print(r.stdout.strip()); print(r.stderr.strip()[-400:]); print('exit code', r.returncode)
shutil.make_archive(f'{ROOT}/aegis_ranker_package', 'zip', PK)
print(f"\npackage zip: {ROOT}/aegis_ranker_package.zip ({os.path.getsize(f'{ROOT}/aegis_ranker_package.zip') / 1e6:.0f} MB)")
print('model sha256:', files['ranker_B_v2.json']['sha256'][:16], '| features.py sha256:', files['aegis_ranker/features.py']['sha256'][:16])

out-of-fold scores done: 534 tracks, 89 positives, 15 asteroids

leave-one-asteroid-out, production procedure:
                               AUC recall_at_5pct_FA   FA_at_90pct_recall
rule: S/N without top frame  0.973  0.84 [0.61-1.00]  0.056 [0.010-0.105]
image-only ranker            0.970  0.89 [0.69-1.00]  0.056 [0.002-0.096]
FINAL ranker                 0.989  0.98 [0.89-1.00]  0.018 [0.000-0.053]

paired comparison (asteroid-level bootstrap; recall difference > 0 and false-alert difference < 0 favour FINAL):
  FINAL vs image-only ranker: recall diff +0.08 [+0.00, +0.24], FINAL better in 94% | false-alert diff @90% recall -0.028 [-0.062, -0.002], FINAL better in 98%
  FINAL vs rule (S/N without top frame): recall diff +0.11 [-0.01, +0.34], FINAL better in 87% | false-alert diff @90% recall -0.036 [-0.069, -0.007], FINAL better in 100%

operating points (thresholds from the out-of-fold control scores):
 false_alert_target  threshold  recall  rec V(14.5, 16.0]  rec V(16.0, 17.0]  r

In [ ]:
# B8: independent check on bright set A through the production module + finalize the package
import os, sys, json, hashlib, shutil, datetime, importlib, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK = f'{ROOT}/package'
sys.path.insert(0, PK); import aegis_ranker.features as AF; importlib.reload(AF)
model = AF.load_model(f'{PK}/ranker_B_v2.json')
THR = 0.3093                                   # the 5% false-alert threshold printed by B7

QA = pd.read_csv(f'{ROOT}/data/manifests/astA_qc.csv')
QA = QA[QA.status.isin(['ok', 'cached']) & (QA.inside == True)]
rg = np.random.default_rng(11); rows = []
visits = [(v, g) for v, g in QA.groupby('visit') if g.obs_id.nunique() >= 4]
for v, g in tqdm(visits):
    g = g.sort_values('mjd')
    fr = [dict(path=r.local_file, ra=float(r.ra), dec=float(r.dec), mjd=float(r.mjd)) for r in g.itertuples()]
    o = AF.rank(fr, model)
    rows.append(dict(visit=v, ast=int(g.ast.iloc[0]), V=float(g.V.median()), kind='mover', score=o['score'],
                     warn=';'.join(w for w in o['warnings'] if w != 'EXTRA_FRAMES_IGNORED')))
    for c in range(3):
        a, r_ = rg.uniform(0, 2 * np.pi), rg.uniform(15, 40)
        o = AF.rank(fr, model, shift=(r_ * np.cos(a), r_ * np.sin(a)))
        rows.append(dict(visit=v, ast=int(g.ast.iloc[0]), V=float(g.V.median()), kind='ctrl', score=o['score'],
                         warn=';'.join(w for w in o['warnings'] if w != 'EXTRA_FRAMES_IGNORED')))
R = pd.DataFrame(rows)
mv, ct = R[R.kind == 'mover'], R[R.kind == 'ctrl']
ms, cs = mv.dropna(subset=['score']), ct.dropna(subset=['score'])
print(f'set A tracks: {len(mv)} mover tracks ({len(ms)} scored), {len(ct)} control tracks ({len(cs)} scored)')
print('why not scored (movers):', mv[mv.score.isna()].warn.value_counts().to_dict())
rec = float((ms.score >= THR).mean()); fa = float((cs.score >= THR).mean())
print(f'recall at the 5% threshold: {rec:.2f} | control false alerts: {fa:.3f}')
print('median score: movers', round(float(ms.score.median()), 3), '| controls', round(float(cs.score.median()), 4))
print('movers scored below the threshold:'); print(ms[ms.score < THR][['visit', 'ast', 'V', 'score']].round(3).to_string(index=False) or 'none')
print('controls scored above the threshold:'); print(cs[cs.score >= THR][['visit', 'ast', 'V', 'score']].round(3).to_string(index=False) or 'none')

# ---- append to the model card, refresh manifest and zip ------------------------------------------------
add = f"""

## Additional check on an independent bright set (not used for training or model selection)
Known asteroids with V about 9-13 (brighter than training), 4-6-frame visits, run through the production module with the 5% threshold {THR}: {len(ms)} scored real-mover tracks, recall {rec:.2f}; {len(cs)} scored displaced-control tracks, false-alert rate {fa:.3f}. Small sample, magnitude range outside training; read as a sanity check, not as a performance estimate.
False-alert rate of the training controls at the 5% threshold, by processing version: l2b-v20 0.048, l2b-v24 0.083, l2b-v27 0.080, other versions 0.015. Newer versions produce more false alerts at the same threshold.

## Known limitation: the module does not test motion
It scores whatever positions it is given. A source fixed on the sky and tracked at constant RA/Dec is consistent across frames and can score high. Callers must supply moving candidates (for example ephemeris positions). Static sources are not candidates.
"""
card = open(f'{PK}/MODEL_CARD.md').read()
if 'Additional check on an independent bright set' not in card:
    open(f'{PK}/MODEL_CARD.md', 'w').write(card + add)

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for blk in iter(lambda: f.read(1 << 20), b''): h.update(blk)
    return h.hexdigest()
old = json.load(open(f'{PK}/MANIFEST.json')); files = {}
for root_, _, names in os.walk(PK):
    for n in sorted(names):
        p = os.path.join(root_, n)
        if n != 'MANIFEST.json' and '__pycache__' not in p:
            files[os.path.relpath(p, PK)] = dict(sha256=sha(p), bytes=os.path.getsize(p))
old.update(files=files, refreshed_utc=datetime.datetime.utcnow().isoformat() + 'Z')
json.dump(old, open(f'{PK}/MANIFEST.json', 'w'), indent=1)
shutil.make_archive(f'{ROOT}/aegis_ranker_package', 'zip', PK)
print('\npackage refreshed:', f"{ROOT}/aegis_ranker_package.zip", f"({os.path.getsize(f'{ROOT}/aegis_ranker_package.zip') / 1e6:.0f} MB)")
print('model sha256:', files['ranker_B_v2.json']['sha256'][:16], '| features.py sha256:', files['aegis_ranker/features.py']['sha256'][:16])

  0%|          | 0/36 [00:00<?, ?it/s]

set A tracks: 36 mover tracks (22 scored), 108 control tracks (72 scored)
why not scored (movers): {'FRAME_FAILED_QC;TOO_FEW_USABLE_FRAMES': 12, 'APERTURE_UNMEASURABLE;TOO_FEW_USABLE_FRAMES': 2}
recall at the 5% threshold: 1.00 | control false alerts: 0.153
median score: movers 1.0 | controls 0.0091
movers scored below the threshold:
Empty DataFrame
Columns: [visit, ast, V, score]
Index: []
controls scored above the threshold:
 visit  ast      V  score
   151   19 12.568  0.339
   165   20 10.709  0.921
   201   40 11.102  0.437
   201   40 11.102  0.484
   201   40 11.102  0.762
   211   40 10.950  0.979
   213   40 10.973  0.931
   233   43 12.726  0.624
   233   43 12.726  0.994
   233   43 12.726  0.996
   241   43 12.772  0.595

package refreshed: /content/drive/MyDrive/spherex_project/aegis_ranker_package.zip (79 MB)
model sha256: 01cb3dbe15e1e041 | features.py sha256: c511c4874dcf3342


In [ ]:
# B9: why do displaced controls on bright set A raise false alerts? (diagnostic only)
import os, sys, importlib, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK = f'{ROOT}/package'
sys.path.insert(0, PK); import aegis_ranker.features as AF; importlib.reload(AF)
model = AF.load_model(f'{PK}/ranker_B_v2.json'); THR = 0.3093

QA = pd.read_csv(f'{ROOT}/data/manifests/astA_qc.csv')
QA = QA[QA.status.isin(['ok', 'cached']) & (QA.inside == True)]
rg = np.random.default_rng(11); rows = []                      # same seed and loop order as B8 -> same controls
visits = [(v, g) for v, g in QA.groupby('visit') if g.obs_id.nunique() >= 4]
for v, g in tqdm(visits):
    g = g.sort_values('mjd')
    fr = [dict(path=r.local_file, ra=float(r.ra), dec=float(r.dec), mjd=float(r.mjd)) for r in g.itertuples()]
    base = dict(visit=v, ast=int(g.ast.iloc[0]), V=float(g.V.median()))
    o = AF.rank(fr, model)
    rows.append(base | dict(kind='mover', radius=0.0, score=o['score'], **(o['evidence'] or {})))
    for c in range(3):
        a, r_ = rg.uniform(0, 2 * np.pi), rg.uniform(15, 40)
        o = AF.rank(fr, model, shift=(r_ * np.cos(a), r_ * np.sin(a)))
        rows.append(base | dict(kind='ctrl', radius=r_, score=o['score'], **(o['evidence'] or {})))
D = pd.DataFrame(rows).dropna(subset=['score']); D['alert'] = D.score >= THR
ct, mv = D[D.kind == 'ctrl'], D[D.kind == 'mover']
cols = ['frac_gt3', 'snr_wo_top', 'track_snr', 'outlier_ap', 'outlier_frames', 'src_frac_ap', 'm_bad_frac', 'm_noise_mad']
TB = pd.read_csv(f'{ROOT}/data/manifests/astB_track_table_v2.csv')

print('control tracks that raised an alert:', int(ct.alert.sum()), 'of', len(ct))
print('\nmedian features:')
print(pd.DataFrame({'alert controls': ct[ct.alert][cols].median(), 'quiet controls': ct[~ct.alert][cols].median(),
                    'real movers (set A)': mv[cols].median(), 'training controls (set B)': TB[TB.y == 0][cols].median(),
                    'training movers (set B)': TB[TB.y == 1][cols].median()}).round(3).to_string())
print('\nfalse-alert rate by asteroid (controls):')
print(ct.groupby('ast').alert.agg(rate='mean', alerts='sum', n='size').round(2).T.to_string())
print('\nby V bin:'); print(ct.groupby(pd.cut(ct.V, [8, 10.5, 11.5, 13]), observed=True).alert.agg(rate='mean', alerts='sum', n='size').round(2).to_string())
print('\nby displacement from the asteroid (px):')
print(ct.groupby(pd.cut(ct.radius, [15, 22, 30, 40]), observed=True).alert.agg(rate='mean', alerts='sum', n='size').round(2).to_string())
print('\nthe alert controls in detail:')
print(ct[ct.alert][['visit', 'ast', 'V', 'radius', 'score'] + cols].round(2).to_string(index=False))

  0%|          | 0/36 [00:00<?, ?it/s]

control tracks that raised an alert: 11 of 72

median features:
                alert controls  quiet controls  real movers (set A)  training controls (set B)  training movers (set B)
frac_gt3                 1.000           0.000                1.000                      0.000                    1.000
snr_wo_top              20.722           0.303              405.537                     -0.330                   21.258
track_snr               31.611           0.721              488.551                      0.266                   25.226
outlier_ap               0.000           0.015                0.820                      0.014                    0.222
outlier_frames           0.000           0.167                1.000                      0.167                    0.833
src_frac_ap              1.000           0.500                0.550                      0.500                    0.667
m_bad_frac               0.031           0.033                0.032                      0.033  

In [ ]:
# B10: test the motion / frame-selection explanation for the set-A false alerts
import os, sys, importlib, warnings
import numpy as np, pandas as pd
from astropy.coordinates import SkyCoord
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK = f'{ROOT}/package'
sys.path.insert(0, PK); import aegis_ranker.features as AF; importlib.reload(AF)
model = AF.load_model(f'{PK}/ranker_B_v2.json'); THR = 0.3093
PIX, GATE_PX = 6.15, 4.0                        # gate fixed in advance: 2 x aperture radius (2 px)

def disp_px(fr):                                 # sky separation first -> last frame, in pixels
    a = SkyCoord(fr[0]['ra'], fr[0]['dec'], unit='deg'); b = SkyCoord(fr[-1]['ra'], fr[-1]['dec'], unit='deg')
    return a.separation(b).arcsec / PIX

def spread(fr, k=6):
    if len(fr) <= k: return fr
    return [fr[i] for i in np.unique(np.linspace(0, len(fr) - 1, k).round().astype(int))]

QA = pd.read_csv(f'{ROOT}/data/manifests/astA_qc.csv')
QA = QA[QA.status.isin(['ok', 'cached']) & (QA.inside == True)]
rg = np.random.default_rng(11); rows = []         # same seed and order as B8/B9 -> same displaced controls
visits = [(v, g) for v, g in QA.groupby('visit') if g.obs_id.nunique() >= 4]
for v, g in tqdm(visits):
    g = g.sort_values('mjd')
    allfr = [dict(path=r.local_file, ra=float(r.ra), dec=float(r.dec), mjd=float(r.mjd)) for r in g.itertuples()]
    sh = []
    for _ in range(3):
        a, r_ = rg.uniform(0, 2 * np.pi), rg.uniform(15, 40); sh.append((r_ * np.cos(a), r_ * np.sin(a)))
    for mode, fr in (('first6', allfr[:6]), ('spread6', spread(allfr))):
        base = dict(visit=v, ast=int(g.ast.iloc[0]), mode=mode, nfr=len(fr),
                    span_h=(fr[-1]['mjd'] - fr[0]['mjd']) * 24, disp_px=disp_px(fr))
        rows.append(base | dict(kind='mover', score=AF.rank(fr, model)['score']))
        for s in sh:
            rows.append(base | dict(kind='ctrl', score=AF.rank(fr, model, shift=s)['score']))
D = pd.DataFrame(rows)

def summarize(d):
    m, c = d[d.kind == 'mover'].dropna(subset=['score']), d[d.kind == 'ctrl'].dropna(subset=['score'])
    return {'movers scored': len(m), 'recall': round(float((m.score >= THR).mean()), 2) if len(m) else np.nan,
            'controls scored': len(c), 'false alerts': round(float((c.score >= THR).mean()), 3) if len(c) else np.nan}
res = {}
for mode in ('first6', 'spread6'):
    d = D[D['mode'] == mode]
    res[f'{mode}, no gate'] = summarize(d)
    res[f'{mode}, gate >= {GATE_PX:.0f} px'] = summarize(d[d.disp_px >= GATE_PX])
print(pd.DataFrame(res).T.to_string())

vt = D[D.kind == 'mover'].drop_duplicates(['visit', 'mode'])
print('\nmotion across the selected frames (median per visit):')
print(vt.groupby('mode')[['span_h', 'disp_px']].median().round(2).to_string())
print('visits below the gate:', {m: f"{int((g_.disp_px < GATE_PX).sum())} of {len(g_)}" for m, g_ in vt.groupby('mode')})

c1 = D[(D['mode'] == 'first6') & (D.kind == 'ctrl')].dropna(subset=['score']).copy(); c1['alert'] = c1.score >= THR
print('\nfirst6 controls: false-alert rate by track displacement (px):')
print(c1.groupby(pd.cut(c1.disp_px, [0, 4, 10, 25, 1000]), observed=True).alert.agg(rate='mean', alerts='sum', n='size').round(2).to_string())

# would the gate remove real training positives? (set B, the 6 frames used in training)
TB = pd.read_csv(f'{ROOT}/data/manifests/astB_track_table_v2.csv')
FB = pd.read_csv(f'{ROOT}/data/manifests/astB_frames_v2.csv')
QB = pd.read_csv(f'{ROOT}/data/manifests/astB_qc.csv').drop_duplicates('local_file').set_index('local_file')
dp = {}
for v, g in FB[FB.kind == 'mover'].groupby('visit'):
    q = QB.loc[g.local_file.unique()].sort_values('mjd')
    dp[v] = disp_px([dict(ra=float(q.ra.iloc[0]), dec=float(q.dec.iloc[0])), dict(ra=float(q.ra.iloc[-1]), dec=float(q.dec.iloc[-1]))])
pos = TB[TB.y == 1].assign(disp_px=lambda d: d.visit.map(dp))
print(f"\ntraining positives (set B): median displacement {pos.disp_px.median():.1f} px, median span {pos.span_h.median():.1f} h; "
      f"below the gate: {int((pos.disp_px < GATE_PX).sum())} of {len(pos)}")

  0%|          | 0/36 [00:00<?, ?it/s]

                       movers scored  recall  controls scored  false alerts
first6, no gate                 22.0     1.0             72.0         0.153
first6, gate >= 4 px             3.0     1.0             12.0         0.000
spread6, no gate                22.0     1.0             72.0         0.111
spread6, gate >= 4 px           15.0     1.0             51.0         0.118

motion across the selected frames (median per visit):
         span_h  disp_px
mode                    
first6     0.07     0.48
spread6    8.20    45.60
visits below the gate: {'first6': '27 of 36', 'spread6': '11 of 36'}

first6 controls: false-alert rate by track displacement (px):
            rate  alerts   n
disp_px                     
(0, 4]      0.18      11  60
(10, 25]    0.00       0   9
(25, 1000]  0.00       0   3

training positives (set B): median displacement 19.8 px, median span 3.3 h; below the gate: 40 of 89


In [ ]:
# B11: rebuild track tables with the production module (spread frames), pool sets A+B, compare V0 vs V1
import os, sys, importlib, warnings
import numpy as np, pandas as pd
from astropy.coordinates import SkyCoord
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
sys.path.insert(0, f'{ROOT}/package'); import aegis_ranker.features as AF; importlib.reload(AF)
PIX = 6.15

def spread(fr, k=6):
    if len(fr) <= k: return fr
    return [fr[i] for i in np.unique(np.linspace(0, len(fr) - 1, k).round().astype(int))]

def disp_px(a, b):
    return SkyCoord(a['ra'], a['dec'], unit='deg').separation(SkyCoord(b['ra'], b['dec'], unit='deg')).arcsec / PIX

def build(name, qc_csv, seed):
    Q = pd.read_csv(qc_csv)
    Q = Q[Q.status.isin(['ok', 'cached']) & (Q.inside == True)].copy()
    Q['ver'] = Q.path.str.extract(r'(l2b-v\d+)')[0]
    rg = np.random.default_rng(seed); rows = []
    for v, g in tqdm(list(Q.groupby('visit')), desc=name):
        g = g.sort_values('mjd'); use = []
        for r in g.itertuples():                                   # production QC + mover measurement
            a = AF.analyze_frame(r.local_file, float(r.ra), float(r.dec))
            if a['qc_pass'] and a['inside'] and a['meas'] is not None:
                use.append(dict(path=r.local_file, ra=float(r.ra), dec=float(r.dec), mjd=float(r.mjd), a=a))
        if len(use) < 4: continue
        sel = spread(use)
        shifts = [(0.0, 0.0)] + [(rr * np.cos(aa), rr * np.sin(aa)) for rr, aa in
                                 zip(rg.uniform(15, 40, 5), rg.uniform(0, 2 * np.pi, 5))]
        base = dict(set=name, visit=int(v), ast=int(g.ast.iloc[0]), V=float(g.V.median()), ver=g.ver.iloc[0],
                    span_h=(sel[-1]['mjd'] - sel[0]['mjd']) * 24, disp_px=disp_px(sel[0], sel[-1]))
        for k, sh in enumerate(shifts):
            fr = [s['a'] if k == 0 else AF.analyze_frame(s['path'], s['ra'], s['dec'], sh) for s in sel]
            fr = [f for f in fr if f['qc_pass'] and f['inside'] and f['meas'] is not None]
            if len(fr) < 4: continue
            rows.append(base | dict(kind='mover' if k == 0 else f'ctrl{k - 1}', y=int(k == 0), **AF.track_features(fr)))
    return pd.DataFrame(rows)

T = pd.concat([build('A', f'{ROOT}/data/manifests/astA_qc.csv', 101),
               build('B', f'{ROOT}/data/manifests/astB_qc.csv', 102)]).reset_index(drop=True)
T['Vbin'] = pd.cut(T.V, [8, 11.5, 14.5, 16, 17, 18.5]).astype(str)
T['snr_noflag'] = T.snr_wo_top_l * (1 - T.outlier_frames)         # S/N counts only where the pipeline flagged the pixels
T.to_csv(f'{ROOT}/data/manifests/astAB_track_table.csv', index=False)
print(T.groupby(['set', 'y']).size().unstack().rename(columns={0: 'controls', 1: 'movers'}).to_string())
print('asteroids:', T.groupby('set').ast.nunique().to_dict(), '| movers by V bin:', T[T.y == 1].Vbin.value_counts().to_dict())

c = T[T.y == 0]; star = c[c.snr_wo_top >= 5]
print('\ncontrols with S/N >= 5 even without the top frame (star-like), by set:',
      c.groupby('set').apply(lambda d: round(float((d.snr_wo_top >= 5).mean()), 3)).to_dict())
print('  of those, share with no OUTLIER flag in any frame:',
      star.groupby('set').outlier_frames.apply(lambda s: round(float((s == 0).mean()), 2)).to_dict())

IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
FLGN = ['m_bad_frac', 'm_noise_mad', 'm_persist_frac', 'm_cr_frac', 'm_outlier_frac', 'm_badpix_frac']
V0 = IMG + FLGN + ['outlier_ap', 'outlier_frames']; V1 = V0 + ['snr_noflag']

def fit(X, yy, C=0.5):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med))
    return med, sc, LogisticRegression(C=C, class_weight='balanced', max_iter=3000).fit(sc.transform(X.fillna(med)), yy)

y, g, isA = T.y.values, T.ast.values, (T.set == 'A').values
ug = np.unique(g); idx = {a: np.where(g == a)[0] for a in ug}

def oof(cols):
    p = np.zeros(len(T))
    for a in ug:
        te = g == a; med, sc, lr = fit(T.loc[~te, cols], y[~te])
        p[te] = lr.predict_proba(sc.transform(T.loc[te, cols].fillna(med)))[:, 1]
    return p

def stats(p, i=None):
    i = np.arange(len(p)) if i is None else i
    pp, yy, aa = p[i], y[i], isA[i]
    t5, t90 = np.quantile(pp[yy == 0], 0.95), np.quantile(pp[yy == 1], 0.10)
    f = lambda m: float(m.mean()) if m.size else np.nan
    return dict(auc=roc_auc_score(yy, pp), rec5=f(pp[yy == 1] >= t5), fa90=f(pp[yy == 0] >= t90),
                faA=f(pp[(yy == 0) & aa] >= t5), recA=f(pp[(yy == 1) & aa] >= t5), recB=f(pp[(yy == 1) & ~aa] >= t5))

P = {'V0': oof(V0), 'V1': oof(V1)}
rg = np.random.default_rng(0); res = {k: [] for k in P}
for _ in range(500):
    i = np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])
    if y[i].sum() < 5 or (y[i] == 0).sum() < 20 or ((y[i] == 0) & isA[i]).sum() < 10: continue
    for k, p in P.items(): res[k].append(stats(p, i))

names = dict(auc='AUC', rec5='recall @5% FA', fa90='FA @90% recall', faA='FA on set-A controls @5%',
             recA='recall set A @5%', recB='recall set B @5%')
rows = []
for k, p in P.items():
    s, b = stats(p), pd.DataFrame(res[k]); r = {'variant': k}
    for m, lab in names.items():
        lo, hi = np.nanpercentile(b[m], [2.5, 97.5]); r[lab] = f'{s[m]:.3f} [{lo:.3f}-{hi:.3f}]'
    rows.append(r)
print('\nleave-one-asteroid-out, pooled sets A+B (95% intervals from resampling asteroids):')
print(pd.DataFrame(rows).T.to_string(header=False))
d = pd.DataFrame(res['V1']) - pd.DataFrame(res['V0'])
for m in ('fa90', 'faA', 'rec5'):
    lo, hi = np.nanpercentile(d[m], [2.5, 97.5])
    print(f"V1 minus V0, {names[m]}: {d[m].mean():+.3f} [{lo:+.3f}, {hi:+.3f}], "
          f"V1 {'lower' if m != 'rec5' else 'higher'} in {((d[m] < 0) if m != 'rec5' else (d[m] > 0)).mean():.0%} of resamples")

t5 = {k: np.quantile(p[y == 0], 0.95) for k, p in P.items()}; lowm = (T.disp_px < 4).values
for k, p in P.items():
    r = lambda m: round(float((p[m] >= t5[k]).mean()), 3)
    print(f'{k}: motion < 4 px -> recall {r((y == 1) & lowm)} (n={int(((y == 1) & lowm).sum())}), FA {r((y == 0) & lowm)} | '
          f'motion >= 4 px -> recall {r((y == 1) & ~lowm)}, FA {r((y == 0) & ~lowm)}')
print('recall @5% FA by V bin:', {k: {b: round(float((p[(y == 1) & (T.Vbin == b).values] >= t5[k]).mean()), 2)
      for b in sorted(T.Vbin.unique()) if ((y == 1) & (T.Vbin == b).values).any()} for k, p in P.items()})

s0, s1 = stats(P['V0']), stats(P['V1'])
adopt = (s0['fa90'] - s1['fa90'] >= 0.02) and (s1['faA'] < 0.10) and (s0['recB'] - s1['recB'] <= 0.05)
print('\npre-set rule (V1 lowers FA@90% by >= 0.02, set-A false alerts < 0.10, set-B recall drop <= 0.05) -> adopt V1:', adopt)

A:   0%|          | 0/36 [00:00<?, ?it/s]

B:   0%|          | 0/100 [00:00<?, ?it/s]

y    controls  movers
set                  
A         120      24
B         450      90
asteroids: {'A': 10, 'B': 15} | movers by V bin: {'(14.5, 16.0]': 42, '(16.0, 17.0]': 37, '(8.0, 11.5]': 18, '(17.0, 18.5]': 11, '(11.5, 14.5]': 6}

controls with S/N >= 5 even without the top frame (star-like), by set: {'A': 0.033, 'B': 0.067}
  of those, share with no OUTLIER flag in any frame: {'A': 0.5, 'B': 0.83}

leave-one-asteroid-out, pooled sets A+B (95% intervals from resampling asteroids):
variant                                    V0                   V1
AUC                       0.990 [0.978-0.997]  0.990 [0.978-0.997]
recall @5% FA             0.965 [0.907-1.000]  0.965 [0.904-1.000]
FA @90% recall            0.025 [0.005-0.049]  0.025 [0.005-0.050]
FA on set-A controls @5%  0.058 [0.018-0.111]  0.058 [0.018-0.114]
recall set A @5%          1.000 [1.000-1.000]  1.000 [1.000-1.000]
recall set B @5%          0.956 [0.883-1.000]  0.956 [0.879-1.000]
V1 minus V0, FA @90% recall: +0.000 [+0

In [ ]:
# B12: final model (V0 features) on pooled A+B: operating points, paired tests, static-source warning, save v3
import os, json, datetime, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
T = pd.read_csv(f'{ROOT}/data/manifests/astAB_track_table.csv')
T['vgrp'] = T.ver.where(T.ver.isin(['l2b-v20', 'l2b-v24', 'l2b-v27']), 'other')
IMG = ['track_snr_l', 'snr_wo_top_l', 'med_snr_l', 'frac_gt3', 'top_share', 'n']
FLGN = ['m_bad_frac', 'm_noise_mad', 'm_persist_frac', 'm_cr_frac', 'm_outlier_frac', 'm_badpix_frac']
V0 = IMG + FLGN + ['outlier_ap', 'outlier_frames']
y, g = T.y.values, T.ast.values
ug = np.unique(g); idx = {a: np.where(g == a)[0] for a in ug}
print(len(T), 'tracks |', int(y.sum()), 'movers |', len(ug), 'asteroids')

def fit_ens(X, yy, seed, nb=50):
    med = X.median(); sc = StandardScaler().fit(X.fillna(med)); Z = sc.transform(X.fillna(med))
    rb = np.random.default_rng(seed); co, ic = [], []
    while len(co) < nb:
        i = rb.integers(0, len(X), len(X))
        if yy[i].sum() < 3: continue
        lr = LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000).fit(Z[i], yy[i])
        co.append(lr.coef_[0]); ic.append(lr.intercept_[0])
    return med, sc, np.array(co), np.array(ic)

def oof_ens(cols):
    p = np.zeros(len(T))
    for a in ug:
        te = g == a
        med, sc, co, ic = fit_ens(T.loc[~te, cols], y[~te], seed=int(a))
        Z = sc.transform(T.loc[te, cols].fillna(med))
        p[te] = (1 / (1 + np.exp(-(Z @ co.T + ic)))).mean(1)
    return p

pF, pI, pR = oof_ens(V0), oof_ens(IMG), T.snr_wo_top.values

def met2(p, yy):
    return (p[yy == 1] >= np.quantile(p[yy == 0], 0.95)).mean(), (p[yy == 0] >= np.quantile(p[yy == 1], 0.10)).mean()
def resample(rg): return np.concatenate([idx[a] for a in rg.choice(ug, len(ug))])
def usable(i): return y[i].sum() >= 5 and (y[i] == 0).sum() >= 20

def boot_ci(p, B=1000, seed=3):
    rg = np.random.default_rng(seed); bs = []
    for _ in range(B):
        i = resample(rg)
        if usable(i): bs.append(met2(p[i], y[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); r5, f90 = met2(p, y)
    return f'AUC {roc_auc_score(y, p):.3f} | recall@5%FA {r5:.2f} [{lo[0]:.2f}-{hi[0]:.2f}] | FA@90%recall {f90:.3f} [{lo[1]:.3f}-{hi[1]:.3f}]'

def paired(pa, pb, B=2000, seed=7):
    rg = np.random.default_rng(seed); d = []
    for _ in range(B):
        i = resample(rg)
        if usable(i):
            a_, b_ = met2(pa[i], y[i]), met2(pb[i], y[i]); d.append((a_[0] - b_[0], a_[1] - b_[1]))
    d = np.array(d)
    return (f'recall diff {d[:, 0].mean():+.2f} [{np.percentile(d[:, 0], 2.5):+.2f}, {np.percentile(d[:, 0], 97.5):+.2f}] (FINAL better {(d[:, 0] > 0).mean():.0%}) | '
            f'FA@90%recall diff {d[:, 1].mean():+.3f} [{np.percentile(d[:, 1], 2.5):+.3f}, {np.percentile(d[:, 1], 97.5):+.3f}] (FINAL better {(d[:, 1] < 0).mean():.0%})')

print('\nleave-one-asteroid-out, production ensemble, pooled A+B:')
for k, p in (('rule: S/N without top frame', pR), ('image-only ranker', pI), ('FINAL (V0)', pF)): print(f'  {k:28s}', boot_ci(p))
print('paired, FINAL vs image-only:', paired(pF, pI))
print('paired, FINAL vs rule      :', paired(pF, pR))

# ---- operating points --------------------------------------------------------------------------------
t5, t10 = float(np.quantile(pF[y == 0], 0.95)), float(np.quantile(pF[y == 0], 0.90))
rows = []
for far, t in ((0.05, t5), (0.10, t10)):
    hit = pF >= t; r = dict(false_alert_target=far, threshold=round(t, 4), recall=round(float(hit[y == 1].mean()), 2))
    for col, key in (('Vbin', 'V'), ('vgrp', ''), ('set', 'set ')):
        for b in sorted(T[col].unique()):
            m = (y == 1) & (T[col] == b).values
            if m.sum() >= 5: r[f'rec {key}{b} (n={int(m.sum())})'] = round(float(hit[m].mean()), 2)
    rows.append(r)
print('\noperating points:'); print(pd.DataFrame(rows).T.to_string(header=False))
print('control false-alert rate at the 5% threshold, by version:', {v: round(float((pF[(y == 0) & (T.vgrp == v).values] >= t5).mean()), 3) for v in sorted(T.vgrp.unique())},
      '| by set:', {s: round(float((pF[(y == 0) & (T.set == s).values] >= t5).mean()), 3) for s in ('A', 'B')})
for far in (0.05, 0.10):
    h, f = [], []
    for a in ug:
        te = g == a; t = np.quantile(pF[~te & (y == 0)], 1 - far)
        h += list(pF[te & (y == 1)] >= t); f += list(pF[te & (y == 0)] >= t)
    print(f'threshold chosen on the other asteroids, target {far:.0%}: realised recall {np.mean(h):.2f}, false alerts {np.mean(f):.3f}')

# ---- static-source warning (rule fixed in advance) ----------------------------------------------------------
cond = ((T.src_frac_ap >= 0.99) & (T.outlier_frames == 0)).values
star = (y == 0) & (T.snr_wo_top.values >= 5); alert = (y == 0) & (pF >= t5)
sm, ss, sa = cond[y == 1].mean(), (cond[star].mean() if star.any() else np.nan), (cond[alert].mean() if alert.any() else np.nan)
print(f'\nLIKELY_STATIC_SOURCE condition (every frame on a catalogued source, no OUTLIER flag): real movers {sm:.2f} (n={int((y == 1).sum())}), '
      f'star-like controls {ss:.2f} (n={int(star.sum())}), controls above the 5% threshold {sa:.2f} (n={int(alert.sum())})')
adopt = (sm <= 0.10) and (ss >= 0.5) and (sa >= 0.5)
print('pre-set rule (movers <= 0.10, star-like >= 0.5, alerts >= 0.5) -> include LIKELY_STATIC_SOURCE warning:', adopt)

# ---- final model on all data and save ------------------------------------------------------------------------
med, sc, co, ic = fit_ens(T[V0], y, seed=1)
model = dict(version='ranker_AB_v3', features=V0, signed_log=['track_snr_l', 'snr_wo_top_l', 'med_snr_l'],
             fill={k: float(v) for k, v in med.items()}, mean=sc.mean_.tolist(), scale=sc.scale_.tolist(),
             coefs=co.tolist(), intercepts=ic.tolist(), threshold_5pct=t5, threshold_10pct=t10,
             frame_selection='4-6 frames spread evenly across the visit', training=dict(
                 tracks=int(len(T)), movers=int(y.sum()), asteroids=int(len(ug)),
                 movers_by_Vbin={k: int(v) for k, v in T[y == 1].Vbin.value_counts().items()},
                 built_utc=datetime.datetime.utcnow().isoformat() + 'Z'))
json.dump(model, open(f'{ROOT}/models/ranker_AB_v3.json', 'w'))
print(f'\nsaved models/ranker_AB_v3.json | thresholds: 5% -> {t5:.4f}, 10% -> {t10:.4f}')

684 tracks | 114 movers | 25 asteroids

leave-one-asteroid-out, production ensemble, pooled A+B:
  rule: S/N without top frame  AUC 0.980 | recall@5%FA 0.85 [0.70-1.00] | FA@90%recall 0.056 [0.018-0.098]
  image-only ranker            AUC 0.975 | recall@5%FA 0.87 [0.72-0.99] | FA@90%recall 0.058 [0.013-0.119]
  FINAL (V0)                   AUC 0.991 | recall@5%FA 0.96 [0.91-1.00] | FA@90%recall 0.016 [0.003-0.048]
paired, FINAL vs image-only: recall diff +0.08 [+0.00, +0.21] (FINAL better 96%) | FA@90%recall diff -0.037 [-0.083, -0.006] (FINAL better 100%)
paired, FINAL vs rule      : recall diff +0.09 [-0.02, +0.23] (FINAL better 81%) | FA@90%recall diff -0.035 [-0.060, -0.011] (FINAL better 100%)

operating points:
false_alert_target        0.0500  0.1000
threshold                 0.2951  0.1159
recall                    0.9600  0.9800
rec V(11.5, 14.5] (n=6)   1.0000  1.0000
rec V(14.5, 16.0] (n=42)  1.0000  1.0000
rec V(16.0, 17.0] (n=37)  0.9500  0.9700
rec V(17.0, 18.5] (n=11)  0

In [ ]:
# B13, cell 1: package v2 = features.py (v2) + candidates.py + model json with metadata
import os, sys, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK2 = f'{ROOT}/package_v2'; os.makedirs(f'{PK2}/aegis_ranker', exist_ok=True)

FEATURES_SRC = r'''
# AegisEarth candidate ranker v2: evidence builder + scorer. Needs only numpy and astropy.
import json, re, warnings
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

R_AP, R_IN, R_OUT, W = 2.0, 5.0, 9.0, 11            # aperture, background annulus (pixels), window half-size
MIN_FRAMES, MAX_FRAMES = 4, 6
PIX_ARCSEC, LOW_MOTION_PX = 6.15, 4.0
INFO = {'MP_SOURCE', 'MP_FULLSAMPLE'}                 # informational flags: never rejected
GROUPS = {'cr': ['TRANSIENT', 'OVERFLOW'], 'outlier': ['OUTLIER'], 'persist': ['PERSIST'],
          'dichroic': ['DICHROIC'],
          'badpix': ['NONFUNC', 'MISSING_DATA', 'HOT', 'COLD', 'PHANMISS', 'NONLINEAR']}
SIGNED_LOG = ['track_snr', 'snr_wo_top', 'med_snr']
CUTOUT_META = ['bad_frac', 'noise_mad', 'persist_frac', 'cr_frac', 'outlier_frac', 'badpix_frac']


def flagdefs(header):
    d = {k: int(header[k]) for k in header if k.startswith('MP_')}
    for k in header:
        if re.fullmatch(r'MSKN\d{4}', k):
            d['MP_' + str(header[k])] = int(header['MSKM' + k[4:]]).bit_length() - 1
    return d


def masks(fl, header):
    d = flagdefs(header); fl = fl.astype(np.uint64)
    bad = (fl & np.uint64(sum(1 << b for k, b in d.items() if k not in INFO))) != 0
    src = ((fl >> np.uint64(d['MP_SOURCE'])) & np.uint64(1)).astype(bool)
    core = (fl & np.uint64(sum(1 << b for k, b in d.items() if k not in INFO and k != 'MP_OUTLIER'))) != 0
    if 'MP_OUTLIER' in d:
        outl = ((fl >> np.uint64(d['MP_OUTLIER'])) & np.uint64(1)).astype(bool)
    else:
        outl = np.zeros(fl.shape, bool)
    grp = {}
    for g, names in GROUPS.items():
        m = sum(1 << d['MP_' + n] for n in names if 'MP_' + n in d)
        grp[g + '_frac'] = float(((fl & np.uint64(m)) != 0).mean()) if m else 0.0
    return bad, core, outl, src, grp


def _meas(im, var, bcore, bfull, outl, src, x, y):
    x0, y0 = int(round(x)), int(round(y)); ny, nx = im.shape
    xa, xb, ya, yb = x0 - W, x0 + W + 1, y0 - W, y0 + W + 1
    if xa < 0 or ya < 0 or xb > nx or yb > ny:
        return None
    I, V, BC, BF, O, S = (a[ya:yb, xa:xb] for a in (im, var, bcore, bfull, outl, src))
    yy, xx = np.mgrid[ya:yb, xa:xb]; d2 = (xx - x) ** 2 + (yy - y) ** 2
    ap = d2 <= R_AP ** 2
    an = (d2 >= R_IN ** 2) & (d2 <= R_OUT ** 2) & ~BF & np.isfinite(I)      # background excludes every flagged pixel
    ok = ap & ~BC & np.isfinite(I) & np.isfinite(V) & (V > 0)                # aperture tolerates OUTLIER pixels
    if an.sum() < 30 or ok.sum() < 0.6 * ap.sum():
        return None
    bkg = np.median(I[an]); sky = 1.4826 * np.median(np.abs(I[an] - bkg)); k = ap.sum() / ok.sum()
    flux = (I[ok] - bkg).sum() * k
    noise = np.hypot(np.sqrt(V[ok].sum() * k), sky * np.sqrt(ap.sum()))
    return float(flux), float(noise), bool(S[ap].any()), float(O[ap].mean())


def analyze_frame(path, ra, dec, shift=(0.0, 0.0)):
    # shift (pixels) is only used to build displaced control tracks for testing; production uses (0, 0)
    with fits.open(path, memmap=False) as h:
        im = h['IMAGE'].data.astype(float); var = h['VARIANCE'].data.astype(float)
        fl = h['FLAGS'].data; fh = h['FLAGS'].header
        hd = h['IMAGE'].header; wcs = WCS(hd)
    bad, core, outl, src, grp = masks(fl, fh)
    ny, nx = im.shape
    out = dict(path=path, dqa=hd.get('L2DQAFLG'), finast=hd.get('FINAST'),
               coverage=float(np.isfinite(im).mean()), bad_frac=float(bad.mean()), src_frac=float(src.mean()))
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    if ok.sum() > 100:
        r = im[ok]; med = np.median(r); out['noise_mad'] = float(1.4826 * np.median(np.abs(r - med)))
    else:
        out['noise_mad'] = float('nan')
    out.update(grp)
    out['qc_pass'] = bool(out['dqa'] == 'Pass' and out['finast'] == 0 and out['bad_frac'] <= 0.25
                          and out['src_frac'] <= 0.90 and out['coverage'] >= 0.90)
    x, y = wcs.world_to_pixel(SkyCoord(ra, dec, unit='deg'))
    out['inside'] = bool(np.isfinite(x) and np.isfinite(y) and 12 <= x <= nx - 13 and 12 <= y <= ny - 13)
    out['meas'] = _meas(im, var, core, bad, outl, src, x + shift[0], y + shift[1]) if out['inside'] else None
    return out


def track_features(frames):
    flux = np.array([f['meas'][0] for f in frames]); noise = np.array([f['meas'][1] for f in frames])
    s = flux / noise; top = int(np.argmax(s)); rest = np.delete(np.arange(len(s)), top)
    pos = np.clip(s, 0, None); outap = np.array([f['meas'][3] for f in frames])
    feat = dict(n=len(frames), track_snr=float(flux.sum() / np.sqrt((noise ** 2).sum())),
                snr_wo_top=float(flux[rest].sum() / np.sqrt((noise[rest] ** 2).sum())),
                med_snr=float(np.median(s)), frac_gt3=float((s > 3).mean()),
                top_share=float(pos.max() / (pos.sum() + 1e-9)),
                outlier_ap=float(outap.mean()), outlier_frames=float((outap > 0).mean()))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for c in CUTOUT_META:
            feat['m_' + c] = float(np.nanmean([f[c] for f in frames]))
    feat['src_frac_ap'] = float(np.mean([f['meas'][2] for f in frames]))      # warning only, not a model feature
    for c in SIGNED_LOG:
        feat[c + '_l'] = float(np.sign(feat[c]) * np.log1p(abs(feat[c])))
    return feat


def load_model(path):
    return json.load(open(path))


def score(feat, model):
    x = []
    for k in model['features']:
        v = feat.get(k)
        x.append(model['fill'][k] if v is None or not np.isfinite(v) else v)
    z = (np.array(x, float) - np.array(model['mean'])) / np.array(model['scale'])
    p = 1 / (1 + np.exp(-(np.array(model['coefs']) @ z + np.array(model['intercepts']))))
    return float(p.mean()), float(p.std())


def spread(items, k=MAX_FRAMES):
    if len(items) <= k:
        return list(items)
    ix = np.unique(np.linspace(0, len(items) - 1, k).round().astype(int))
    return [items[i] for i in ix]


def rank(frames, model, shift=(0.0, 0.0), select=True, v_mag=None, version=None):
    # frames: list of dicts with path, ra, dec (degrees) and mjd, ideally 6-12 frames spread across ONE visit.
    # select=True keeps up to 6 usable frames spread evenly across the visit (the rule used in training);
    # select=False uses the frames as given (first 6). v_mag (predicted V) and version (e.g. 'l2b-v27') are optional.
    warns, used = [], []
    for f in sorted(frames, key=lambda d: d.get('mjd') or 0.0):
        a = analyze_frame(f['path'], f['ra'], f['dec'], shift)
        if not a['qc_pass']:
            warns.append('FRAME_FAILED_QC')
        elif not a['inside']:
            warns.append('POSITION_NEAR_EDGE_OR_OUTSIDE')
        elif a['meas'] is None:
            warns.append('APERTURE_UNMEASURABLE')
        else:
            used.append(dict(frame=f, a=a))
    res = dict(score=None, uncertainty=None, score_kind='candidate_score',
               model_version_id=model.get('version', 'unknown'), classification_hint='uncertain',
               explanation_codes=[], warnings=[], evidence=None, n_frames_used=len(used), frames_used=[],
               displacement_arcsec=None, span_hours=None, meets_5pct_threshold=False, meets_10pct_threshold=False)
    if len(used) < MIN_FRAMES:
        res['warnings'] = sorted(set(warns + ['TOO_FEW_USABLE_FRAMES']))
        return res
    if select:
        sel = spread(used)
    else:
        sel = used[:MAX_FRAMES]
        if len(used) > MAX_FRAMES:
            warns.append('EXTRA_FRAMES_IGNORED')
    feat = track_features([x['a'] for x in sel])
    s, u = score(feat, model)
    f0, f1 = sel[0]['frame'], sel[-1]['frame']
    disp = float(SkyCoord(f0['ra'], f0['dec'], unit='deg').separation(SkyCoord(f1['ra'], f1['dec'], unit='deg')).arcsec)
    mj = [x['frame'].get('mjd') for x in sel]
    span_h = float((max(mj) - min(mj)) * 24) if all(m is not None for m in mj) else None
    codes = []
    if feat['snr_wo_top'] >= 3: codes.append('CONSISTENT_ACROSS_FRAMES')
    elif feat['track_snr'] >= 3: codes.append('SINGLE_FRAME_DOMINATES')
    else: codes.append('LOW_TRACK_SNR')
    if feat['outlier_frames'] >= 0.5: codes.append('PIPELINE_OUTLIER_FLAG_AT_POSITION')
    if feat['src_frac_ap'] >= 0.99: warns.append('APERTURE_ON_CATALOG_SOURCE')       # every frame overlaps a catalogued source
    if disp / PIX_ARCSEC < LOW_MOTION_PX: warns.append('LOW_MOTION')
    t5, t10 = model.get('threshold_5pct'), model.get('threshold_10pct')
    hint = 'possible_motion' if (t5 is not None and s >= t5) else 'uncertain'
    vr = model.get('v_range')
    if v_mag is not None and vr:
        if v_mag > vr[1]:
            warns.append('PREDICTED_FAINTER_THAN_TRAINING_RANGE'); hint = 'uncertain'
        elif v_mag < vr[0]:
            warns.append('PREDICTED_BRIGHTER_THAN_TRAINING_RANGE')
    if version and version in model.get('limited_versions', []):
        warns.append('LIMITED_VALIDATION_FOR_PROCESSING_VERSION')
    res.update(score=s, uncertainty=u, classification_hint=hint, explanation_codes=codes,
               warnings=sorted(set(warns)), evidence=feat, frames_used=[x['frame']['path'] for x in sel],
               displacement_arcsec=disp, span_hours=span_h,
               meets_5pct_threshold=bool(t5 is not None and s >= t5),
               meets_10pct_threshold=bool(t10 is not None and s >= t10))
    return res
'''

CANDIDATES_SRC = r'''
# Known-object candidate finder using IMCCE SkyBoT. Needs astroquery and network access to IMCCE.
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time


def _cone(ra, dec, mjd, radius_deg, location):
    from astroquery.imcce import Skybot
    try:
        t = Skybot.cone_search(SkyCoord(ra, dec, unit='deg'), radius_deg * u.deg,
                               Time(mjd, format='mjd'), location=location)
        return t.to_pandas()
    except Exception as e:
        if 'No solar system object' in str(e):
            return None
        raise


def _num(x):
    try:
        v = float(x)
        return int(v) if np.isfinite(v) and v > 0 else None
    except Exception:
        return None


def find_candidates(frames, radius_deg=0.18, location='500', min_frames=4):
    # frames: dicts with ra, dec (search centre, degrees; 0.18 deg covers a 15 arcmin cutout) and mjd (mid-exposure).
    # One SkyBoT query per frame (about 3 s each). Returns known objects seen in at least min_frames frames,
    # sorted by predicted magnitude, each with per-frame ra, dec (degrees), v_mag.
    seen = {}
    for f in sorted(frames, key=lambda d: d['mjd']):
        t = _cone(f['ra'], f['dec'], f['mjd'], radius_deg, location)
        if t is None or len(t) == 0:
            continue
        for _, r in t.iterrows():
            name = str(r['Name'])
            c = seen.setdefault(name, dict(name=name, number=_num(r['Number']) if 'Number' in t.columns else None,
                                           frames=[]))
            c['frames'].append(dict(mjd=float(f['mjd']), ra=float(r['RA']), dec=float(r['DEC']),
                                    v_mag=float(r['V'])))
    out = []
    for c in seen.values():
        if len(c['frames']) >= min_frames:
            c['v_mag'] = float(np.nanmedian([x['v_mag'] for x in c['frames']]))
            out.append(c)
    return sorted(out, key=lambda c: c['v_mag'])
'''
open(f'{PK2}/aegis_ranker/features.py', 'w').write(FEATURES_SRC.lstrip())
open(f'{PK2}/aegis_ranker/candidates.py', 'w').write(CANDIDATES_SRC.lstrip())
open(f'{PK2}/aegis_ranker/__init__.py', 'w').write('')

# model file with metadata (v3 model trained on pooled A+B, V0 features)
model = json.load(open(f'{ROOT}/models/ranker_AB_v3.json'))
T = pd.read_csv(f'{ROOT}/data/manifests/astAB_track_table.csv'); mv = T[T.y == 1]
model['v_range'] = [round(float(mv.V.min()), 1), round(float(mv.V.max()), 1)]
vg = mv.ver.where(mv.ver.isin(['l2b-v20', 'l2b-v24', 'l2b-v27']), 'other').value_counts()
model['limited_versions'] = [v for v in ('l2b-v20', 'l2b-v24', 'l2b-v27') if int(vg.get(v, 0)) < 15]
model['min_frames'], model['max_frames'] = 4, 6
json.dump(model, open(f'{PK2}/ranker_AB_v3.json', 'w'))

for k in [k for k in sys.modules if k.startswith('aegis_ranker')]: del sys.modules[k]
sys.path = [p for p in sys.path if p != f'{ROOT}/package']; sys.path.insert(0, PK2)
import aegis_ranker.features as AF
print('package_v2 written | functions:', [n for n in ('analyze_frame', 'track_features', 'score', 'spread', 'rank') if hasattr(AF, n)])
print('trained V range:', model['v_range'], '| versions with <15 real movers:', model['limited_versions'],
      '| thresholds 5%/10%:', round(model['threshold_5pct'], 4), round(model['threshold_10pct'], 4))

package_v2 written | functions: ['analyze_frame', 'track_features', 'score', 'spread', 'rank']
trained V range: [10.1, 18.1] | versions with <15 real movers: ['l2b-v27'] | thresholds 5%/10%: 0.2951 0.1159


In [ ]:
# B13, cell 2: parity test -> golden cases -> model card, manifest, zip (package v2)
import os, sys, json, shutil, hashlib, subprocess, platform, datetime, time, warnings
import numpy as np, pandas as pd, astropy
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK2 = f'{ROOT}/package_v2'
for k in [k for k in sys.modules if k.startswith('aegis_ranker')]: del sys.modules[k]
sys.path = [p for p in sys.path if p != f'{ROOT}/package']; sys.path.insert(0, PK2)
import aegis_ranker.features as AF
model = AF.load_model(f'{PK2}/ranker_AB_v3.json')

T = pd.read_csv(f'{ROOT}/data/manifests/astAB_track_table.csv')
QC = {}
for s_ in ('A', 'B'):
    q = pd.read_csv(f'{ROOT}/data/manifests/ast{s_}_qc.csv')
    QC[s_] = q[q.status.isin(['ok', 'cached']) & (q.inside == True)].copy()
SEED = {'A': 101, 'B': 102}

def reconstruct(s_):          # shifts of every visit, in the order the training table was built
    rg = np.random.default_rng(SEED[s_]); sh = {}
    for v in sorted(T[(T.set == s_) & (T.kind == 'mover')].visit.unique()):
        r = rg.uniform(15, 40, 5); a = rg.uniform(0, 2 * np.pi, 5)
        sh[int(v)] = [(float(r[k] * np.cos(a[k])), float(r[k] * np.sin(a[k]))) for k in range(5)]
    return sh
SH = {s_: reconstruct(s_) for s_ in ('A', 'B')}

def frames_of(s_, v):
    g = QC[s_][QC[s_].visit == v].sort_values('mjd')
    return [dict(path=r.local_file, ra=float(r.ra), dec=float(r.dec), mjd=float(r.mjd)) for r in g.itertuples()]

def selected(s_, v):
    use = []
    for f in frames_of(s_, v):
        a = AF.analyze_frame(f['path'], f['ra'], f['dec'])
        if a['qc_pass'] and a['inside'] and a['meas'] is not None: use.append(f)
    return AF.spread(use), use

def dif(a, b):
    a, b = float(a), float(b)
    return 0.0 if (np.isnan(a) and np.isnan(b)) else abs(a - b)

movers = T[T.kind == 'mover']
gold = {}
for b in ('(8.0, 11.5]', '(14.5, 16.0]', '(17.0, 18.5]'):
    r = movers[movers.Vbin == b].sample(1, random_state=3).iloc[0]; gold[b] = (getattr(r, 'set'), int(r.visit))
others = movers[~movers.visit.isin([v for _, v in gold.values()])].sample(9, random_state=5)
sample = list(gold.values()) + [(getattr(r, 'set'), int(r.visit)) for r in others.itertuples()]

# ---- parity: features and scores against the training table; selection rule against rank(select=True) ----------
rows, cache, sel_ok = [], {}, []
for s_, v in tqdm(sample):
    sel, use = selected(s_, v); cache[(s_, v)] = (sel, use)
    for kind, shift in (('mover', (0.0, 0.0)), ('ctrl0', SH[s_][v][0]), ('ctrl1', SH[s_][v][1])):
        trow = T[(T.set == s_) & (T.visit == v) & (T.kind == kind)]
        if trow.empty: continue
        o = AF.rank(sel, model, shift=shift, select=False)
        if o['score'] is None:
            rows.append(dict(set=s_, visit=v, kind=kind, status='NOT SCORED: ' + ','.join(o['warnings']))); continue
        trow = trow.iloc[0].to_dict()
        d = {k: dif(o['evidence'][k], trow[k]) for k in model['features']}
        ref, _ = AF.score(trow, model)
        rows.append(dict(set=s_, visit=v, kind=kind, status='ok', max_feat_diff=max(d.values()),
                         worst=max(d, key=d.get), score_diff=abs(o['score'] - ref)))
    o = AF.rank(frames_of(s_, v), model, select=True)
    sel_ok.append([os.path.basename(p) for p in o['frames_used']] == [os.path.basename(f['path']) for f in sel])
D = pd.DataFrame(rows); good = D[D.status == 'ok']
print(f'tracks compared: {len(good)} of {len(D)} | max feature difference {good.max_feat_diff.max():.2e} | '
      f'max score difference {good.score_diff.max():.2e} | selection rule matches in {sum(sel_ok)} of {len(sel_ok)} visits')
if len(D) > len(good): print(D[D.status != 'ok'].to_string(index=False))
PARITY_OK = bool(len(good) == len(D) and good.max_feat_diff.max() < 1e-6 and good.score_diff.max() < 1e-6 and all(sel_ok))
print('PARITY', 'PASSED' if PARITY_OK else 'FAILED')
print('how often APERTURE_ON_CATALOG_SOURCE would fire (every frame on a catalogued source): '
      f"real movers {float((T[T.y == 1].src_frac_ap >= 0.99).mean()):.2f}, controls {float((T[T.y == 0].src_frac_ap >= 0.99).mean()):.2f}")

if PARITY_OK:
    gdir = f'{PK2}/golden'; os.makedirs(f'{gdir}/fits', exist_ok=True); cases = []
    def to_json(o):
        if isinstance(o, dict): return {k: to_json(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [to_json(v) for v in o]
        if isinstance(o, (float, np.floating)): return None if not np.isfinite(o) else float(o)
        if isinstance(o, np.integer): return int(o)
        return o
    def add_case(name, frames, kwargs, note):
        fr = []
        for f in frames:
            base = os.path.basename(f['path'])
            if not os.path.exists(f'{gdir}/fits/{base}'): shutil.copy(f['path'], f'{gdir}/fits/{base}')
            fr.append(dict(file='fits/' + base, ra=f['ra'], dec=f['dec'], mjd=f.get('mjd')))
        kw = dict(shift=[0.0, 0.0], select=True, v_mag=None, version=None) | kwargs
        res = dict(AF.rank(frames, model, shift=tuple(kw['shift']), select=kw['select'], v_mag=kw['v_mag'], version=kw['version']))
        res['frames_used'] = [os.path.basename(p) for p in res['frames_used']]
        cases.append(dict(name=name, note=note, kwargs=kw, frames=fr, expected=to_json(res)))
    for b, (s_, v) in gold.items():
        sel, use = cache[(s_, v)]
        add_case(f'real_mover_V{b}', sel, dict(select=False), 'known asteroid track, 4-6 spread frames')
        add_case(f'displaced_control_V{b}', sel, dict(select=False, shift=list(SH[s_][v][0])),
                 'same frames, path displaced by 15-40 px (shift is only for building controls)')
    (s0, v0) = list(gold.values())[1]; sel0, use0 = cache[(s0, v0)]
    big = [c for c in cache.values() if len(c[1]) > 6]
    if big: add_case('spread_selection_from_many_frames', big[0][1], dict(select=True),
                     'more than 6 usable frames: 6 frames spread across the visit are chosen')
    add_case('warn_low_motion_closely_spaced', use0[:4], dict(select=True), 'four consecutive exposures: little motion')
    add_case('warn_faint_prediction', sel0, dict(select=False, v_mag=19.5), 'predicted V above the trained range: hint capped at uncertain')
    add_case('warn_limited_version', sel0, dict(select=False, version='l2b-v27'), 'processing version with little validation')
    add_case('edge_too_few_frames', sel0[:3], dict(select=False), '3 frames: no score expected')
    add_case('edge_position_outside', [dict(f, ra=f['ra'] + 1.0) for f in sel0], dict(select=False), 'position 1 degree away')
    badf = pd.concat(QC.values()); badf = badf[badf.pass_gate == False].iloc[0]
    add_case('edge_qc_failed_frame', sel0[:3] + [dict(path=badf.local_file, ra=float(badf.ra), dec=float(badf.dec), mjd=float(badf.mjd))],
             dict(select=False), 'one frame fails QC and is dropped, leaving too few frames')
    json.dump(dict(tolerance=dict(abs=1e-6), cases=cases), open(f'{gdir}/cases.json', 'w'), indent=1)

    CHECK = r'''
import json, sys, os
here = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(here, '..'))
import aegis_ranker.features as AF
model = AF.load_model(sys.argv[1])
cases = json.load(open(os.path.join(here, 'cases.json'))); tol = cases['tolerance']['abs']; ok_all = True
for c in cases['cases']:
    frames = [dict(path=os.path.join(here, f['file']), ra=f['ra'], dec=f['dec'], mjd=f.get('mjd')) for f in c['frames']]
    kw = c['kwargs']
    got = AF.rank(frames, model, shift=tuple(kw['shift']), select=kw['select'], v_mag=kw['v_mag'], version=kw['version'])
    exp = c['expected']; errs = []
    if (got['score'] is None) != (exp['score'] is None): errs.append('scored/unscored mismatch')
    elif got['score'] is not None:
        for k in ('score', 'uncertainty', 'displacement_arcsec'):
            if abs(got[k] - exp[k]) > tol: errs.append(k + ' differs')
        for k, v in exp['evidence'].items():
            if v is not None and abs(got['evidence'][k] - v) > tol: errs.append('evidence ' + k + ' differs')
        if [os.path.basename(p) for p in got['frames_used']] != exp['frames_used']: errs.append('frames_used differ')
    for k in ('warnings', 'explanation_codes'):
        if sorted(got[k]) != sorted(exp[k]): errs.append(k + ' differ')
    if got['classification_hint'] != exp['classification_hint']: errs.append('classification_hint differs')
    print(c['name'], 'OK' if not errs else 'FAILED: ' + '; '.join(errs)); ok_all = ok_all and not errs
sys.exit(0 if ok_all else 1)
'''
    open(f'{gdir}/check_golden.py', 'w').write(CHECK.lstrip())

    # ---- model card (numbers copied from the B11/B12 printouts) ---------------------------------------------------
    c0 = cases[0]; fr0 = [dict(path=f'{gdir}/' + f['file'], ra=f['ra'], dec=f['dec'], mjd=f.get('mjd')) for f in c0['frames']]
    lat = min((lambda t0: (AF.rank(fr0, model, select=False), time.perf_counter() - t0)[1])(time.perf_counter()) for _ in range(3))
    hw = f"{platform.processor() or platform.machine()}, {os.cpu_count()} CPU cores, Python {platform.python_version()}"
    vr, lim = model['v_range'], model['limited_versions']
    card = f"""# AegisEarth candidate ranker ({model['version']}, package v2)

Optional hint for the AegisEarth viewer; the viewer works without it. The score is an UNCALIBRATED candidate score (it ranks candidates), not a probability.

## What it does
Given native SPHEREx L2 cutouts of ONE visit and, for a candidate moving object, one ICRS position per frame, it scores whether a moving source is consistently present across 4-6 frames spread over the visit. Output: score in [0,1] or null; uncertainty (spread over 50 bootstrap models); classification_hint ('possible_motion' when the score reaches the 5% false-alert threshold {model['threshold_5pct']:.4f}, otherwise 'uncertain'); explanation_codes; warnings; displacement_arcsec; span_hours; the evidence features; the frames used.
It does not find candidates. candidates.py proposes known objects from IMCCE SkyBoT; user-drawn paths can be scored too. Candidates must move on the sky.

## Input boundary
- Native IRSA cutout FITS (IMAGE, VARIANCE, FLAGS, WCS), not the prepared product. Flag bit definitions are read from the FLAGS header (QR2 MP_* and QR3 MSKN/MSKM are both handled).
- Cutout of 15 arcmin (0.25 deg, about 146 px) or similar, so that every candidate position is at least 12 px from the edge. A 3 arcmin cutout is too small for tracks that last hours.
- Send 6-12 frames spread across one visit (exposures separated by less than 12 h) with mjd; the module drops frames that fail QC and keeps up to 6 usable frames spread evenly in time, the rule used in training. Fewer than 4 usable frames gives score = null.
- Per-frame QC gate: L2DQAFLG == Pass, FINAST == 0, bad-pixel fraction <= 0.25, SOURCE fraction <= 0.90, non-NaN coverage >= 0.90.
- Optional: v_mag (predicted V from SkyBoT) and version (for example l2b-v27).

## Processing (identical in training and serving: aegis_ranker/features.py)
Aperture 2 px, background annulus 5-9 px, native pixels, IMAGE used directly (no ZODI subtraction). Rejected pixels: every flag defined in the header except MP_SOURCE and MP_FULLSAMPLE. In the aperture only, MP_OUTLIER pixels are tolerated (a static-sky pipeline flags moving objects as outliers); the background excludes them.

## Model
Logistic-regression ensemble (50 bootstrap models, class-balanced, C = 0.5) on {len(model['features'])} features: {', '.join(model['features'])}. Signed-log transform for S/N features. Wavelength/time and source-overlap features were tested and not used. Inference needs only numpy and astropy.

## Training data
114 real-mover tracks and 570 displaced-control tracks from 25 known asteroids (JPL Horizons positions; trained V range {vr[0]}-{vr[1]}), SPHEREx QR2/QR3 cutouts, processing versions l2b-v20 to v27, 6 frames spread over each visit. Movers by magnitude bin: V 8-11.5: 18, 11.5-14.5: 6, 14.5-16: 42, 16-17: 37, 17-18.5: 11. Controls are the same frames with the path displaced by 15-40 px (5 per visit). No synthetic data.

## Evaluation (pooled sets, leave-one-asteroid-out, production ensemble, 95% intervals from resampling asteroids)
| method | AUC | recall at 5% false alerts | false alerts at 90% recall |
|---|---|---|---|
| rule: S/N without top frame | 0.980 | 0.85 [0.70-1.00] | 0.056 [0.018-0.098] |
| image-only ranker | 0.975 | 0.87 [0.72-0.99] | 0.058 [0.013-0.119] |
| final model | 0.991 | 0.96 [0.91-1.00] | 0.016 [0.003-0.048] |

Paired, final minus image-only: recall +0.08 [+0.00, +0.21]; false alerts at 90% recall -0.037 [-0.083, -0.006]. Final minus rule: recall +0.09 [-0.02, +0.23]; false alerts -0.035 [-0.060, -0.011]. The false-alert reduction is supported; the recall gain is only suggestive.

Operating points (thresholds from out-of-fold control scores): 5% target: threshold {model['threshold_5pct']:.4f}, recall 0.96; 10% target: threshold {model['threshold_10pct']:.4f}, recall 0.98.
Recall at the 5% threshold by magnitude: V 8-11.5 1.00 (n=18); 11.5-14.5 1.00 (n=6); 14.5-16 1.00 (n=42); 16-17 0.95 (n=37); 17-18.5 0.82 (n=11). By processing version: l2b-v20 0.97 (n=36); v24 0.93 (n=30); v27 0.90 (n=10); other versions 1.00 (n=38).
Control false-alert rate at the 5% threshold: v20 0.061, v24 0.053, v27 0.100, other 0.026; set A 0.058, set B 0.049.
Threshold chosen on the other asteroids (held out): 5% target gives recall 0.96 with false alerts 0.054; 10% target gives recall 0.98 with false alerts 0.105.
Motion: tracks with under 4 px of motion between the first and last frame (49 of 114 real movers): recall 0.98, control false alerts 0.029; tracks with 4 px or more: recall 0.954, false alerts 0.068.

## Limitations
- Trained on {vr[0]}-{vr[1]} mag known asteroids only. V 11.5-14.5 has 6 and V 17-18.5 has 11 real tracks, so those rows are qualitative. Fainter objects, comets and other fields are untested; a predicted V above {vr[1]} cannot be labeled possible_motion (conservative design rule, not a measured one).
- The remaining false alerts are mostly bright catalogued stars on the path with no OUTLIER flag in the frames. A warning for this was tested and rejected by a pre-set rule (it fired on 13% of real movers). A fixed star tracked at constant RA/Dec would also score high: callers must supply moving candidates.
- Processing version(s) {', '.join(lim) or 'none'} have fewer than 15 real movers in training; their false-alert estimates are noisy and v27 showed twice the target rate. A warning is attached.
- Negatives are displaced paths through real frames, not output of a candidate generator.
- The feature set and thresholds were chosen on the same data used for the reported figures (leave-one-asteroid-out reduces but does not remove this optimism). An earlier check on the bright set with closely spaced frames and the first model gave 15.3% control false alerts; spread frames, pooled training and a re-derived threshold brought it to 5.8% and the individual contributions were not separated.
- Part of the performance comes from the pipeline's own OUTLIER flag at the candidate position; the ranker is not an independent detector.
- Frames in crowded fields often fail the QC gate; then the module returns warnings and no score.
- Not for the preset targets (Barnard's Star, Epsilon Indi B, WISE 0855, RW Aurigae, R Draconis): their motion between frames is far below one pixel or they are variable stars.
- Scores are not calibrated probabilities.

## Resources
One 6-frame track: {lat:.2f} s on {hw} (dominated by reading the FITS files). Environment: numpy {np.__version__}, astropy {astropy.__version__}; scikit-learn was used for training only. candidates.py needs astroquery and network access to IMCCE (about 3 s per frame).

## Golden cases
golden/cases.json with FITS files and expected outputs (absolute tolerance 1e-6). Run: python golden/check_golden.py ranker_AB_v3.json
"""
    open(f'{PK2}/MODEL_CARD.md', 'w').write(card)
    open(f'{PK2}/requirements.txt', 'w').write(f'numpy=={np.__version__}\nastropy=={astropy.__version__}\n# astroquery is needed only for candidates.py\n# python {platform.python_version()}\n')
    open(f'{PK2}/README.md', 'w').write(f"""# aegis_ranker package v2 ({model['version']})
    import aegis_ranker.features as AF, aegis_ranker.candidates as AC
    model = AF.load_model('ranker_AB_v3.json')
    cands = AC.find_candidates([dict(ra=..., dec=..., mjd=...), ...])           # known objects (SkyBoT), one query per frame
    res = AF.rank(frames, model, v_mag=cand['v_mag'], version='l2b-v27')        # frames: dicts with path, ra, dec, mjd
Keys: score (null when not scored), uncertainty, score_kind, model_version_id, classification_hint, explanation_codes, warnings,
evidence, n_frames_used, frames_used, displacement_arcsec, span_hours, meets_5pct_threshold, meets_10pct_threshold.
When score is null, store the candidate with classification 'uncertain', score NULL and the warnings in evidence.
See MODEL_CARD.md. Check: python golden/check_golden.py ranker_AB_v3.json
""")
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for blk in iter(lambda: f.read(1 << 20), b''): h.update(blk)
        return h.hexdigest()
    files = {}
    for root_, _, names in os.walk(PK2):
        for n in sorted(names):
            p = os.path.join(root_, n)
            if n != 'MANIFEST.json' and '__pycache__' not in p:
                files[os.path.relpath(p, PK2)] = dict(sha256=sha(p), bytes=os.path.getsize(p))
    json.dump(dict(model_version=model['version'], created_utc=datetime.datetime.utcnow().isoformat() + 'Z', hardware=hw,
                   numpy=np.__version__, astropy=astropy.__version__, score_kind='candidate_score', calibrated=False,
                   files=files), open(f'{PK2}/MANIFEST.json', 'w'), indent=1)
    r = subprocess.run([sys.executable, f'{gdir}/check_golden.py', f'{PK2}/ranker_AB_v3.json'], capture_output=True, text=True)
    print('\ngolden check (separate process):'); print(r.stdout.strip()); print(r.stderr.strip()[-400:]); print('exit code', r.returncode)
    print('\ngolden case summaries:')
    for c in cases:
        e = c['expected']
        print(f"  {c['name']:38s} score={None if e['score'] is None else round(e['score'], 3)} hint={e['classification_hint']} "
              f"warnings={e['warnings']}")
    shutil.make_archive(f'{ROOT}/aegis_ranker_package_v2', 'zip', PK2)
    print(f"\nzip: {ROOT}/aegis_ranker_package_v2.zip ({os.path.getsize(f'{ROOT}/aegis_ranker_package_v2.zip') / 1e6:.0f} MB)")

  0%|          | 0/12 [00:00<?, ?it/s]

tracks compared: 36 of 36 | max feature difference 4.44e-16 | max score difference 1.83e-15 | selection rule matches in 12 of 12 visits
PARITY PASSED
how often APERTURE_ON_CATALOG_SOURCE would fire (every frame on a catalogued source): real movers 0.40, controls 0.36

golden check (separate process):
real_mover_V(8.0, 11.5] OK
displaced_control_V(8.0, 11.5] OK
real_mover_V(14.5, 16.0] OK
displaced_control_V(14.5, 16.0] OK
real_mover_V(17.0, 18.5] OK
displaced_control_V(17.0, 18.5] OK
spread_selection_from_many_frames OK
warn_low_motion_closely_spaced OK
warn_faint_prediction OK
warn_limited_version OK
edge_too_few_frames OK
edge_position_outside OK
edge_qc_failed_frame OK

exit code 0

golden case summaries:
  real_mover_V(8.0, 11.5]                score=1.0 hint=possible_motion warnings=[]
  displaced_control_V(8.0, 11.5]         score=0.001 hint=uncertain warnings=[]
  real_mover_V(14.5, 16.0]               score=0.941 hint=possible_motion warnings=['APERTURE_ON_CATALOG_SOURCE', 'LOW

In [ ]:
# B13, cell 3: does candidates.py recover asteroids we already know? (live test, needs network access to IMCCE)
import sys, time, warnings
import numpy as np, pandas as pd
from astropy.coordinates import SkyCoord
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK2 = f'{ROOT}/package_v2'
for k in [k for k in sys.modules if k.startswith('aegis_ranker')]: del sys.modules[k]
sys.path = [p for p in sys.path if p != f'{ROOT}/package']; sys.path.insert(0, PK2)
import aegis_ranker.candidates as AC

FB = pd.read_csv(f'{ROOT}/data/manifests/feasibility_frames_B.csv')
big = FB.groupby('visit').filter(lambda g: g.obs_id.nunique() >= 5)
rows = []
for v in big.visit.drop_duplicates().sample(3, random_state=2):
    g = big[big.visit == v].drop_duplicates('obs_id').sort_values('mjd').head(8)
    ast = int(g.ast.iloc[0]); cen = SkyCoord(float(g.ra.mean()), float(g.dec.mean()), unit='deg')
    rad = max(cen.separation(SkyCoord(r.ra, r.dec, unit='deg')).deg for r in g.itertuples()) + 0.05
    frames = [dict(ra=cen.ra.deg, dec=cen.dec.deg, mjd=float(m)) for m in g.mjd]
    t0 = time.time(); cands = AC.find_candidates(frames, radius_deg=rad); dt = time.time() - t0
    hit = [c for c in cands if c['number'] == ast]
    if not hit:
        rows.append(dict(visit=int(v), ast=ast, found=False, candidates=len(cands), seconds=round(dt))); continue
    c = hit[0]; seps, dv = [], []
    for fr in c['frames']:
        r = g.iloc[int(np.abs(g.mjd.values - fr['mjd']).argmin())]
        seps.append(SkyCoord(fr['ra'], fr['dec'], unit='deg').separation(SkyCoord(r.ra, r.dec, unit='deg')).arcsec)
        dv.append(fr['v_mag'] - r.V)
    rows.append(dict(visit=int(v), ast=ast, found=True, candidates=len(cands), frames_sent=len(frames),
                     frames_matched=len(c['frames']), median_sep_arcsec=np.median(seps), max_sep_arcsec=np.max(seps),
                     median_dV=np.median(dv), seconds=round(dt)))
R = pd.DataFrame(rows); print(R.round(3).to_string(index=False))
ok = bool(R.found.all() and (R.frames_matched >= 4).all() and (R.max_sep_arcsec < 1.0).all())
print('\ncandidates.py check', 'PASSED' if ok else 'FAILED')


 visit  ast  found  candidates  frames_sent  frames_matched  median_sep_arcsec  max_sep_arcsec  median_dV  seconds
100125  731   True           1            6               6              0.027           0.027      0.041       26
100141  752   True           8            6               6              0.038           0.040      0.079       17
100282  920   True           7            8               8              0.035           0.038      0.065       21

candidates.py check PASSED


In [ ]:
# B13b: slim the golden FITS, label known misses/false alerts, re-check, refresh manifest + zip
import os, sys, json, shutil, hashlib, subprocess, datetime, warnings
from astropy.io import fits
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK2 = f'{ROOT}/package_v2'; gdir = f'{PK2}/golden'
size = lambda d: sum(os.path.getsize(f'{d}/{f}') for f in os.listdir(d)) / 1e6
print(f'golden FITS before: {size(gdir + "/fits"):.0f} MB')

for fn in sorted(os.listdir(f'{gdir}/fits')):
    p = f'{gdir}/fits/{fn}'
    with fits.open(p, memmap=False) as h:                     # keep only what the module reads
        fits.HDUList([h[0], h['IMAGE'], h['FLAGS'], h['VARIANCE']]).writeto(p + '.tmp', overwrite=True)
    os.replace(p + '.tmp', p)
print(f'golden FITS after:  {size(gdir + "/fits"):.0f} MB')

cases = json.load(open(f'{gdir}/cases.json'))
for c in cases['cases']:
    e = c['expected']
    if c['name'].startswith('real_mover') and e['classification_hint'] == 'uncertain':
        c['note'] += ' | KNOWN MISS: a real mover scoring below the 5% threshold (faintest bin); not an error'
    if c['name'].startswith('displaced_control') and e['classification_hint'] == 'possible_motion':
        c['note'] += ' | KNOWN FALSE ALERT: a displaced control above the threshold; not an error'
json.dump(cases, open(f'{gdir}/cases.json', 'w'), indent=1)

r = subprocess.run([sys.executable, f'{gdir}/check_golden.py', f'{PK2}/ranker_AB_v3.json'], capture_output=True, text=True)
print('\ngolden check on the slimmed files:'); print(r.stdout.strip()); print(r.stderr.strip()[-300:]); print('exit code', r.returncode)

if r.returncode == 0:
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for blk in iter(lambda: f.read(1 << 20), b''): h.update(blk)
        return h.hexdigest()
    man = json.load(open(f'{PK2}/MANIFEST.json')); files = {}
    for root_, _, names in os.walk(PK2):
        for n in sorted(names):
            p = os.path.join(root_, n)
            if n != 'MANIFEST.json' and '__pycache__' not in p:
                files[os.path.relpath(p, PK2)] = dict(sha256=sha(p), bytes=os.path.getsize(p))
    man.update(files=files, refreshed_utc=datetime.datetime.utcnow().isoformat() + 'Z')
    json.dump(man, open(f'{PK2}/MANIFEST.json', 'w'), indent=1)
    shutil.make_archive(f'{ROOT}/aegis_ranker_package_v2', 'zip', PK2)
    print(f"\nzip refreshed: {os.path.getsize(f'{ROOT}/aegis_ranker_package_v2.zip') / 1e6:.0f} MB")
    print('model sha256:', files['ranker_AB_v3.json']['sha256'][:16], '| features.py sha256:', files['aegis_ranker/features.py']['sha256'][:16])
else:
    print('\ngolden check FAILED on the slimmed files: zip NOT refreshed; paste this output')

golden FITS before: 254 MB
golden FITS after:  15 MB

golden check on the slimmed files:
real_mover_V(8.0, 11.5] OK
displaced_control_V(8.0, 11.5] OK
real_mover_V(14.5, 16.0] OK
displaced_control_V(14.5, 16.0] OK
real_mover_V(17.0, 18.5] OK
displaced_control_V(17.0, 18.5] OK
spread_selection_from_many_frames OK
warn_low_motion_closely_spaced OK
warn_faint_prediction OK
warn_limited_version OK
edge_too_few_frames OK
edge_position_outside OK
edge_qc_failed_frame OK

exit code 0

zip refreshed: 8 MB
model sha256: 2171854d7a668681 | features.py sha256: 876d04c3b3dbab81


In [ ]:
# B14: false-alert test on the original three crowded fields (random moving paths through real frames)
import os, sys, json, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK2 = f'{ROOT}/package_v2'
for k in [k for k in sys.modules if k.startswith('aegis_ranker')]: del sys.modules[k]
sys.path = [p for p in sys.path if p != f'{ROOT}/package']; sys.path.insert(0, PK2)
import aegis_ranker.features as AF
model = AF.load_model(f'{PK2}/ranker_AB_v3.json'); T5, T10 = model['threshold_5pct'], model['threshold_10pct']

FL = json.load(open(f'{ROOT}/configs/fields.json'))
Q = pd.read_csv(f'{ROOT}/data/manifests/qc_manifest.csv')
Q = Q[Q.local_file.notna() & Q.mjd_avg.notna()].copy()
Q = Q[Q.local_file.map(os.path.exists)]
print(len(Q), 'frames on disk,', Q.groupby(['field', 'visit']).ngroups, 'visits;', Q.field.value_counts().to_dict())

def analyze_multi(path, radecs):                    # same steps as AF.analyze_frame, but one file read for many paths
    with fits.open(path, memmap=False) as h:
        im = h['IMAGE'].data.astype(float); var = h['VARIANCE'].data.astype(float)
        fl = h['FLAGS'].data; fh = h['FLAGS'].header; hd = h['IMAGE'].header; wcs = WCS(hd)
    bad, core, outl, src, grp = AF.masks(fl, fh); ny, nx = im.shape
    base = dict(path=path, dqa=hd.get('L2DQAFLG'), finast=hd.get('FINAST'), coverage=float(np.isfinite(im).mean()),
                bad_frac=float(bad.mean()), src_frac=float(src.mean()))
    ok = ~bad & ~src & np.isfinite(im) & np.isfinite(var) & (var > 0)
    if ok.sum() > 100:
        r = im[ok]; med = np.median(r); base['noise_mad'] = float(1.4826 * np.median(np.abs(r - med)))
    else:
        base['noise_mad'] = float('nan')
    base.update(grp)
    base['qc_pass'] = bool(base['dqa'] == 'Pass' and base['finast'] == 0 and base['bad_frac'] <= 0.25
                           and base['src_frac'] <= 0.90 and base['coverage'] >= 0.90)
    xs, ys = wcs.world_to_pixel(SkyCoord([p[0] for p in radecs], [p[1] for p in radecs], unit='deg'))
    out = []
    for x, y in zip(xs, ys):
        a = dict(base); a['inside'] = bool(np.isfinite(x) and np.isfinite(y) and 12 <= x <= nx - 13 and 12 <= y <= ny - 13)
        a['meas'] = AF._meas(im, var, core, bad, outl, src, x, y) if a['inside'] else None
        out.append(a)
    return out

rows, TR = [], {}
rg = np.random.default_rng(2026); K = 15
for (fld, vis), g in tqdm(list(Q.groupby(['field', 'visit']))):
    g = g.sort_values('mjd_avg')
    if len(g) < 5: continue
    t = g.mjd_avg.values; span_h = (t.max() - t.min()) * 24
    if span_h < 0.5: continue
    vmax = min(60.0, 0.12 * 3600 / span_h)                       # keep the path within 0.12 deg of the centre
    if vmax < 5.0: continue
    ra0, dec0 = FL[fld]['ra'], FL[fld]['dec']; dt = (t - t.mean()) * 24; tracks = []
    for k in range(K):
        rate = float(np.exp(rg.uniform(np.log(5.0), np.log(vmax)))); th = rg.uniform(0, 2 * np.pi); off = rg.uniform(-0.06, 0.06, 2)
        dra = (off[0] + rate * np.cos(th) * dt / 3600.0) / np.cos(np.radians(dec0)); ddec = off[1] + rate * np.sin(th) * dt / 3600.0
        tracks.append(dict(rate=rate, pos=[(ra0 + dra[i], dec0 + ddec[i]) for i in range(len(t))]))
    per_frame = [analyze_multi(r.local_file, [tr['pos'][i] for tr in tracks]) for i, r in enumerate(g.itertuples())]
    paths = list(g.local_file)
    for k, tr in enumerate(tracks):
        used = [dict(a=per_frame[i][k]) for i in range(len(t))
                if per_frame[i][k]['qc_pass'] and per_frame[i][k]['inside'] and per_frame[i][k]['meas'] is not None]
        row = dict(field=fld, visit=vis, track=k, rate=tr['rate'], scored=len(used) >= AF.MIN_FRAMES)
        if row['scored']:
            feat = AF.track_features([u['a'] for u in AF.spread(used)]); s, _ = AF.score(feat, model)
            row.update(score=s, src_ap=feat['src_frac_ap'], outlier_frames=feat['outlier_frames'], snr_wo_top=feat['snr_wo_top'])
            TR[(fld, vis, k)] = (paths, tr['pos'], list(t))
        rows.append(row)
R = pd.DataFrame(rows); sc = R[R.scored].reset_index(drop=True)

chk = []                                                         # the batch shortcut must equal the production rank()
for i in (0, len(sc) // 2, len(sc) - 1):
    r = sc.iloc[i]; paths, pos, mj = TR[(r.field, r.visit, int(r.track))]
    o = AF.rank([dict(path=p, ra=pp[0], dec=pp[1], mjd=m) for p, pp, m in zip(paths, pos, mj)], model, select=True)
    chk.append(abs(o['score'] - r.score))
print(f'parity with AF.rank on 3 tracks: max score difference {max(chk):.2e}')

R['alert5'] = R.scored & (R.score >= T5); R['alert10'] = R.scored & (R.score >= T10)
def rate_ci(d, col, B=2000):
    v = d[d.scored].groupby(['field', 'visit']).agg(a=(col, 'sum'), n=(col, 'size'))
    if len(v) == 0: return (np.nan,) * 3
    r = np.random.default_rng(1); A, N = v.a.values, v.n.values
    bs = [A[i].sum() / N[i].sum() for i in (r.integers(0, len(v), len(v)) for _ in range(B))]
    return (A.sum() / N.sum(), *np.percentile(bs, [2.5, 97.5]))
out = []
for name in ['elon240', 'elon150', 'elon030', 'ALL']:
    d = R if name == 'ALL' else R[R.field == name]
    a, b = rate_ci(d, 'alert5'), rate_ci(d, 'alert10')
    out.append(dict(field=name, visits=d.groupby(['field', 'visit']).ngroups, tracks=len(d), share_scored=round(float(d.scored.mean()), 2),
                    FA_5pct=f'{a[0]:.3f} [{a[1]:.3f}-{a[2]:.3f}]', FA_10pct=f'{b[0]:.3f} [{b[1]:.3f}-{b[2]:.3f}]',
                    median_score=round(float(d[d.scored].score.median()), 4)))
print('\nfalse-alert rate of random moving paths (scored tracks; intervals resample visits):'); print(pd.DataFrame(out).to_string(index=False))
sc['alert5'] = sc.score >= T5
print('\nby path rate (arcsec/hour):'); print(sc.groupby(pd.cut(sc.rate, [4, 10, 20, 61]), observed=True).alert5.agg(rate='mean', alerts='sum', n='size').round(3).to_string())
al = sc[sc.alert5]
if len(al): print(f'\nalerts: {len(al)} | every frame on a catalogued source: {float((al.src_ap >= 0.99).mean()):.2f} | '
                  f'median OUTLIER share of frames {al.outlier_frames.median():.2f} | median S/N without top frame {al.snr_wo_top.median():.1f}')
pooled = rate_ci(R, 'alert5')
print(f'\npre-set criterion (pooled false alerts at the 5% threshold <= 0.10 with upper end <= 0.15): '
      f'{"PASSED" if pooled[0] <= 0.10 and pooled[2] <= 0.15 else "NOT MET"} ({pooled[0]:.3f}, upper {pooled[2]:.3f})')

386 frames on disk, 24 visits; {'elon240': 139, 'elon150': 131, 'elon030': 116}


  0%|          | 0/24 [00:00<?, ?it/s]

parity with AF.rank on 3 tracks: max score difference 0.00e+00

false-alert rate of random moving paths (scored tracks; intervals resample visits):
  field  visits  tracks  share_scored             FA_5pct            FA_10pct  median_score
elon240       9     135           1.0 0.126 [0.067-0.178] 0.252 [0.185-0.319]        0.0352
elon150       7     105           1.0 0.048 [0.019-0.086] 0.095 [0.057-0.143]        0.0043
elon030       8     120           1.0 0.042 [0.017-0.058] 0.058 [0.025-0.092]        0.0030
    ALL      24     360           1.0 0.075 [0.047-0.106] 0.142 [0.100-0.189]        0.0089

by path rate (arcsec/hour):
           rate  alerts    n
rate                        
(4, 10]   0.092      11  119
(10, 20]  0.063       9  142
(20, 61]  0.071       7   99

alerts: 27 | every frame on a catalogued source: 0.63 | median OUTLIER share of frames 0.17 | median S/N without top frame 6.8

pre-set criterion (pooled false alerts at the 5% threshold <= 0.10 with upper end <= 0.15

In [ ]:
# B15: finalize the package: B14 in the model card, integration sheet, example output, golden check, manifest, zip
import os, sys, json, shutil, hashlib, subprocess, datetime, warnings
warnings.filterwarnings('ignore')
ROOT = globals().get('ROOT', '/content/drive/MyDrive/spherex_project')
PK2 = f'{ROOT}/package_v2'
m = json.load(open(f'{PK2}/ranker_AB_v3.json'))
T5, T10, vr = m['threshold_5pct'], m['threshold_10pct'], m['v_range']
lim = ', '.join(m['limited_versions']) or 'none'
gsize = sum(os.path.getsize(f'{PK2}/golden/fits/{f}') for f in os.listdir(f'{PK2}/golden/fits')) / 1e6
print(f'golden FITS size: {gsize:.0f} MB' + ('   <- still large: run B13b first' if gsize > 40 else ''))

# 1. model card: add the B14 result once
card_p = f'{PK2}/MODEL_CARD.md'; card = open(card_p).read()
if 'Crowded-field false-alert test' not in card:
    card += f"""

## Crowded-field false-alert test (original three fields, nothing injected)
Random moving paths (rates 5-60 arcsec/hour, random directions, 15 per visit) were drawn through the real frames of 24 visits in three crowded ecliptic fields and scored with the production module. All 360 tracks could be scored. False-alert rate at the 5% threshold ({T5:.4f}): 7.5% pooled [4.7-10.6] (intervals resample visits); elon030 4.2% [1.7-5.8], elon150 4.8% [1.9-8.6], and the most crowded field elon240 12.6% [6.7-17.8]. At the 10% threshold ({T10:.4f}) the pooled rate was 14.2% [10.0-18.9] and 25.2% in elon240, so the 10% threshold is not reliable in crowded fields and must not be used for labels. Of the 27 alerts at the 5% threshold, 63% had every frame on a catalogued source and the median share of frames with the OUTLIER flag was 0.17. A pre-set criterion (pooled rate <= 0.10 with upper end <= 0.15) was met, but one field exceeded the 5% target. Random paths approximate user-drawn paths, not SkyBoT candidates; recall cannot be measured in these fields.
"""
    open(card_p, 'w').write(card)

# 2. integration sheet: the one document the backend developer needs
WARN = {
 'TOO_FEW_USABLE_FRAMES': 'Not enough usable frames to score this candidate.',
 'FRAME_FAILED_QC': 'Some frames were excluded for data-quality reasons.',
 'POSITION_NEAR_EDGE_OR_OUTSIDE': 'The predicted position is at or beyond the edge of some frames.',
 'APERTURE_UNMEASURABLE': 'Too many masked pixels at the predicted position in some frames.',
 'LOW_MOTION': 'Little motion across these frames; a fixed source would look the same.',
 'APERTURE_ON_CATALOG_SOURCE': 'A catalogued source overlaps the position in every frame; flux may be contaminated.',
 'PREDICTED_FAINTER_THAN_TRAINING_RANGE': 'Predicted brightness is outside the range this ranker was tested on.',
 'PREDICTED_BRIGHTER_THAN_TRAINING_RANGE': 'Same, on the bright side.',
 'LIMITED_VALIDATION_FOR_PROCESSING_VERSION': 'Few validation examples exist for this data-processing version.',
 'EXTRA_FRAMES_IGNORED': 'More than six frames were supplied and only the first six were used.'}
INTEG = '''# Integration sheet: aegis_ranker package v2

Optional hint for the AegisEarth viewer. The viewer must keep working when this module fails or is switched off.

## What is in the package
- aegis_ranker/features.py: reads native IRSA cutouts and scores a candidate track. Needs numpy and astropy.
- aegis_ranker/candidates.py: finds known objects with IMCCE SkyBoT. Needs astroquery and network access to IMCCE.
- ranker_AB_v3.json: the model (50 small logistic models, scores are averaged). MANIFEST.json holds a SHA-256 for every file.
- golden/: 13 test cases with FITS files and expected outputs. MODEL_CARD.md: evidence and limits. example_output.json: one result.

## Per-job flow
1. Pick the visit: exposures of the same sky position, split where the gap between exposures exceeds 12 hours. Use 6-12 exposures spread across the visit (not the first few).
2. Download one NATIVE cutout of 15 arcmin per exposure (the same IRSA cutout URL pattern as cutouts_manifest.json, with size=15arcmin, centred on the search centre). Not the prepared/resampled product. Download in parallel and cache.
3. frames = [dict(ra=centre_ra, dec=centre_dec, mjd=mid_mjd), ...] with mid_mjd = (t_min + t_max) / 2.
   cands = candidates.find_candidates(frames, radius_deg=0.18)   # about 3 s per frame, one SkyBoT query each
4. For each candidate c, build its frames: for every exposure whose mjd appears in c['frames'], dict(path=cutout_file, ra=<that frame's ra>, dec=<that frame's dec>, mjd=<mjd>).
5. res = features.rank(cand_frames, model, v_mag=c['v_mag'], version='l2b-v27')   # version = processing version from the file path, optional
6. Store the result as below. Run steps 3-6 inside the async job, never in the request.

## Output fields and the candidates table
- score (0-1, or None): candidate_score, uncalibrated, NOT a probability. None means it could not score. -> candidates.score (NULL allowed), score_kind = 'candidate_score'.
- classification_hint: 'possible_motion' or 'uncertain' -> candidates.classification. The model never says 'artifact' or 'brightness_change'.
- displacement_arcsec -> candidates.displacement_arcsec. Use the candidate's position in the middle frame for ra/dec.
- uncertainty: spread of the score over 50 models, not a confidence interval -> needs a new optional field score_uncertainty.
- warnings, explanation_codes, span_hours, frames_used, evidence, n_frames_used -> candidates.evidence (JSON). model_version_id -> candidates.model_version_id.
- meets_10pct_threshold is informational only. Do not use it for labels.

## What the viewer should show
Score as a rank (not a percentage), the uncertainty, per-frame positions and timestamps and wavelengths from the backend's own frame metadata, and the warnings in plain words:
@WARNTABLE@
For a 'possible_motion' candidate that also carries APERTURE_ON_CATALOG_SOURCE add: "In crowded areas this label is wrong more often (about 1 in 8 random paths in our most crowded test field)."

## Rules
- Do NOT run it on the five preset targets. Do not call the score a probability. Do not substitute snr_proxy or the prepared FITS.
- The only validated threshold is the 5% one (@T5@). classification_hint already uses it.
- Trained V range is @VLO@-@VHI@. A predicted V above @VHI@ is capped at 'uncertain' by the module (pass v_mag).
- The module drops frames that fail QC or sit near the edge. Fewer than 4 usable frames gives score None with warnings only.
- Candidates must move on the sky (SkyBoT objects or user-drawn paths).
- Processing versions with few validation examples: @LIM@.

## Time and resources
Cutout download 14-43 s each (from your manifest); SkyBoT about 3 s per frame; scoring about 1 s per candidate on 2 CPU cores (reading the FITS dominates). Several candidates per field is normal (7-8 in our test).

## Your acceptance checks
1. pip install -r requirements.txt (plus astroquery for candidates.py).
2. python golden/check_golden.py ranker_AB_v3.json -> all 13 cases OK, also in your Docker image.
3. Verify the SHA-256 of every file against MANIFEST.json.
4. Run one real visit end to end and send me the result JSON.
5. A failure of this module must never disable playback, spectra or survey context.
'''
INTEG = (INTEG.replace('@WARNTABLE@', '\n'.join(f'- {k}: {v}' for k, v in WARN.items()))
         .replace('@T5@', f'{T5:.4f}').replace('@VLO@', str(vr[0])).replace('@VHI@', str(vr[1])).replace('@LIM@', lim))
open(f'{PK2}/INTEGRATION.md', 'w').write(INTEG)
cases = json.load(open(f'{PK2}/golden/cases.json'))
json.dump(cases['cases'][0]['expected'], open(f'{PK2}/example_output.json', 'w'), indent=1)

# 3. golden check, then manifest and zip (only if the check passes)
r = subprocess.run([sys.executable, f'{PK2}/golden/check_golden.py', f'{PK2}/ranker_AB_v3.json'], capture_output=True, text=True)
print('\ngolden check (separate process):'); print(r.stdout.strip()); print(r.stderr.strip()[-300:]); print('exit code', r.returncode)
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for blk in iter(lambda: f.read(1 << 20), b''): h.update(blk)
    return h.hexdigest()
if r.returncode == 0:
    man = json.load(open(f'{PK2}/MANIFEST.json')); files = {}
    for root_, _, names in os.walk(PK2):
        for n in sorted(names):
            p = os.path.join(root_, n)
            if n != 'MANIFEST.json' and '__pycache__' not in p:
                files[os.path.relpath(p, PK2)] = dict(sha256=sha(p), bytes=os.path.getsize(p))
    man.update(files=files, refreshed_utc=datetime.datetime.utcnow().isoformat() + 'Z')
    json.dump(man, open(f'{PK2}/MANIFEST.json', 'w'), indent=1)
    shutil.make_archive(f'{ROOT}/aegis_ranker_package_v2', 'zip', PK2)
    z = f'{ROOT}/aegis_ranker_package_v2.zip'
    print(f'\nzip: {z} ({os.path.getsize(z) / 1e6:.0f} MB) | zip sha256: {sha(z)}')
    print('files in the package:', sorted(files))
else:
    print('\ngolden check FAILED: zip not refreshed; paste this output')

golden FITS size: 15 MB

golden check (separate process):
real_mover_V(8.0, 11.5] OK
displaced_control_V(8.0, 11.5] OK
real_mover_V(14.5, 16.0] OK
displaced_control_V(14.5, 16.0] OK
real_mover_V(17.0, 18.5] OK
displaced_control_V(17.0, 18.5] OK
spread_selection_from_many_frames OK
warn_low_motion_closely_spaced OK
warn_faint_prediction OK
warn_limited_version OK
edge_too_few_frames OK
edge_position_outside OK
edge_qc_failed_frame OK

exit code 0

zip: /content/drive/MyDrive/spherex_project/aegis_ranker_package_v2.zip (8 MB) | zip sha256: d86698b8d5bf03012b09c3a21aae1606218b08bfe2415b55484e0e12b73e9029
files in the package: ['INTEGRATION.md', 'MODEL_CARD.md', 'README.md', 'aegis_ranker/__init__.py', 'aegis_ranker/candidates.py', 'aegis_ranker/features.py', 'example_output.json', 'golden/cases.json', 'golden/check_golden.py', 'golden/fits/level2_2025W51_2A_0573_1D1_spx_l2b-v21-2025-357_ast20.fits', 'golden/fits/level2_2025W51_2A_0573_1D4_spx_l2b-v21-2025-357_ast20.fits', 'golden/fits/lev